# 11 dang (D35, D36, D37, D39, D44, D45, D47, D48, D50, D52, D55) - FULL90 - DeepSeek-R1-Distill-Qwen-1.5B

Gop 11 file full90 rieng le (moi file da build Y CHANG 100% ban Qwen3-4B da verified - chi doi MODEL/temperature, KHONG warm-up, KHONG doi du lieu/prompt - va da doi chieu FEWSHOT khop 100% voi backend that) thanh 1 notebook chay tuan tu.

**Moi dang giu NGUYEN 100% block backend+prompt+vong lap cua chinh no** (khac nhau that su ve backend/MAX_TOOL_CALLS giua cac dang - khac biet co chu dich, khong gop thanh 1 logic dung chung).

**Chi khoi DAU TIEN (D35) nap tok/llm that su**; 10 khoi con lai dung lai bien da nap - tranh loi GPU het bo nho khi nap lai LLM(...) nhieu lan (da xac nhan qua thuc te chay tren Kaggle).

Doc du lieu tu 1 file JSON gop chung `plan_solve_prompts_11dang_merged.json` - **CAN UPLOAD** len Kaggle (hoac sua `MERGED_DATA_PATH` cho khop).

`temperature=0.6, top_p=0.95` (khuyen nghi chinh thuc DeepSeek cho dong R1-Distill, MUC TOI THIEU chong lap lai - khong giam nua).

In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec
import vllm; print('vLLM:', vllm.__version__)

## Dang D35

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

OUT_PATH  = '/kaggle/working/d35_react_calculator_full90_DeepSeek-R1-Distill-Qwen-1.5B.csv'

MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = {'x', 'y'}

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/11-dang-full/plan_solve_prompts_11dang_merged.json'  # SUA NEU KHAC
with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D35']
print('So cau:', len(records))

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('San sang.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D35_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai [C] PART_HUONGGIAI va
# [D] PART_FEWSHOT, giu nguyen [A] PART_TOOL va [B] PART_KIENTHUC.

# [A] PART_TOOL - HUONG DAN DUNG TOOL (DUNG CHUNG CHO MOI DANG TOAN)
# =====================================================================
PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- To test whether two expressions are **exactly** equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`, decided exactly; never approximately.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher-degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation-solve mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non-real ones when the polynomial's real roots don't account for its full degree; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`; `Im` is not recognized and silently fails to evaluate) and discard it immediately if not. For a root confirmed real, use `re(root)`; not the raw value; in every later threshold or substitution, since even a numerically-real root can carry a residual non-zero imaginary part too small to matter but large enough to break a direct comparison. Any comparison built on such numeric roots (a threshold like `t > 0`, or a self-consistency check like `Abs(a - b) < 1e-6`) should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root; in particular the final count of solutions and matching it against the answer options; stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


# =====================================================================
# [B] PART_KIENTHUC - KIEN THUC NEN SO PHUC (DUNG CHUNG MOI DANG SO PHUC)
# =====================================================================
PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
'''


# =====================================================================
# [C] PART_HUONGGIAI - HUONG GIAI RIENG CUA DANG D35 (THAY KHI DOI DANG)
# =====================================================================
PART_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: count how many complex numbers $z$ satisfy BOTH (a) a modulus equation $|z-z_0|=R$, and (b) a fraction $\dfrac{z}{z-c}$ ($c$ a given real constant) is **pure imaginary**.

This is pure algebra on $x,y$; NOT the triangle-inequality method from other problem types. To find the real and imaginary parts of a fraction, multiply numerator and denominator by the conjugate of the denominator, expand the product with `expand(...)`, and read off which terms carry a factor of `I` (the imaginary part) and which do not (the real part).

Follow this method exactly; do not invent a shorter route.

1. **Set $z=x+yi$** with $x,y$ real unknowns (do not assign them a value; they stay symbolic until solved for near the end).
2. **Equation (1) from the modulus condition.** Square $|z-z_0|=R$: this becomes $(x-x_0)^2+(y-y_0)^2-R^2=0$, i.e. an expression in $x,y$ containing the cluster $x^2+y^2$. Compute this expanded expression with the Calculator (`expand(...)`) and store it.
3. **Equation (2) from the pure-imaginary condition.** Multiply the numerator $z=x+yi$ by the conjugate of the denominator, $(x-c)-yi$ (never divide directly; division must always be cleared this way first). Expand this product with the Calculator. In the result, the terms **without** `I` are the real part; the terms **with** a factor of `I` are the imaginary part.
   - Set the real part equal to $0$: this is equation (2), and it also contains the cluster $x^2+y^2$.
   - The imaginary part will be $-c \cdot y$ (a real number times $y$), which must be **non-zero** since $c\ne0$; this is the **exclusion condition** $y \ne 0$, which you must remember for the final filtering step.
   - The denominator must be non-zero too; this is a second exclusion condition, $x \ne c$.
4. **Eliminate $x^2+y^2$.** Subtract equation (2) from equation (1) with the Calculator (`expand(eq1 - eq2)`). Since both share the same $x^2+y^2$ cluster, it cancels, leaving a **linear** equation in $x,y$.
5. **Isolate $y$.** Use the Calculator's equation solver on that linear equation to express $y$ as a formula in $x$ (`solve(...)`, exactly like using a calculator's equation-solving mode).
6. **Substitute back into equation (2)** to eliminate $y$, giving a single quadratic equation in $x$ only. Use the Calculator's equation solver again to find the numeric roots of $x$.
7. **Recover $y$ for each root of $x$** using the linear formula from step 5.
8. **Filter.** For every candidate pair $(x,y)$, check both exclusion conditions from step 3 ($x\ne c$ and $y\ne0$) using an exact equality test. Discard any pair where either condition is violated (i.e. where the equality test returns `True`, meaning it hits the excluded value).
9. **Count and match.** Store the number of surviving pairs as `count` with the Calculator. Then, for each of the four options in order A, B, C, D: if the option is the word "infinite" (Vô số), it cannot match a finite count, so skip it without a Calculator call; otherwise test `Eq(count, <that option's number>)` and stop at the first `True`. Never decide the matching letter by eye; a correct count matched to the wrong letter is still a wrong final answer.
'''


# =====================================================================
# [D] PART_FEWSHOT - VI DU MAU RIENG CUA DANG D35 (THAY KHI DOI DANG)
# Cac dong "Observation:" duoi day la KET QUA THAT (chay bang chinh class
# MayTinh o backend, khong phai sympy roi rac va khong phai so bia).
# =====================================================================
PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Có bao nhiêu số phức $z$ thỏa mãn $|z-3i|=5$ và $\dfrac{z}{z-4}$ là số thuần ảo?
A. $0$ | B. Vô số | C. $1$ | D. $2$

<think>
Here $z_0 = 3i$ (so $x_0=0, y_0=3$), $R=5$, and the denominator constant is $c=4$.

Thought: Set z=x+yi and build equation (1) from |z-3i|=5.
Action: Calculator
Action Input: eq1 = expand(x**2 + (y-3)**2 - 25)
Observation: eq1 = x**2 + y**2 - 6*y - 16

Thought: Multiply numerator by conjugate of denominator (x-4)-yi, and expand fully.
Action: Calculator
Action Input: numerator = expand((x+y*I)*((x-4)-y*I))
Observation: numerator = x**2 - 4*x + y**2 - 4*I*y

Thought: Reading numerator: the part without I is x**2-4*x+y**2 (this is equation 2, set to 0); the part with I has coefficient -4*y, so the exclusion condition is y != 0. The denominator x-4 must also be nonzero, so x != 4.
Action: Calculator
Action Input: eq2 = x**2 - 4*x + y**2
Observation: eq2 = x**2 - 4*x + y**2

Thought: Subtract eq2 from eq1 to eliminate the x**2+y**2 cluster.
Action: Calculator
Action Input: diff = expand(eq1 - eq2)
Observation: diff = 4*x - 6*y - 16

Thought: Solve this linear equation for y in terms of x.
Action: Calculator
Action Input: y_formula = solve(diff, y)[0]
Observation: y_formula = 2*x/3 - 8/3

Thought: Substitute this into eq2 to get a quadratic purely in x.
Action: Calculator
Action Input: quadratic = expand(eq2.subs(y, y_formula))
Observation: quadratic = 13*x**2/9 - 68*x/9 + 64/9

Thought: Solve the quadratic for the numeric roots of x.
Action: Calculator
Action Input: roots_x = solve(quadratic, x)
Observation: roots_x = [16/13, 4]

Thought: Recover x1 and its matching y1.
Action: Calculator
Action Input: x1 = roots_x[0]
Observation: x1 = 16/13

Thought: Compute y1 from the linear formula.
Action: Calculator
Action Input: y1 = y_formula.subs(x, x1)
Observation: y1 = -24/13

Thought: Recover x2 and its matching y2.
Action: Calculator
Action Input: x2 = roots_x[1]
Observation: x2 = 4

Thought: Compute y2 from the linear formula.
Action: Calculator
Action Input: y2 = y_formula.subs(x, x2)
Observation: y2 = 0

Thought: Check if pair 1 (x1,y1) violates an exclusion condition (x=4 or y=0).
Action: Calculator
Action Input: Eq(x1, 4)
Observation: False

Thought: Also check y1 != 0 for pair 1.
Action: Calculator
Action Input: Eq(y1, 0)
Observation: False

Thought: Pair 1 violates neither condition, so it is VALID. Now check pair 2 (x2,y2).
Action: Calculator
Action Input: Eq(x2, 4)
Observation: True

Thought: Pair 2 has x2=4, which violates the exclusion condition, so pair 2 is DISCARDED regardless of y2. Only pair 1 survives.
Action: Calculator
Action Input: count = 1
Observation: count = 1

Thought: Option A is 0. Test it.
Action: Calculator
Action Input: Eq(count, 0)
Observation: False

Thought: Option B is "Vô số" (infinite), which cannot match a finite count; skip without a Calculator call.

Thought: Option C is 1. Test it.
Action: Calculator
Action Input: Eq(count, 1)
Observation: True

Option C matches; stop here, no need to test D.
</think>
Final Answer: \boxed{C}
'''


# =====================================================================
# [E] PART_NHIEMVU - CHOT NHIEM VU (sua danh sach buoc khi doi dang)
# =====================================================================
PART_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Identify $z_0$, $R$ from the modulus condition and $c$ from the denominator. Build `eq1 = expand(...)` for equation (1).
[Step 2] Build `numerator = expand((x+y*I)*((x-c)-y*I))`. Read off the part without `I` (real part) and the coefficient of `I` (imaginary part) by eye.
[Step 3] Store `eq2 = <the real part you read>` (equation 2). Note the exclusion conditions: $x\ne c$ and (imaginary-part coefficient) $\ne 0$.
[Step 4] Compute `diff = expand(eq1 - eq2)`.
[Step 5] Compute `y_formula = solve(diff, y)[0]`.
[Step 6] Compute `quadratic = expand(eq2.subs(y, y_formula))`, then `roots_x = solve(quadratic, x)`.
[Step 7] For each root in `roots_x`, recover $x_i$ and $y_i = $ `y_formula.subs(x, x_i)`.
[Step 8] For each pair, test both exclusion conditions with `Eq(...)` and discard the pair if either returns `True`.
[Step 9] Store `count = <number of surviving pairs>` with the Calculator, then test the options in order with `Eq(count, <option>)`; skip any "Vô số" (infinite) option without a Calculator call; and stop at the first `True`. Do not pick the letter by eye.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''

PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

In [ ]:
# BACKEND: TOOL "Calculator" - may tinh sympy CO BO NHO BIEN
# =====================================================================
# Tong quat cho MOI dang toan (khong chi D53): nhan 1 bieu thuc sympy bat
# ky, tra ve gia tri chinh xac tuyet doi. Ho tro:
#   - gan bien:  "ten = bieu_thuc"  -> luu vao bo nho, dung lai o luot sau
#     (xoa han lo hoi model chep tay lai so dai - nguon loi lon nhat)
#   - so khop :  "Eq(a, b)"         -> True/False chinh xac, khong xap xi
#   - moi phep cong tru nhan chia phan so, can thuc, so phuc, mo dun, lien hop
import sympy as sp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            parsed = sp.sympify(s, locals=self.ns)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau = sp.simplify(parsed.lhs - parsed.rhs) == 0
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri = sp.expand(sp.simplify(parsed))
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'


# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 25
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        # EP SAN token dau tien (giong ban 1 cau): model bi buoc noi tiep tu
        # "Thought:" ngay sau <think>, khong con quyen tu chon viet van xuoi
        # mo dau (hanh vi mac dinh de lech khoi dinh dang ReAct).
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        khop = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            khop = mm
        bieu_thuc = khop.group(1).strip() if khop else ''

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)

## Dang D36

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

OUT_PATH  = '/kaggle/working/d36_react_calculator_full90_DeepSeek-R1-Distill-Qwen-1.5B.csv'

MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = {'x', 'y'}  # x,y: phan thuc/ao cua z=x+yi

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/11-dang-full/plan_solve_prompts_11dang_merged.json'  # SUA NEU KHAC
with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D36']
print('So cau:', len(records))

# ---- Backend Calculator: dat O DAY, TRUOC khi LLM(...)/CUDA khoi tao ----
# (an toan multiprocessing.fork - xem giai thich trong comment ben duoi)
import sympy as sp
import multiprocessing as mp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Luoi an toan CHO CA BATCH: du da dung radsimp() de tranh treo may o hau
# het truong hop, sympy van khong dam bao toc do cho MOI to hop can thuc
# bat ky - chi can 1/90 cau roi vao truong hop xau la ca batch nghen theo
# (Calculator chay tuan tu tung cau). Dung TIEN TRINH CON that (co the bi
# giet cuong buc bang tin hieu he dieu hanh) thay vi thread: thread chi
# ngat duoc tai diem GIL duoc nhuong lai, KHONG dam bao neu tinh toan ket
# sau trong 1 loi goi C lien tuc (da xac nhan qua thuc te chay tren
# Kaggle). Pool tien trinh con nay duoc tao NGAY TAI DAY, TRUOC KHI
# LLM(...)/CUDA khoi tao ben duoi - vi vay an toan tuyet doi voi
# multiprocessing.fork (fork() SAU KHI CUDA da khoi tao moi la nguy hiem,
# tung gay treo o mot lan thu truoc do).
TIMEOUT_SECONDS = 20


def fast_simplify(expr):
    """Rut gon nhanh va ON DINH hon sp.simplify() thuan tuy.

    sp.simplify() la ham "thu tat ca chien luoc roi chon ket qua ngan nhat",
    rat cham (co the treo may) khi bieu thuc co nhieu MAU SO chua can bac
    hai khac goc (vd tu phep chia (Bx-Ay)/(Bx-Ax)) - dung sinh ra qua nhieu
    dang the hien khac nhau ma khong bao gio hop nhat lai. radsimp() giai
    quyet dung goc van de nay: no huu ti hoa mau so chua can NGAY LAP TUC,
    nen ket qua o moi buoc luon o dang gon, khong de cac mau can long tich
    luy qua tung phep tinh tiep theo. Dung radsimp() lam buoc rut gon CHINH
    (nhanh, gan nhu luon du); chi roi sang simplify() lam buoc du phong khi
    radsimp() chua dua duoc ve dang 0/dang gon nhat.
    """
    return sp.expand(sp.radsimp(expr))


def is_zero(expr):
    """Kiem tra bieu thuc co bang 0 khong.

    Buoc dau (radsimp) bat duoc phan lon truong hop that nhanh va tuyet
    doi chinh xac. Neu chua ket luan duoc, KHONG roi sang sp.simplify()
    (chung minh dai so - cham, khong dam bao toc do, day la duong tung
    gay treo may). Thay vao do, so sanh gia tri SO HOC voi do chinh xac
    rat cao (50 chu so thap phan) - dung nguyen tac may tinh Casio: so
    hai so thap phan thay vi chung minh dang thuc dai so. Voi mien bai
    toan nay (cac hang so dai so co dinh tu de bai, khong phai gia tri
    adversarial), sai so gan nhu khong the xay ra o do chinh xac nay.
    """
    rut_gon = sp.radsimp(expr)
    if rut_gon == 0:
        return True
    return abs(complex(rut_gon.evalf(50))) < 1e-40


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            # 'Eq' duoc thay bang phien ban evaluate=False: sp.Eq() mac dinh
            # TU DONG thu kiem tra 2 ve co bang nhau NGAY LUC KHOI TAO (co che
            # rieng cua sympy, khac hoan toan ham is_zero() tu viet ben duoi) -
            # voi bieu thuc can long phuc tap, chinh buoc TU DONG nay co the
            # treo may, va treo TRUOC CA KHI chay_co_timeout kip can thiep (vi
            # no xay ra ngay trong luc sympify dang parse chuoi). Dung ban
            # evaluate=False de hoan toan doi viec so khop cho ham is_zero() -
            # da duoc kiem chung nhanh va dang tin cay - dam nhiem.
            ns_de_parse = dict(self.ns)
            ns_de_parse['Eq'] = lambda a, b: sp.Eq(a, b, evaluate=False)
            parsed = sp.sympify(s, locals=ns_de_parse)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau, loi_timeout = chay_co_timeout(is_zero, parsed.lhs - parsed.rhs)
                if loi_timeout:
                    return None, loi_timeout
                ket_qua_bool = sp.true if bang_nhau else sp.false
                if ten:
                    self.ns[ten] = ket_qua_bool
                    return f'{ten} = {ket_qua_bool}', None
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri, loi_timeout = chay_co_timeout(fast_simplify, parsed)
            if loi_timeout:
                return None, loi_timeout
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

try:
    _CALC_POOL = mp.get_context('fork').Pool(1)
except ValueError:
    _CALC_POOL = None  # khong co fork (vd may local Windows) -> chay khong timeout


def chay_co_timeout(ham, *args):
    """Chay ham(*args) trong tien trinh con (fork, tao TRUOC CUDA nen an
    toan), gioi han TIMEOUT_SECONDS. Neu qua han, HUY va TAO LAI pool (vi
    tien trinh con cu van con chay ngam, khong the tai su dung duoc nua)
    roi tra ve loi ro rang thay vi treo may."""
    global _CALC_POOL
    if _CALC_POOL is None:
        return ham(*args), None
    ar = _CALC_POOL.apply_async(ham, args)
    try:
        return ar.get(timeout=TIMEOUT_SECONDS), None
    except mp.TimeoutError:
        _CALC_POOL.terminate()
        _CALC_POOL = mp.get_context('fork').Pool(1)
        return None, (f'computation timed out after {TIMEOUT_SECONDS}s '
                      '(the expression is too complex to simplify exactly). '
                      'Do not resend the exact same expression; try continuing '
                      'with a different, smaller step instead.')


# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) nhieu lan trong cung kernel tung gay loi GPU het bo
# nho (da xac nhan qua thuc te chay tren Kaggle o ban 5 dang truoc).
print('Dung lai tok/llm da nap san.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D36_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai PART_HUONGGIAI va PART_FEWSHOT,
# giu nguyen PART_TOOL va PART_KIENTHUC.

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- If a stored result is a LIST (e.g. from `solve(...)` with more than one solution), you can reference one element of it directly by index instead of retyping it: `name = solve(...)` then `name[0]` for the first element, `name[1]` for the second, and so on; use this the moment you need one specific element again in a later expression (e.g. `x_expr.subs(y, name[0])`). Retyping a long value from an `Observation` by hand (especially one with nested radicals) is exactly where a digit or a sign gets copied wrong; indexing the stored list can never have that problem, so prefer it every time.
- You can also build a list yourself, by hand, not only receive one from `solve(...)`: write `name = [val1, val2, val3, val4]` to store several numbers together in a single call, then index into it the same way (`name[0]`, `name[1]`, ...; see PART 4 for when to use this).
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. For most problem types this is decided by an exact symbolic proof. For this problem type specifically, it is decided by evaluating both sides to 50 significant decimal digits and checking they agree to that precision; this is not a formal proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True`; test all four, every time (see PART 4's matching step for why). Just like any other computation, `Eq(...)` can be given a name too, e.g. `test_A = Eq(left, right)`; the `True`/`False` result is then genuinely stored under that name and can be referred to again later by that name, exactly like a numeric result.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Every equation in this problem type is at most quadratic (in $y$, after substituting the line), so `solve(...)` always returns an exact closed form; a numeric-only fallback is never needed here.

**NEVER solve a system of equations all at once.** Calling `solve([eq1, eq2], [x, y])` with a LIST of equations and a LIST of unknowns is unreliable: depending on the exact expressions involved, it can return a dictionary instead of a list, and this Calculator cannot handle a dictionary result (it will error). Instead, always solve ONE equation for ONE unknown at a time: solve the first equation for one unknown (giving a list with one expression, possibly in terms of the other unknown), substitute that expression into the second equation, then solve THAT for the remaining unknown. This always returns a plain list, exactly like every other use of `solve(...)` already covered above.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo; retyping a small variation of the same idea will keep producing the same error. Stop and re-read PART 2's method for this exact step before trying again, rather than guessing another small variation of the same idea; never invent a placeholder word like `undefined` as if it were a value, since the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$; there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change; so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).
20. **Vieta's formulas for a quadratic.** For $z^2+pz+q=0$ with roots $z_1,z_2$ (real or complex), the coefficients determine the roots' sum and product directly, without solving anything: $z_1+z_2=-p$ and $z_1z_2=q$. This holds whether the roots are real or a complex-conjugate pair (fact 15); the SAME two equations hold either way, since they simply come from matching coefficients in the identity $z^2+pz+q=(z-z_1)(z-z_2)=z^2-(z_1+z_2)z+z_1z_2$, which does not care whether $z_1,z_2$ happen to be real.
21. **Conjugate of a sum, and conjugate of a real multiple of $i$.** For any complex numbers $u,v$: $\overline{u+v}=\overline{u}+\overline{v}$ and $\overline{u-v}=\overline{u}-\overline{v}$ (conjugation distributes over addition/subtraction, since conjugating just flips the sign of every imaginary part, and imaginary parts add/subtract termwise). In particular, for a REAL constant $c$: $\overline{ci}=-ci$ (because $ci$ is purely imaginary with imaginary part $c$, flipping its sign gives $-ci$). Combining these: for any complex $z$ and real constant $c$, $\overline{z+ci}=\overline{z}+\overline{ci}=\overline{z}-ci$. This is the key trick that lets you replace $\overline{z}-ci$ by $\overline{z+ci}$ wherever it appears, which (by fact 4, $|\overline{w}|=|w|$) means $|\overline{z}-ci|=|z+ci|$; turning an expression that LOOKS like it needs both $z$ and $\overline{z}$ separately into one that depends only on the single quantity $z+ci$.
22. **Sum and difference of a complex number and its conjugate.** For $z=x+yi$ ($x,y$ real): combining fact 1 ($z=x+yi$) and fact 3 ($\overline{z}=x-yi$) directly by addition and subtraction gives $z+\overline{z}=2x$ and $z-\overline{z}=2yi$. This is the key move whenever a condition mixes $z$ and $\overline{z}$ through a sum or difference: $z+\overline{z}$ collapses to the single REAL number $2x$ (twice the real part, with no $y$ or $i$ left in it at all), and $z-\overline{z}$ collapses to the single PURELY IMAGINARY number $2yi$ (so $|z-\overline{z}|=2|y|$).
'''


PART_HUONGGIAI = r'''
**PART 2: HOW TO SOLVE THIS PROBLEM TYPE**

**Turning both conditions into plain equations in $x,y$.** Write $z=x+yi$ with $x,y$ real (fact 1). By fact 22, $z+\overline{z}=2x$, and by fact 4, $|z|^2=x^2+y^2$. So the first condition $|z|^2=K|z+\overline{z}|+C$ becomes $x^2+y^2=K|2x|+C=2K|x|+C$: an equation with no $z$ or $\overline{z}$ left in it, just $x,y$, except for that one absolute value around $x$.

**Splitting on the sign of $x$: an absolute value always forces a case split.** $|x|=x$ when $x\ge0$ and $|x|=-x$ when $x<0$, so the single equation $x^2+y^2=2K|x|+C$ is really two DIFFERENT equations depending on which half-plane $x$ falls in:
* If $x\ge0$: $x^2-2Kx+y^2-C=0$.
* If $x<0$: $x^2+2Kx+y^2-C=0$.
Each is the equation of a circle (centered at $(K,0)$ and $(-K,0)$ respectively); but only the half of that circle actually lying in the matching half-plane counts; a point computed from the wrong branch's formula must be checked against ITS OWN sign condition before being trusted, never assumed.

**The second condition, read directly: no abstraction needed.** A condition like $|z-A+Bi|=|z-C+Di|$ (for whatever real numbers $A,B,C,D$ are shown, with whatever signs) turns into a plain equation the moment you substitute $z=x+yi$: $z-A+Bi=(x-A)+(y+B)i$ directly; read each sign exactly as displayed, there is no need to match it against any template or flip anything. Square both sides using the real/imaginary parts you just built: $(x-A)^2+(y+B)^2=(x-C)^2+(y+D)^2$. By fact 11, both sides contain the cluster $x^2+y^2$; solving this single equation for $x$ (in terms of $y$) eliminates that cluster automatically and gives a line $x=\dots$. (If a problem ever showed two points with the same $x$-coordinate, this would fail to solve for $x$; fact 19 explains that case: solve for $y$ in terms of $x$ instead. This does not happen in this problem family, but it is worth knowing why.)

**Why both branches always have to be worked out in full, no matter what the first one gives.** The split in the previous paragraph came from $x\ge0$ and $x<0$; together these cover every real number exactly once, with no overlap and no gap between them. That is what makes the two branches genuinely INDEPENDENT problems rather than two attempts at the same problem: the $x\ge0$ branch's equation only ever describes points with $x\ge0$, and the $x<0$ branch's equation only ever describes points with $x<0$, so nothing about how many valid points turn up in one branch can say anything about how many turn up in the other; they were built from different substitutions ($|x|=x$ versus $|x|=-x$) applied to a completely different half of the plane. Finding two valid candidates in the $x\ge0$ branch does not make the $x<0$ branch less likely to also have valid candidates, and finding zero valid candidates in the $x\ge0$ branch does not make the $x<0$ branch more likely to have some either; each branch's count of valid candidates depends only on where THAT branch's own circle happens to cross the line, a completely separate geometric question. So solving and checking one branch is never a substitute for solving and checking the other; both must always be carried through to completion before the final count is known.

**Feeding the line into each half separately.** Substitute the line's expression for $x$ into the FIRST branch's circle equation ($x\ge0$'s version) and solve for $y$: this is now a single-unknown quadratic. Solving it can give TWO real roots, or it can give NONE (an empty list) when the line simply does not cross that particular half of the circle at all; both outcomes are normal and expected, not an error; an empty list just means this branch contributes 0 candidates, and you move directly to the other branch without trying to invent a root that is not there. Each root $y$ that IS returned gives a candidate point: recover $x$ from the line's formula. Do the exact same substitution into the SECOND branch's circle equation ($x<0$'s version) to get a second, independent list of $y$ roots (again possibly empty) and candidate points.

**Every candidate must be checked against its OWN branch's sign condition: never assumed.** A point coming from the $x\ge0$ branch's equation is only a genuine solution if its actual computed $x$ is $\ge0$; a point from the $x<0$ branch's equation is only genuine if its computed $x$ is $<0$ (note: exactly $x=0$ fails this second check, even though it passes the first; a point landing exactly on the boundary belongs only to the $x\ge0$ branch, never to the $x<0$ branch). Check each of the (up to 4) candidates individually; do not assume a branch's roots are automatically valid just because you solved its equation correctly; solving the equation only finds points ON that branch's full circle, not necessarily on the correct half of it.

**Totalling.** The final count is exactly the number of candidates (across both branches) that passed their own branch's sign check; simply count them; no separate overlap check is needed, since a point cannot simultaneously satisfy $x\ge0$ and $x<0$.
'''


PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (ReAct format)**

Đề bài: Có bao nhiêu số phức $z$ thỏa mãn $|z|^2 = 2|z+\overline{z}|+4$ và $|z-1-i| = |z-3+3i|$?
A. $4$.   B. $3$.   C. $1$.   D. $2$.

<think>
Thought: First condition, quoted verbatim: "$|z|^2=2|z+\overline{z}|+4$"; the coefficient is $2$, so $k=2$; the constant is $+4$, so $c=4$. I store them.
Action: Calculator
Action Input: k = 2
Observation: k = 2

Thought: Now read off $c$, the constant term.
Action: Calculator
Action Input: c = 4
Observation: c = 4

Thought: Second condition, quoted verbatim: "$|z-1-i|=|z-3+3i|$". Reading each sign exactly as shown: $z-1-i=(x-1)+(y-1)i$, and $z-3+3i=(x-3)+(y+3)i$. Squaring both sides and setting them equal gives the line's equation; solve it for $x$.
Action: Calculator
Action Input: solve(Eq((x-1)**2+(y-1)**2, (x-3)**2+(y+3)**2), x)
Observation: [2*y + 4]

Thought: So $x=2y+4$ on this line. Store it.
Action: Calculator
Action Input: x_expr = 2*y+4
Observation: x_expr = 2*y + 4

Thought: Before checking any candidate, start a running count at 0; I will update this variable after every single check instead of adding things up later from memory.
Action: Calculator
Action Input: count = 0
Observation: count = 0

Thought: Substitute into the $x\ge0$ branch's circle equation and solve for $y$. I store the whole list under a name so I can index into it afterward, instead of retyping any root by hand.
Action: Calculator
Action Input: roots1 = solve(Eq(x_expr**2 - 2*k*x_expr + y**2, c), y)
Observation: roots1 = [-2, 2/5]

Thought: This list has 2 roots, so branch 1 has exactly 2 candidates to check: `roots1[0]` and `roots1[1]`. Starting with `roots1[0]`; recover its $x$ directly by index, never by retyping the value.
Action: Calculator
Action Input: x_a = x_expr.subs(y, roots1[0])
Observation: x_a = 0

Thought: Check the $x\ge0$ condition for this candidate.
Action: Calculator
Action Input: x_a >= 0
Observation: True

Thought: Observation is True, so this candidate is valid; I now call `count = count + 1`.
Action: Calculator
Action Input: count = count + 1
Observation: count = 1

Thought: Now the second candidate from this branch: `roots1[1]`.
Action: Calculator
Action Input: x_b = x_expr.subs(y, roots1[1])
Observation: x_b = 24/5

Thought: Check the $x\ge0$ condition.
Action: Calculator
Action Input: x_b >= 0
Observation: True

Thought: Observation is True, so this candidate is valid; I now call `count = count + 1`.
Action: Calculator
Action Input: count = count + 1
Observation: count = 2

Thought: Branch 1's list had 2 roots, and both are now checked ($x_a$ valid, $x_b$ valid); branch 1 is complete. Moving to branch 2 next, which is mandatory regardless of what branch 1 found: substitute the line into its circle equation and solve for $y$, again storing the whole list under a name.
Action: Calculator
Action Input: roots2 = solve(Eq(x_expr**2 + 2*k*x_expr + y**2, c), y)
Observation: roots2 = [-14/5, -2]

Thought: This list has 2 roots, so branch 2 has exactly 2 candidates to check: `roots2[0]` and `roots2[1]`. Starting with `roots2[0]`.
Action: Calculator
Action Input: x_c = x_expr.subs(y, roots2[0])
Observation: x_c = -8/5

Thought: Check the $x<0$ condition for this candidate.
Action: Calculator
Action Input: x_c < 0
Observation: True

Thought: Observation is True, so this candidate is valid; I now call `count = count + 1`.
Action: Calculator
Action Input: count = count + 1
Observation: count = 3

Thought: Now the second candidate from this branch: `roots2[1]`.
Action: Calculator
Action Input: x_d = x_expr.subs(y, roots2[1])
Observation: x_d = 0

Thought: Check the $x<0$ condition for this candidate; note this is the SAME point as $x_a$, but here it must satisfy $x<0$, not $x\ge0$.
Action: Calculator
Action Input: x_d < 0
Observation: False

Thought: Observation is False, so this candidate is not valid; I do NOT call `count = count + 1` for it. Branch 2's list had 2 roots, and both are now checked ($x_c$ valid, $x_d$ invalid); branch 2 is complete. Branch 1's 2 roots were also both actually checked earlier ($x_a$, $x_b$, with real Action calls, not assumed). With both branches' real computations done, `count` now holds the final total; I query it to confirm.
Action: Calculator
Action Input: count
Observation: 3

Thought: `count` is confirmed to be $3$. Before testing anything, I read all four options verbatim from the problem text, in order A, B, C, D, and store them together in a single list, in that same order. This is the only time I read these values from the problem text; from now on I refer to each one only by index, never by retyping it.
Action: Calculator
Action Input: opts = [4, 3, 1, 2]
Observation: opts = [4, 3, 1, 2]

Thought: Now I go through the four options one at a time, carefully, in order, comparing `count` against the matching element of `opts` by index; I test every single one, even after a match is found, since `Eq()` here is a numeric check, not a formal proof. Starting with option A, which is `opts[0]`. Test it.
Action: Calculator
Action Input: Eq(count, opts[0])
Observation: False

Thought: A did not match. Option B is `opts[1]`. Test it.
Action: Calculator
Action Input: Eq(count, opts[1])
Observation: True

Thought: B matched - but I still do not stop, since I have not yet tested every option. Option C is `opts[2]`. Test it.
Action: Calculator
Action Input: Eq(count, opts[2])
Observation: False

Thought: C did not match. Option D is `opts[3]`. Test it.
Action: Calculator
Action Input: Eq(count, opts[3])
Observation: False

Thought: All four options are now tested. Exactly one came back True: option B (`opts[1]`). That is the answer.
</think>

Final Answer: \boxed{B}
'''


PART_NHIEMVU = r'''
**PART 4: YOUR TASK, STEP BY STEP**

The problem always has the form: how many complex numbers $z$ satisfy $|z|^2=K|z+\overline{z}|+C$ (for a given coefficient $K$ and constant $C$, $C$ possibly negative) AND $|z-A+Bi|=|z-C_2+D i|$ (for given real numbers, whatever signs are shown)? Follow these steps, using the Calculator for every computation.

1. **Finding $k$ and $c$.** Quote the first condition verbatim, then read off the coefficient of $|z+\overline{z}|$ as $k$ (if none is written, $k=1$), and the constant term as $c$, keeping whatever sign is shown. Store both.
2. **Building the line.** Quote the second condition verbatim. Reading each sign exactly as displayed (no flipping, no template-matching; substitute $z=x+yi$ directly into each side), write the two sides as $(x\pm A)^2+(y\pm B)^2$ and $(x\pm C_2)^2+(y\pm D)^2$. Solve `Eq(...)` of these two squared expressions for $x$ to get the line as `x_expr` (a formula in $y$).
3. **Start a running count.** Store `count = 0`. Every time, and only when, a candidate passes its branch's sign check below, update it immediately with `count = count + 1`; never sum several results later from memory.
4. **How to process ONE branch.** Substitute `x_expr` into that branch's circle equation and solve for $y$, storing the WHOLE list under a name (e.g. `roots1 = solve(...)`), even if it has 0, 1, or 2 roots (an empty list is not an error; it simply means this branch contributes nothing). In the Thought immediately before your next Action (never a standalone Thought with no Action of its own), state how many roots the list has. For EACH root, in order, indexed (`roots1[0]`, then `roots1[1]` if it exists; never retyped): compute the matching $x$, check the branch's own sign condition, and update `count` right away if it passes. Once every root is checked, fold a one-line confirmation into the Thought that introduces your NEXT Action (no extra Action just for this), e.g. "branch 1's 2 roots are all checked, 1 valid".
5. **Branch 1 ($x\ge0$).** Apply step 4 to $x^2-2kx+y^2=c$, checking `x >= 0`. Once branch 1's confirmation is stated, your very next Action, with nothing else in between, is branch 2's first computation, step 6's `roots2 = solve(...)`. Branch 1 finishing is never, by itself, a sign that branch 2 is also done, or that `count` is already final; branch 2 has not been computed at all yet, and skipping straight to querying `count` here would leave branch 2 entirely unchecked.
6. **Branch 2 ($x<0$): MANDATORY, regardless of what branch 1 found.** Apply step 4 to $x^2+2kx+y^2=c$, checking `x < 0` (a computed $x$ of exactly $0$ fails this, even if it passed branch 1's check for a different root). PART 2 explains why the two branches never predict each other's results: finishing branch 1, with any outcome at all, is never a reason to skip or shortcut branch 2.
7. Once both branches' closing confirmations (from step 4, applied once per branch) have been stated, `count` holds the final total; there is no separate summing step. Your very next Action, with nothing else in between and no Final Answer yet, is step 8: storing the four options. Reaching a final `count` is never, by itself, a reason to stop; the options have not been read or tested yet.
8. **Storing all four options before testing any of them.** Read all four options' values VERBATIM from the problem text, in order A, B, C, D, and store them together in ONE Calculator call as a single list: `opts = [<A>, <B>, <C>, <D>]`. This is the only time you ever read these values from the problem text; from here on, refer to each one only by index (`opts[0]` for A, `opts[1]` for B, `opts[2]` for C, `opts[3]` for D), never by retyping a number.
9. For each option A, B, C, D in turn: compare with `Eq(count, opts[<matching index>])`. Test every option even after one already returns `True`; `Eq()` here is a numeric check, not a formal proof.
10. Write the final line `Final Answer: \boxed{<Letter>}` using whichever option matched.

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

In [ ]:
# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# (MayTinh/fast_simplify/is_zero da duoc dinh nghia o Cell 3, truoc khi
# LLM(...) khoi tao - xem giai thich an toan CUDA/fork o do)
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 36          # quy trinh day du ~20 luot tinh (k,c,solve-duong,x_expr,count=0,solve-TH1(list),4x(x-index+check[+count++]),solve-TH2(list),4x(x-index+check[+count++])) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')
FINAL_ANSWER_RE = re.compile(r'Final\s+Answer\s*:\s*\\boxed\{\s*[A-D]\s*\}')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        # EP SAN token dau tien (giong ban 1 cau): model bi buoc noi tiep tu
        # "Thought:" ngay sau <think>, khong con quyen tu chon viet van xuoi
        # mo dau (hanh vi mac dinh de lech khoi dinh dang ReAct).
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        # CHOT CUNG: mot so lan model viet xong Final Answer nhung khong
        # dung sinh sach (khong phat EOS), roi viet tiep 1 vong giai lai
        # tu dau, ket thuc vong do bang "Observation:" moi khien harness
        # tuong la 1 luot goi tool that va lap tiep. Cat ngay tai cho
        # Final Answer XUAT HIEN LAN DAU, bat ke stop_reason la gi.
        m_final = FINAL_ANSWER_RE.search(s.full_text)
        if m_final:
            s.full_text = s.full_text[:m_final.end()]
            s.xong = True
            s.ly_do_dung = 'model_ket_thuc'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        khop = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            khop = mm
        bieu_thuc = khop.group(1).strip() if khop else ''

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)

## Dang D37

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

OUT_PATH  = '/kaggle/working/d37_react_calculator_full90_DeepSeek-R1-Distill-Qwen-1.5B.csv'

MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = {'u'}  # u = t^2 = |z|^2, an so trung gian can giai

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/11-dang-full/plan_solve_prompts_11dang_merged.json'  # SUA NEU KHAC
with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D37']
print('So cau:', len(records))

# ---- Backend Calculator: dat O DAY, TRUOC khi LLM(...)/CUDA khoi tao ----
# (an toan multiprocessing.fork - xem giai thich trong comment ben duoi)
import sympy as sp
import multiprocessing as mp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Luoi an toan CHO CA BATCH: du da dung radsimp() de tranh treo may o hau
# het truong hop, sympy van khong dam bao toc do cho MOI to hop can thuc
# bat ky - chi can 1/90 cau roi vao truong hop xau la ca batch nghen theo
# (Calculator chay tuan tu tung cau). Dung TIEN TRINH CON that (co the bi
# giet cuong buc bang tin hieu he dieu hanh) thay vi thread: thread chi
# ngat duoc tai diem GIL duoc nhuong lai, KHONG dam bao neu tinh toan ket
# sau trong 1 loi goi C lien tuc (da xac nhan qua thuc te chay tren
# Kaggle). Pool tien trinh con nay duoc tao NGAY TAI DAY, TRUOC KHI
# LLM(...)/CUDA khoi tao ben duoi - vi vay an toan tuyet doi voi
# multiprocessing.fork (fork() SAU KHI CUDA da khoi tao moi la nguy hiem,
# tung gay treo o mot lan thu truoc do).
TIMEOUT_SECONDS = 20


def fast_simplify(expr):
    """Rut gon nhanh va ON DINH hon sp.simplify() thuan tuy.

    sp.simplify() la ham "thu tat ca chien luoc roi chon ket qua ngan nhat",
    rat cham (co the treo may) khi bieu thuc co nhieu MAU SO chua can bac
    hai khac goc (vd tu phep chia (Bx-Ay)/(Bx-Ax)) - dung sinh ra qua nhieu
    dang the hien khac nhau ma khong bao gio hop nhat lai. radsimp() giai
    quyet dung goc van de nay: no huu ti hoa mau so chua can NGAY LAP TUC,
    nen ket qua o moi buoc luon o dang gon, khong de cac mau can long tich
    luy qua tung phep tinh tiep theo. Dung radsimp() lam buoc rut gon CHINH
    (nhanh, gan nhu luon du); chi roi sang simplify() lam buoc du phong khi
    radsimp() chua dua duoc ve dang 0/dang gon nhat.
    """
    return sp.expand(sp.radsimp(expr))


def is_zero(expr):
    """Kiem tra bieu thuc co bang 0 khong.

    Buoc dau (radsimp) bat duoc phan lon truong hop that nhanh va tuyet
    doi chinh xac. Neu chua ket luan duoc, KHONG roi sang sp.simplify()
    (chung minh dai so - cham, khong dam bao toc do, day la duong tung
    gay treo may). Thay vao do, so sanh gia tri SO HOC voi do chinh xac
    rat cao (50 chu so thap phan) - dung nguyen tac may tinh Casio: so
    hai so thap phan thay vi chung minh dang thuc dai so. Voi mien bai
    toan nay (cac hang so dai so co dinh tu de bai, khong phai gia tri
    adversarial), sai so gan nhu khong the xay ra o do chinh xac nay.
    """
    rut_gon = sp.radsimp(expr)
    if rut_gon == 0:
        return True
    return abs(complex(rut_gon.evalf(50))) < 1e-40


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong, VA
        ten HAM chua dinh nghia thanh mot AppliedUndef chua tinh (vd goi
        "evalf(...)" - khong phai ham that) - ca 2 deu khong bao loi, ket
        qua se vo nghia hoac khong tien trien ma model khong he hay biet.
        Chan lai moi ten bien KHONG nam trong bo nho VA khong nam trong
        danh sach an so tu do duoc khai bao truoc, VA moi loi goi ham chua
        duoc dinh nghia."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        ham_la = {str(h.func) for h in getattr(bieu_thuc, 'atoms', lambda *a: set())(sp.core.function.AppliedUndef)}
        if not ten_thieu and not ham_la:
            return None
        if ham_la:
            ham_la_str = ', '.join(sorted(ham_la))
            return (f'undefined function name(s): {ham_la_str}. This is not a '
                    'recognized Calculator function. Only use the functions '
                    'documented in the tool instructions (sqrt, Abs, conjugate, '
                    'solve, Eq, nroots, re, im, expand, and ordinary +-*/**); do '
                    'not invent or guess a function name that is not listed there.')
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            # 'Eq' duoc thay bang phien ban evaluate=False: sp.Eq() mac dinh
            # TU DONG thu kiem tra 2 ve co bang nhau NGAY LUC KHOI TAO (co che
            # rieng cua sympy, khac hoan toan ham is_zero() tu viet ben duoi) -
            # voi bieu thuc can long phuc tap, chinh buoc TU DONG nay co the
            # treo may, va treo TRUOC CA KHI chay_co_timeout kip can thiep (vi
            # no xay ra ngay trong luc sympify dang parse chuoi). Dung ban
            # evaluate=False de hoan toan doi viec so khop cho ham is_zero() -
            # da duoc kiem chung nhanh va dang tin cay - dam nhiem.
            ns_de_parse = dict(self.ns)
            ns_de_parse['Eq'] = lambda a, b: sp.Eq(a, b, evaluate=False)
            parsed = sp.sympify(s, locals=ns_de_parse)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau, loi_timeout = chay_co_timeout(is_zero, parsed.lhs - parsed.rhs)
                if loi_timeout:
                    return None, loi_timeout
                ket_qua_bool = sp.true if bang_nhau else sp.false
                if ten:
                    self.ns[ten] = ket_qua_bool
                    return f'{ten} = {ket_qua_bool}', None
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri, loi_timeout = chay_co_timeout(fast_simplify, parsed)
            if loi_timeout:
                return None, loi_timeout
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

try:
    _CALC_POOL = mp.get_context('fork').Pool(1)
except ValueError:
    _CALC_POOL = None  # khong co fork (vd may local Windows) -> chay khong timeout


def chay_co_timeout(ham, *args):
    """Chay ham(*args) trong tien trinh con (fork, tao TRUOC CUDA nen an
    toan), gioi han TIMEOUT_SECONDS. Neu qua han, HUY va TAO LAI pool (vi
    tien trinh con cu van con chay ngam, khong the tai su dung duoc nua)
    roi tra ve loi ro rang thay vi treo may."""
    global _CALC_POOL
    if _CALC_POOL is None:
        return ham(*args), None
    ar = _CALC_POOL.apply_async(ham, args)
    try:
        return ar.get(timeout=TIMEOUT_SECONDS), None
    except mp.TimeoutError:
        _CALC_POOL.terminate()
        _CALC_POOL = mp.get_context('fork').Pool(1)
        return None, (f'computation timed out after {TIMEOUT_SECONDS}s '
                      '(the expression is too complex to simplify exactly). '
                      'Do not resend the exact same expression; try continuing '
                      'with a different, smaller step instead.')


# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) nhieu lan trong cung kernel tung gay loi GPU het bo
# nho (da xac nhan qua thuc te chay tren Kaggle o ban 5 dang truoc).
print('Dung lai tok/llm da nap san.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D37_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai PART_HUONGGIAI va PART_FEWSHOT,
# giu nguyen PART_TOOL va PART_KIENTHUC.

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- If a stored result is a LIST (e.g. from `solve(...)` with more than one solution), you can reference one element of it directly by index instead of retyping it: `name = solve(...)` then `name[0]` for the first element, `name[1]` for the second, and so on; use this the moment you need one specific element again in a later expression (e.g. `x_expr.subs(y, name[0])`). Retyping a long value from an `Observation` by hand (especially one with nested radicals) is exactly where a digit or a sign gets copied wrong; indexing the stored list can never have that problem, so prefer it every time.
- You can also build a list yourself, by hand, not only receive one from `solve(...)`: write `name = [val1, val2, val3, val4]` to store several numbers together in a single call, then index into it the same way (`name[0]`, `name[1]`, ...; see PART 4 for when to use this).
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. For most problem types this is decided by an exact symbolic proof. For this problem type specifically, it is decided by evaluating both sides to 50 significant decimal digits and checking they agree to that precision; this is not a formal proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True`; test all four, every time (see PART 4's matching step for why).
- To test a numeric inequality, write it directly with `<` or `>`, exactly like ordinary math notation: `a < b`, `a > b`, or even a chained double inequality in one call, `a < x < b`. The `Observation` will be `True` or `False`, decided the same way as `Eq()` (high-precision numeric comparison, not a symbolic proof). This works for any concrete numbers or stored names, including ones holding a nested radical.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Every equation in this problem type is at most quadratic (in $u$, after the substitution $u=t^2$), so `solve(...)` always returns an exact closed form; a numeric-only fallback is never needed here.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo; retyping a small variation of the same idea will keep producing the same error. Stop and re-read PART 2's method for this exact step before trying again, rather than guessing another small variation of the same idea; never invent a placeholder word like `undefined` as if it were a value, since the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$; there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change; so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).
20. **Vieta's formulas for a quadratic.** For $z^2+pz+q=0$ with roots $z_1,z_2$ (real or complex), the coefficients determine the roots' sum and product directly, without solving anything: $z_1+z_2=-p$ and $z_1z_2=q$. This holds whether the roots are real or a complex-conjugate pair (fact 15); the SAME two equations hold either way, since they simply come from matching coefficients in the identity $z^2+pz+q=(z-z_1)(z-z_2)=z^2-(z_1+z_2)z+z_1z_2$, which does not care whether $z_1,z_2$ happen to be real.
21. **Conjugate of a sum, and conjugate of a real multiple of $i$.** For any complex numbers $u,v$: $\overline{u+v}=\overline{u}+\overline{v}$ and $\overline{u-v}=\overline{u}-\overline{v}$ (conjugation distributes over addition/subtraction, since conjugating just flips the sign of every imaginary part, and imaginary parts add/subtract termwise). In particular, for a REAL constant $c$: $\overline{ci}=-ci$ (because $ci$ is purely imaginary with imaginary part $c$, flipping its sign gives $-ci$). Combining these: for any complex $z$ and real constant $c$, $\overline{z+ci}=\overline{z}+\overline{ci}=\overline{z}-ci$. This is the key trick that lets you replace $\overline{z}-ci$ by $\overline{z+ci}$ wherever it appears, which (by fact 4, $|\overline{w}|=|w|$) means $|\overline{z}-ci|=|z+ci|$; turning an expression that LOOKS like it needs both $z$ and $\overline{z}$ separately into one that depends only on the single quantity $z+ci$.
22. **Sum and difference of a complex number and its conjugate.** For $z=x+yi$ ($x,y$ real): combining fact 1 ($z=x+yi$) and fact 3 ($\overline{z}=x-yi$) directly by addition and subtraction gives $z+\overline{z}=2x$ and $z-\overline{z}=2yi$. This is the key move whenever a condition mixes $z$ and $\overline{z}$ through a sum or difference: $z+\overline{z}$ collapses to the single REAL number $2x$ (twice the real part, with no $y$ or $i$ left in it at all), and $z-\overline{z}$ collapses to the single PURELY IMAGINARY number $2yi$ (so $|z-\overline{z}|=2|y|$).
'''


PART_HUONGGIAI = r'''
**PART 2: HOW TO SOLVE THIS PROBLEM TYPE**

**Recognizing the shape, and the hidden link between its two constants.** The given equation always has the form $w|z|=\dfrac{K}{z}+(\text{a free complex constant})$, where $w=a+bi$ is the (constant, given) coefficient multiplying $|z|$ on the left, and $K$ is the (constant, given) numerator on the right. The free complex constant on the right is never independent of $w$: it is always EXACTLY $w \cdot i$. This is not a coincidence to verify each time; it is how this problem type is built, and it is the reason the equation can be factored at all. Multiplying $w$ by $i$ rotates it $90°$ (fact 2), turning $a+bi$ into $-b+ai$; recognizing that the free constant printed in the problem is precisely this rotated $w$ is what makes the next step possible.

**Factoring to isolate a single copy of $|z|-i$.** Since the free constant equals $w\cdot i$, the equation $w|z|=\dfrac{K}{z}+wi$ rearranges (moving $wi$ to the left) to $w|z|-wi=\dfrac{K}{z}$, and factoring $w$ out of the left side gives $w(|z|-i)=\dfrac{K}{z}$. Everything involving the unknown $z$ is now trapped inside two places only: the modulus $|z|$ (inside the factor $|z|-i$) and the lone $z$ in the denominator on the right.

**Taking the modulus of both sides.** By fact 12, $|uv|=|u|\cdot|v|$, and dividing works the same way for moduli: $\left|\dfrac{K}{z}\right|=\dfrac{|K|}{|z|}$. Applying this to $w(|z|-i)=\dfrac{K}{z}$ gives $|w|\cdot\big||z|-i\big|=\dfrac{|K|}{|z|}$. Every quantity here is now a plain nonnegative real number: $|w|$ and $|K|$ are fixed numbers computable directly from the problem's constants, and $|z|$ (which appears twice, once alone and once inside $\big||z|-i\big|$) is the one true unknown left.

**Introducing $t=|z|$, and evaluating $\big||z|-i\big|$ in terms of it.** Since $|z|$ is by definition a nonnegative real number, naming it $t=|z|$ turns the equation into one involving a single real unknown $t>0$ instead of the complex unknown $z$. The expression $|z|-i$ then becomes exactly $t-i$: a complex number with real part $t$ and imaginary part $-1$. By fact 4, its modulus is $\big|t-i\big|=\sqrt{t^2+(-1)^2}=\sqrt{t^2+1}$. Substituting this in gives a single equation purely in $t$: $|w|\cdot\sqrt{t^2+1}=\dfrac{|K|}{t}$.

**Clearing the square root and the fraction together.** Squaring both sides removes the square root ($|w|^2$ appears on the left, since $|w|$ was squared too; the right side becomes $\dfrac{K^2}{t^2}$, since $|K|^2=K^2$). Then multiplying both sides by $t^2$ (valid since $t>0$, so $t^2\ne0$) clears the fraction: $|w|^2\,t^2\,(t^2+1)=K^2$. Expanded, this contains only $t^4$ and $t^2$ (no odd power of $t$ at all): a biquadratic equation.

**Solving the biquadratic by substitution.** A biquadratic in $t$ is exactly a quadratic in disguise: setting $u=t^2$ (with the constraint $u>0$, inherited from $t>0$) turns $|w|^2t^4+|w|^2t^2-K^2=0$ into the ordinary quadratic $|w|^2u^2+|w|^2u-K^2=0$. By Vieta's formulas (fact 20, which holds for any quadratic regardless of the sign of its coefficients), the product of this quadratic's two roots is $\dfrac{-K^2}{|w|^2}$, a strictly negative number (since $K\ne0$ always here); a negative product means the two roots always have opposite signs, so exactly one of them is positive and the other is negative, every single time, with no case analysis needed beyond checking each root's own sign. Discard the negative root; the positive one is the true value of $u$.

**Recovering $|z|$ and reading off the answer.** Since $u=t^2$ and $t=|z|>0$, the answer is $t=\sqrt{u}$ (the positive square root; $t$ can never be negative, since it is a modulus). This number $t$ IS $|z|$: the exact value the four options are describing. The four options themselves come in different shapes: some state a single inequality ($|z|<c$ or $|z|>c$), some state a double inequality bracketing $|z|$ between two numbers ($c_1<|z|<c_2$), and some state an exact equality ($|z|=c$); reading which shape a given option has, and testing $t$ against exactly that shape, is the last step.
'''


PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (ReAct format)**

Đề bài: Xét số phức $z$ thỏa mãn $(1+2i)|z| = \dfrac{\sqrt{10}}{z}-2+i$. Mệnh đề nào dưới đây đúng?

A. $\dfrac{3}{2} < |z| < 2$.   B. $|z| > 2$.   C. $|z| < \dfrac{1}{2}$.   D. $\dfrac{1}{2} < |z| < \dfrac{3}{2}$.

<think>
Thought: The coefficient of $|z|$, quoted verbatim from "$(1+2i)|z|$", is $w=1+2i$. I store it.
Action: Calculator
Action Input: w = 1+2*I
Observation: w = 1 + 2*I

Thought: The numerator on the right, quoted verbatim from "$\dfrac{\sqrt{10}}{z}$", is $K=\sqrt{10}$. I store it.
Action: Calculator
Action Input: K = sqrt(10)
Observation: K = sqrt(10)

Thought: I check that the free constant matches PART 2's guarantee: I compute $w \cdot i$ and compare it against the free part read verbatim from the problem, $-2+i$.
Action: Calculator
Action Input: free_constant = w * I
Observation: free_constant = -2 + I

Thought: `free_constant` came back $-2+i$, matching the free part quoted verbatim from the problem exactly. So the equation really does factor as $w(|z|-i)=K/z$, and $|w|^2$ is what I need next, from fact 12's modulus-of-a-product step.
Action: Calculator
Action Input: w_abs_sq = Abs(w)**2
Observation: w_abs_sq = 5

Thought: The right side of the equation gets squared too, once both sides are squared; so I also need $K^2$, the other quantity PART 2's biquadratic requires.
Action: Calculator
Action Input: K_sq = K**2
Observation: K_sq = 10

Thought: Following PART 2's derivation, the biquadratic $|w|^2t^4+|w|^2t^2-K^2=0$ becomes, with $u=t^2$, the quadratic $|w|^2u^2+|w|^2u-K^2=0$. I solve it for $u$.
Action: Calculator
Action Input: u_sols = solve(Eq(w_abs_sq*u**2 + w_abs_sq*u - K_sq, 0), u)
Observation: u_sols = [-2, 1]

Thought: Two roots, as expected (opposite signs, by PART 2's Vieta argument). I check the sign of the first one, `u_sols[0]`.
Action: Calculator
Action Input: u_sols[0] > 0
Observation: False

Thought: `u_sols[0]` is not positive, so it is discarded. I check the second one, `u_sols[1]`.
Action: Calculator
Action Input: u_sols[1] > 0
Observation: True

Thought: `u_sols[1]` is positive; that is the valid $u$. I store it by name, using the index rather than retyping the number.
Action: Calculator
Action Input: u_val = u_sols[1]
Observation: u_val = 1

Thought: With $u$ known, $t=\sqrt{u}$ recovers the actual unknown by PART 2's substitution $u=t^2$; and $t$, in turn, IS $|z|$, so this is exactly the value every option is describing.
Action: Calculator
Action Input: t_val = sqrt(u_val)
Observation: t_val = 1

Thought: `t_val` is confirmed. Before testing anything, here are all four options copied fresh, side by side, exactly as printed: A: "$\dfrac{3}{2} < |z| < 2$."; B: "$|z| > 2$."; C: "$|z| < \dfrac{1}{2}$."; D: "$\dfrac{1}{2} < |z| < \dfrac{3}{2}$." I test them in this order, using exactly the symbol just copied for each one; I never approximate `t_val` to eyeball how close it looks to a number. Starting with A: two `<` symbols, $|z|$ between $\dfrac{3}{2}$ and $2$; the double-inequality shape.
Action: Calculator
Action Input: test_A = 3/2 < t_val < 2
Observation: test_A = False

Thought: B, copied above as "$|z| > 2$.": the symbol right after $|z|$ there is `>`.
Action: Calculator
Action Input: test_B = t_val > 2
Observation: test_B = False

Thought: C, copied above as "$|z| < \dfrac{1}{2}$.": the symbol right after $|z|$ there is `<`.
Action: Calculator
Action Input: test_C = t_val < 1/2
Observation: test_C = False

Thought: D, copied above as "$\dfrac{1}{2} < |z| < \dfrac{3}{2}$.": two `<` symbols, the double-inequality shape once more.
Action: Calculator
Action Input: test_D = 1/2 < t_val < 3/2
Observation: test_D = True

Thought: Before deciding, I check every result against the exact quotes copied at the start of this phase: A "$\dfrac{3}{2} < |z| < 2$." $\to$ `test_A=False`; B "$|z| > 2$." $\to$ `test_B=False`; C "$|z| < \dfrac{1}{2}$." $\to$ `test_C=False`; D "$\dfrac{1}{2} < |z| < \dfrac{3}{2}$." $\to$ `test_D=True`. Exactly one is `True`: `test_D`, matching the quote copied for D above; the letter is D.
</think>

Final Answer: \boxed{D}
'''


PART_NHIEMVU = r'''
**PART 4: YOUR TASK, STEP BY STEP**

The problem always has the form: $w|z|=\dfrac{K}{z}+(\text{free constant})$, with the free constant equal to $w\cdot i$; which one of four mentions about $|z|$ (an inequality, a double inequality, or an equality) is correct? Follow these steps, using the Calculator for every computation.

1. **Reading $w$ and $K$.** Quote the coefficient of $|z|$ on the left verbatim and store it as $w=a+bi$. Quote the numerator on the right (the constant divided by $z$) verbatim and store it as $K$.
2. **Confirming the factoring, and $|w|^2$, $K^2$.** Compute `w*I` with the Calculator (never by mental arithmetic, even though it looks simple) and compare the result against the free constant read verbatim from the problem; by PART 2, they always match, and this is what allows the equation to factor as $w(|z|-i)=K/z$. Then compute `w_abs_sq = Abs(w)**2` and `K_sq = K**2`. Never introduce `z` itself into a Calculator expression, here or anywhere else, to try to "verify" the original equation directly; `z` is never a declared free unknown in this problem type (only `u` is, from step 3 onward) and can never appear in any Calculator call; every quantity from here on is built purely from the known constants $w,K$ and the single unknown $u=t^2$.
3. **Solving the biquadratic via $u=t^2$.** Compute `u_sols = solve(Eq(w_abs_sq*u**2 + w_abs_sq*u - K_sq, 0), u)`. This always returns exactly two roots.
4. **Picking the positive root.** For each element of `u_sols` in turn, test its sign with `u_sols[<index>] > 0`. By PART 2's Vieta argument, exactly one element will test `True`; store that one (by indexing, never by retyping its value) as `u_val`.
5. **Recovering $|z|$.** Compute `t_val = sqrt(u_val)`. This is exactly $|z|$.
6. **Copying all four options fresh, together, before testing any of them.** Do not assume any option's shape or direction from the worked example; a different problem can pair completely different directions with the letters A, B, C, D. In the Thought that introduces your first test, copy all four options' exact text side by side (A: "...", B: "...", C: "...", D: "..."), reading each one character by character, with particular attention to whether its symbol is `<`, `>`, or `=`. This copied set is what every test below is checked against.
7. **Testing each option, storing each result under its own name.** For each option A, B, C, D in turn, using the shape identified from the text copied in step 6 (a single inequality $|z|<c$ or $|z|>c$, a double inequality $c_1<|z|<c_2$, or an equality $|z|=c$): compute it as a NAMED result, `test_A = <matching expression>` (likewise `test_B`, `test_C`, `test_D`), using `t_val < c`, `t_val > c`, `c_1 < t_val < c_2`, or `Eq(t_val, c)`, keeping the `<`/`>` direction exactly as copied. Do this for all four, even after one already returns `True`; this numeric check is not a formal proof, and never approximate `t_val` (e.g. with `.evalf()`) to eyeball how close it looks to an option's number.
8. **Final cross-check against the copied text, before deciding.** In one closing Thought, line up each option's text copied in step 6 next to its stored result (A → `test_A`, B → `test_B`, C → `test_C`, D → `test_D`). Exactly one should be `True`; that letter is the answer. If that is not what this side-by-side check shows, some test's expression does not actually match its copied text; find and fix that mismatch and recompute before deciding. Write the final line `Final Answer: \boxed{<Letter>}` using that letter.

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

In [ ]:
# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# (MayTinh/fast_simplify/is_zero da duoc dinh nghia o Cell 3, truoc khi
# LLM(...) khoi tao - xem giai thich an toan CUDA/fork o do)
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 26          # quy trinh day du ~9 luot tinh (w,K,w_abs_sq,K_sq,u_sols,2x kiem tra dau,u_val,t_val) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')
FINAL_ANSWER_RE = re.compile(r'Final\s+Answer\s*:\s*\\boxed\{\s*[A-D]\s*\}')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        m_final = FINAL_ANSWER_RE.search(s.full_text)
        if m_final:
            s.full_text = s.full_text[:m_final.end()]
            s.xong = True
            s.ly_do_dung = 'model_ket_thuc'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        # D37-only fix: khi 1 luot generate() lo sinh QUA 1 khoi
        # Thought/Action/Action-Input (stop=['Observation:'] chi kich hoat
        # khi CHINH MODEL tu viet chu do, nen no co the ramble sang khoi
        # thu 2 truoc khi dung), luon lay Action Input DAU TIEN (dung tinh
        # than ReAct - moi luot chi 1 action) va cat bo phan sinh them sau
        # do khoi s.full_text, thay vi am tham lay Action Input CUOI CUNG
        # va bo qua dong dau (day la nguyen nhan loi "undefined name" quan
        # sat duoc o STT776).
        cac_khop = list(ACTION_INPUT_RE.finditer(o.text))
        khop = cac_khop[0] if cac_khop else None
        bieu_thuc = khop.group(1).strip() if khop else ''
        if len(cac_khop) > 1:
            vi_tri_cat = len(s.full_text) - len(o.text) + khop.end()
            s.full_text = s.full_text[:vi_tri_cat]

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)

## Dang D39

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

OUT_PATH  = '/kaggle/working/d39_react_calculator_full90_DeepSeek-R1-Distill-Qwen-1.5B.csv'

MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = {'a', 'b'}  # a,b: toa do thuc/ao cua z tren duong thang trung truc (nhanh 2)

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/11-dang-full/plan_solve_prompts_11dang_merged.json'  # SUA NEU KHAC
with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D39']
print('So cau:', len(records))

# ---- Backend Calculator: dat O DAY, TRUOC khi LLM(...)/CUDA khoi tao ----
# (an toan multiprocessing.fork - xem giai thich trong comment ben duoi)
import sympy as sp
import multiprocessing as mp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Luoi an toan CHO CA BATCH: du da dung radsimp() de tranh treo may o hau
# het truong hop, sympy van khong dam bao toc do cho MOI to hop can thuc
# bat ky - chi can 1/90 cau roi vao truong hop xau la ca batch nghen theo
# (Calculator chay tuan tu tung cau). Dung TIEN TRINH CON that (co the bi
# giet cuong buc bang tin hieu he dieu hanh) thay vi thread: thread chi
# ngat duoc tai diem GIL duoc nhuong lai, KHONG dam bao neu tinh toan ket
# sau trong 1 loi goi C lien tuc (da xac nhan qua thuc te chay tren
# Kaggle). Pool tien trinh con nay duoc tao NGAY TAI DAY, TRUOC KHI
# LLM(...)/CUDA khoi tao ben duoi - vi vay an toan tuyet doi voi
# multiprocessing.fork (fork() SAU KHI CUDA da khoi tao moi la nguy hiem,
# tung gay treo o mot lan thu truoc do).
TIMEOUT_SECONDS = 20


def fast_simplify(expr):
    """Rut gon nhanh va ON DINH hon sp.simplify() thuan tuy.

    sp.simplify() la ham "thu tat ca chien luoc roi chon ket qua ngan nhat",
    rat cham (co the treo may) khi bieu thuc co nhieu MAU SO chua can bac
    hai khac goc (vd tu phep chia (Bx-Ay)/(Bx-Ax)) - dung sinh ra qua nhieu
    dang the hien khac nhau ma khong bao gio hop nhat lai. radsimp() giai
    quyet dung goc van de nay: no huu ti hoa mau so chua can NGAY LAP TUC,
    nen ket qua o moi buoc luon o dang gon, khong de cac mau can long tich
    luy qua tung phep tinh tiep theo. Dung radsimp() lam buoc rut gon CHINH
    (nhanh, gan nhu luon du); chi roi sang simplify() lam buoc du phong khi
    radsimp() chua dua duoc ve dang 0/dang gon nhat.
    """
    return sp.expand(sp.radsimp(expr))


def is_zero(expr):
    """Kiem tra bieu thuc co bang 0 khong.

    Buoc dau (radsimp) bat duoc phan lon truong hop that nhanh va tuyet
    doi chinh xac. Neu chua ket luan duoc, KHONG roi sang sp.simplify()
    (chung minh dai so - cham, khong dam bao toc do, day la duong tung
    gay treo may). Thay vao do, so sanh gia tri SO HOC voi do chinh xac
    rat cao (50 chu so thap phan) - dung nguyen tac may tinh Casio: so
    hai so thap phan thay vi chung minh dang thuc dai so. Voi mien bai
    toan nay (cac hang so dai so co dinh tu de bai, khong phai gia tri
    adversarial), sai so gan nhu khong the xay ra o do chinh xac nay.
    """
    rut_gon = sp.radsimp(expr)
    if rut_gon == 0:
        return True
    return abs(complex(rut_gon.evalf(50))) < 1e-40


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            # 'Eq' duoc thay bang phien ban evaluate=False: sp.Eq() mac dinh
            # TU DONG thu kiem tra 2 ve co bang nhau NGAY LUC KHOI TAO (co che
            # rieng cua sympy, khac hoan toan ham is_zero() tu viet ben duoi) -
            # voi bieu thuc can long phuc tap, chinh buoc TU DONG nay co the
            # treo may, va treo TRUOC CA KHI chay_co_timeout kip can thiep (vi
            # no xay ra ngay trong luc sympify dang parse chuoi). Dung ban
            # evaluate=False de hoan toan doi viec so khop cho ham is_zero() -
            # da duoc kiem chung nhanh va dang tin cay - dam nhiem.
            ns_de_parse = dict(self.ns)
            ns_de_parse['Eq'] = lambda a, b: sp.Eq(a, b, evaluate=False)
            parsed = sp.sympify(s, locals=ns_de_parse)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau, loi_timeout = chay_co_timeout(is_zero, parsed.lhs - parsed.rhs)
                if loi_timeout:
                    return None, loi_timeout
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri, loi_timeout = chay_co_timeout(fast_simplify, parsed)
            if loi_timeout:
                return None, loi_timeout
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

try:
    _CALC_POOL = mp.get_context('fork').Pool(1)
except ValueError:
    _CALC_POOL = None  # khong co fork (vd may local Windows) -> chay khong timeout


def chay_co_timeout(ham, *args):
    """Chay ham(*args) trong tien trinh con (fork, tao TRUOC CUDA nen an
    toan), gioi han TIMEOUT_SECONDS. Neu qua han, HUY va TAO LAI pool (vi
    tien trinh con cu van con chay ngam, khong the tai su dung duoc nua)
    roi tra ve loi ro rang thay vi treo may."""
    global _CALC_POOL
    if _CALC_POOL is None:
        return ham(*args), None
    ar = _CALC_POOL.apply_async(ham, args)
    try:
        return ar.get(timeout=TIMEOUT_SECONDS), None
    except mp.TimeoutError:
        _CALC_POOL.terminate()
        _CALC_POOL = mp.get_context('fork').Pool(1)
        return None, (f'computation timed out after {TIMEOUT_SECONDS}s '
                      '(the expression is too complex to simplify exactly). '
                      'Do not resend the exact same expression; try continuing '
                      'with a different, smaller step instead.')


# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) nhieu lan trong cung kernel tung gay loi GPU het bo
# nho (da xac nhan qua thuc te chay tren Kaggle o ban 5 dang truoc).
print('Dung lai tok/llm da nap san.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D39_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai PART_HUONGGIAI va PART_FEWSHOT,
# giu nguyen PART_TOOL va PART_KIENTHUC.

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. For most problem types this is decided by an exact symbolic proof. For this problem type specifically, it is decided by evaluating both sides to 50 significant decimal digits and checking they agree to that precision - this is not a formal proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True` - test all four, every time (see PART 4's matching step for why).
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher-degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation-solve mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non-real ones when the polynomial's real roots don't account for its full degree; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`; `Im` is not recognized and silently fails to evaluate) and discard it immediately if not. For a root confirmed real, use `re(root)`; not the raw value; in every later threshold or substitution, since even a numerically-real root can carry a residual non-zero imaginary part too small to matter but large enough to break a direct comparison. Any comparison built on such numeric roots (a threshold like `t > 0`, or a self-consistency check like `Abs(a - b) < 1e-6`) should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root; in particular the final count of solutions and matching it against the answer options; stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo - retyping a small variation of the same idea will keep producing the same error. Stop, go back to PART 2's method for this exact situation (a formula giving `zoo`/an undefined result almost always means a special case described somewhere in PART 2 applies here - e.g. a division that is only valid when some quantity is nonzero), and use the alternative formula PART 2 gives for that case, typed as a normal exact expression - never invent a placeholder word like `undefined` as if it were a value; the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$ - there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change - so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).

20. **Vieta's formulas for a quadratic.** For $z^2+pz+q=0$ with roots $z_1,z_2$ (real or complex), the coefficients determine the roots' sum and product directly, without solving anything: $z_1+z_2=-p$ and $z_1z_2=q$. This holds whether the roots are real or a complex-conjugate pair (fact 15) - the SAME two equations hold either way, since they simply come from matching coefficients in the identity $z^2+pz+q=(z-z_1)(z-z_2)=z^2-(z_1+z_2)z+z_1z_2$, which does not care whether $z_1,z_2$ happen to be real.


21. **Conjugate of a sum, and conjugate of a real multiple of $i$.** For any complex numbers $u,v$: $\overline{u+v}=\overline{u}+\overline{v}$ and $\overline{u-v}=\overline{u}-\overline{v}$ (conjugation distributes over addition/subtraction, since conjugating just flips the sign of every imaginary part, and imaginary parts add/subtract termwise). In particular, for a REAL constant $c$: $\overline{ci}=-ci$ (because $ci$ is purely imaginary with imaginary part $c$, flipping its sign gives $-ci$). Combining these: for any complex $z$ and real constant $c$, $\overline{z+ci}=\overline{z}+\overline{ci}=\overline{z}-ci$. This is the key trick that lets you replace $\overline{z}-ci$ by $\overline{z+ci}$ wherever it appears, which (by fact 4, $|\overline{w}|=|w|$) means $|\overline{z}-ci|=|z+ci|$ - turning an expression that LOOKS like it needs both $z$ and $\overline{z}$ separately into one that depends only on the single quantity $z+ci$.
'''


PART_HUONGGIAI = r'''
**PART 2: HOW TO SOLVE THIS PROBLEM TYPE**

**Two conditions, two very different roles.** The first condition, $|z^2|=k|z-\overline{z}|$, is a single equation relating $z$ to itself - by itself it describes a whole curve of possible $z$, not a finite list. The second condition, $|(z-c)(\overline{z}-ci)|=|z+ci|^2$, at first glance seems to mix $z$ and $\overline{z}$ in a way that is hard to disentangle - but this is exactly the one worth simplifying FIRST, because doing so collapses it from "an equation mixing $z$ and $\overline{z}$" down to a small, explicit case split, each branch of which is far easier to combine with the first condition afterward.

**Recognizing $\overline{z}-ci$ in disguise.** Look at the second factor, $\overline{z}-ci$. By fact 21, this is exactly $\overline{z+ci}$ - the conjugate of the SAME quantity $z+ci$ that appears on the right-hand side. And by fact 4, $|\overline{w}|=|w|$ for any $w$, so $|\overline{z}-ci|=|\overline{z+ci}|=|z+ci|$. This single observation is the whole key to the problem: using fact 12 ($|uv|=|u||v|$), the second condition becomes
$$|z-c|\cdot|\overline{z}-ci| = |z+ci|^2 \iff |z-c|\cdot|z+ci| = |z+ci|^2,$$
an equation in $|z-c|$ and $|z+ci|$ only - no more separate, unresolved $\overline{z}$.

**From one equation to two branches.** Move everything to one side and factor out the common term $|z+ci|$:
$$|z+ci|\big(|z-c|-|z+ci|\big)=0.$$
A product of two real, nonnegative numbers is zero exactly when at least one factor is zero - so this single equation is EXACTLY equivalent to "$|z+ci|=0$ OR $|z-c|=|z+ci|$". These are two genuinely different situations (not two sub-cases of the same algebra), so each is worked out on its own, and every $z$ satisfying either one (and ALSO satisfying the first condition) counts toward the answer.

**Branch 1: $|z+ci|=0$ pins down a single concrete number.** A modulus is zero only when the number itself is zero, so $|z+ci|=0 \iff z=-ci$ - already fully concrete, no unknowns left. This branch never used the first condition, so $z=-ci$ is only an actual solution to the PROBLEM if it also satisfies $|z^2|=k|z-\overline{z}|$: compute $|z^2|$ and $k|z-\overline{z}|$ for this specific $z=-ci$ and compare them. If they are equal, branch 1 contributes exactly this one solution; if not, branch 1 contributes nothing at all - do not assume it always counts.

**Branch 2: $|z-c|=|z+ci|$ is itself a whole locus - describe it before touching the first condition.** This says $z$ is equidistant from two fixed points, $c$ (on the real axis) and $-ci$ (on the imaginary axis) (fact 17) - geometrically the perpendicular bisector of the segment between them, a full line, not a single point. To find WHICH points on that line also satisfy the first condition, write $z=a+bi$ ($a,b$ real) and square both sides: $(a-c)^2+b^2=a^2+(b+c)^2$. Both sides contain the cluster $a^2+b^2$ (fact 11) - solving this single equation for $b$ eliminates that cluster automatically and gives $b=-a$: the whole line collapses to "$b$ is always the negative of $a$", i.e. every point on it has the form $z=a-ai$ for some real $a$ still free to vary.

**Feeding $z=a-ai$ into the first condition turns the whole line into a short list of points.** With $z=a-ai$: $z^2=(a-ai)^2=-2a^2i$ (fact 2, using $i^2=-1$), so $|z^2|=2a^2$; and $z-\overline{z}=(a-ai)-(a+ai)=-2ai$, so $|z-\overline{z}|=2|a|$ (the absolute value matters - $a$ can be either sign). The first condition $|z^2|=k|z-\overline{z}|$ becomes $2a^2=2k|a|$. Solve this single equation for $a$ (do not simplify it by hand into $|a|(|a|-k)=0$ yourself - let the Calculator solve the equation directly, exactly as it is): it always has exactly three real solutions, $a=0$, $a=k$, and $a=-k$ (since $k$ here is always a stated positive coefficient, these three values are always distinct). Each distinct $a$ gives a distinct point $z=a-ai$ on the line, so branch 2 always contributes exactly three solutions: $z=0$, $z=k-ki$, $z=-k+ki$.

**Totalling - and checking the branches don't secretly share a solution.** The final count is (branch 1's contribution, 0 or 1) plus (branch 2's contribution, always 3). Before just adding, check whether branch 1's candidate $z=-ci$ could coincide with any of branch 2's three points: $z=-ci$ has real part $0$ and imaginary part $-c$; among branch 2's points, only $z=0$ also has real part $0$, but its imaginary part is $0$, not $-c$ - and $c\ne0$ in every version of this problem, so they can never be equal. The two branches never overlap, so the total is a plain sum, with no double-counting to worry about.
'''


PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (ReAct format)**

Problem: How many complex numbers $z$ satisfy $|z^2|=2|z-\overline{z}|$ and $|(z-4)(\overline{z}-4i)|=|z+4i|^2$?
A. $3$.   B. $1$.   C. $2$.   D. $4$.

Thought: First condition, quoted verbatim: "$|z^2|=2|z-\overline{z}|$" - the coefficient of $|z-\overline{z}|$ is $2$, so $k=2$. I store it.
Action: Calculator
Action Input: k = 2
Observation: k = 2

Thought: Second condition, quoted verbatim: "$|(z-4)(\overline{z}-4i)|=|z+4i|^2$" - matching against the template $(z-c)(\overline{z}-ci)=|z+ci|^2$: it shows "$z-4$", "$\overline{z}-4i$", "$z+4i$", every sign already matching the template as written, so $c=4$ directly (all three occurrences agree). I store it.
Action: Calculator
Action Input: c = 4
Observation: c = 4

Thought: Branch 1 candidate is $z=-ci$; I compute it.
Action: Calculator
Action Input: z_th1 = -c*I
Observation: z_th1 = -4*I

Thought: I check whether this candidate satisfies the first condition. First $|z^2|$.
Action: Calculator
Action Input: lhs1_th1 = Abs(z_th1**2)
Observation: lhs1_th1 = 16

Thought: Now $k|z-\overline{z}|$ for the same candidate.
Action: Calculator
Action Input: rhs1_th1 = k*Abs(z_th1 - conjugate(z_th1))
Observation: rhs1_th1 = 16

Thought: Compare the two sides to decide whether branch 1's candidate is a genuine solution.
Action: Calculator
Action Input: Eq(lhs1_th1, rhs1_th1)
Observation: True

Thought: Branch 1 is valid, contributing exactly 1 solution ($z=-4i$). Now branch 2: solve $(a-c)^2+b^2=a^2+(b+c)^2$ for $b$.
Action: Calculator
Action Input: solve(Eq((a-c)**2+b**2, a**2+(b+c)**2), b)
Observation: [-a]

Thought: So $b=-a$ on this line.
Action: Calculator
Action Input: b = -a
Observation: b = -a

Thought: Build the point on the line, $z=a+bi$.
Action: Calculator
Action Input: z_th2 = a + b*I
Observation: z_th2 = a - I*a

Thought: Compute $|z^2|$ for this point.
Action: Calculator
Action Input: lhs1_th2 = Abs(z_th2**2)
Observation: lhs1_th2 = 2*a**2

Thought: Compute $k|z-\overline{z}|$ for this point.
Action: Calculator
Action Input: rhs1_th2 = k*Abs(z_th2 - conjugate(z_th2))
Observation: rhs1_th2 = 4*Abs(a)

Thought: Solve the first condition for $a$ to find every point on the line that also satisfies it.
Action: Calculator
Action Input: solve(Eq(lhs1_th2, rhs1_th2), a)
Observation: [-2, 0, 2]

Thought: Three distinct values of $a$, so branch 2 contributes exactly 3 solutions. Branch 1 contributed 1, and I already know from PART 2's reasoning that branch 1's candidate ($z=-4i$, real part $0$, imaginary part $-4$) cannot coincide with any branch-2 point (only $z=0$ has real part $0$, and its imaginary part is $0\ne-4$), so no overlap to worry about. Total is a plain sum.
Action: Calculator
Action Input: tong = 1 + 3
Observation: tong = 4

Thought: Eq() here is a numeric check, not a symbolic proof, so I test ALL FOUR options every time, never stopping early. Read option A verbatim: "A. $3$."; A is 3. Test it.
Action: Calculator
Action Input: Eq(tong, 3)
Observation: False

Thought: Not A. Read option B verbatim: "B. $1$."; B is 1. Test it.
Action: Calculator
Action Input: Eq(tong, 1)
Observation: False

Thought: Not B. Read option C verbatim: "C. $2$."; C is 2. Test it.
Action: Calculator
Action Input: Eq(tong, 2)
Observation: False

Thought: Not C. Read option D verbatim: "D. $4$."; D is 4. Test it.
Action: Calculator
Action Input: Eq(tong, 4)
Observation: True

Thought: Exactly one option came back True: D. That is the answer.
Final Answer: \boxed{D}
'''


PART_NHIEMVU = r'''
**PART 4: YOUR TASK, STEP BY STEP**

The problem always has the form: how many complex numbers $z$ satisfy $|z^2|=k|z-\overline{z}|$ AND $|(z-c)(\overline{z}-ci)|=|z+ci|^2$ (for specific constants $k,c$ given in this problem)? Follow these steps, using the Calculator for every computation.

1. **Finding $k$.** Quote the first condition verbatim from the problem (e.g. "$|z^2|=2|z-\overline{z}|$"), then read off the coefficient of $|z-\overline{z}|$ as $k$ (if no coefficient is written, $k=1$). Do not let any other number in the problem influence this - $k$ comes ONLY from this first condition.
2. **Finding $c$ - this is the step most likely to go wrong, so do it carefully.** Quote the second condition verbatim (e.g. "$|(z-4)(\overline{z}-4i)|=|z+4i|^2$" or "$|(z+3)(\overline{z}+3i)|=|z-3i|^2$"). It must match the template $(z-c)(\overline{z}-ci)=|z+ci|^2$ for SOME real number $c$ (possibly negative) - find that $c$ by matching signs, not by copying a digit: if the problem shows "$z-4$", "$\overline{z}-4i$", "$z+4i$" (every sign matches the template as written), then $c=4$. If instead it shows "$z+3$", "$\overline{z}+3i$", "$z-3i$" (every sign is the OPPOSITE of the template), then $c=-3$ (because $z+3=z-(-3)$). All three occurrences in the problem must agree with the SAME value of $c$ - if you find a value that only matches one or two of the three occurrences, you have the wrong sign; re-read and flip it.
3. Compute branch 1's candidate $z_{th1}=-ci$.
4. Check whether $z_{th1}$ satisfies the first condition: compute $|z_{th1}^2|$ and $k|z_{th1}-\overline{z_{th1}}|$ separately, then compare them with `Eq(...)`. Remember the exact result (branch 1 contributes 1 solution if they match, 0 if not) - do not assume either outcome in advance.
5. Solve $(a-c)^2+b^2=a^2+(b+c)^2$ for $b$ (declared free unknowns $a,b$) to get the line's equation in the form $b=\dots$, then store that as `b`.
6. Build $z_{th2}=a+bi$ using this $b$, then compute $|z_{th2}^2|$ and $k|z_{th2}-\overline{z_{th2}}|$.
7. Solve the equation "$|z_{th2}^2|$ equals $k|z_{th2}-\overline{z_{th2}}|$" for $a$. This always gives exactly 3 real values - branch 2 always contributes exactly 3 solutions, one for each value of $a$.
8. Add branch 1's contribution (from step 4) and branch 2's contribution (always 3) to get the total count. (The two branches never overlap - see PART 2's reasoning - so a plain sum is always correct; you do not need to re-derive this each time.)
9. For each option A, B, C, D in turn: read that option's value VERBATIM from the problem text (do not reuse a number from any earlier example), then compare with `Eq(total, <that option's value>)`. Test every option even after one already returns `True` - `Eq()` here is a numeric check, not a formal proof.
10. Write the final line `Final Answer: \boxed{<Letter>}` using whichever option matched.

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

In [ ]:
# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# (MayTinh/fast_simplify/is_zero da duoc dinh nghia o Cell 3, truoc khi
# LLM(...) khoi tao - xem giai thich an toan CUDA/fork o do)
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 30          # quy trinh day du ~13 luot tinh (k,c,z_th1,lhs1_th1,rhs1_th1,check,solve-b,b,z_th2,lhs1_th2,rhs1_th2,solve-a,tong) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        # EP SAN token dau tien (giong ban 1 cau): model bi buoc noi tiep tu
        # "Thought:" ngay sau <think>, khong con quyen tu chon viet van xuoi
        # mo dau (hanh vi mac dinh de lech khoi dinh dang ReAct).
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        khop = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            khop = mm
        bieu_thuc = khop.group(1).strip() if khop else ''

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)

## Dang D44

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

OUT_PATH  = '/kaggle/working/d44_react_calculator_full90_DeepSeek-R1-Distill-Qwen-1.5B.csv'

MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = {'x', 'y'}  # x,y: phan thuc/ao cua w=x+yi

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/11-dang-full/plan_solve_prompts_11dang_merged.json'  # SUA NEU KHAC
with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D44']
print('So cau:', len(records))

# ---- Backend Calculator: dat O DAY, TRUOC khi LLM(...)/CUDA khoi tao ----
# (an toan multiprocessing.fork - xem giai thich trong comment ben duoi)
import sympy as sp
import multiprocessing as mp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Luoi an toan CHO CA BATCH: du da dung radsimp() de tranh treo may o hau
# het truong hop, sympy van khong dam bao toc do cho MOI to hop can thuc
# bat ky - chi can 1/90 cau roi vao truong hop xau la ca batch nghen theo
# (Calculator chay tuan tu tung cau). Dung TIEN TRINH CON that (co the bi
# giet cuong buc bang tin hieu he dieu hanh) thay vi thread: thread chi
# ngat duoc tai diem GIL duoc nhuong lai, KHONG dam bao neu tinh toan ket
# sau trong 1 loi goi C lien tuc (da xac nhan qua thuc te chay tren
# Kaggle). Pool tien trinh con nay duoc tao NGAY TAI DAY, TRUOC KHI
# LLM(...)/CUDA khoi tao ben duoi - vi vay an toan tuyet doi voi
# multiprocessing.fork (fork() SAU KHI CUDA da khoi tao moi la nguy hiem,
# tung gay treo o mot lan thu truoc do).
TIMEOUT_SECONDS = 20


def fast_simplify(expr):
    """Rut gon nhanh va ON DINH hon sp.simplify() thuan tuy.

    sp.simplify() la ham "thu tat ca chien luoc roi chon ket qua ngan nhat",
    rat cham (co the treo may) khi bieu thuc co nhieu MAU SO chua can bac
    hai khac goc (vd tu phep chia (Bx-Ay)/(Bx-Ax)) - dung sinh ra qua nhieu
    dang the hien khac nhau ma khong bao gio hop nhat lai. radsimp() giai
    quyet dung goc van de nay: no huu ti hoa mau so chua can NGAY LAP TUC,
    nen ket qua o moi buoc luon o dang gon, khong de cac mau can long tich
    luy qua tung phep tinh tiep theo. Dung radsimp() lam buoc rut gon CHINH
    (nhanh, gan nhu luon du); chi roi sang simplify() lam buoc du phong khi
    radsimp() chua dua duoc ve dang 0/dang gon nhat.
    """
    return sp.expand(sp.radsimp(expr))


def is_zero(expr):
    """Kiem tra bieu thuc co bang 0 khong.

    Buoc dau (radsimp) bat duoc phan lon truong hop that nhanh va tuyet
    doi chinh xac. Neu chua ket luan duoc, KHONG roi sang sp.simplify()
    (chung minh dai so - cham, khong dam bao toc do, day la duong tung
    gay treo may). Thay vao do, so sanh gia tri SO HOC voi do chinh xac
    rat cao (50 chu so thap phan) - dung nguyen tac may tinh Casio: so
    hai so thap phan thay vi chung minh dang thuc dai so. Voi mien bai
    toan nay (cac hang so dai so co dinh tu de bai, khong phai gia tri
    adversarial), sai so gan nhu khong the xay ra o do chinh xac nay.
    """
    rut_gon = sp.radsimp(expr)
    if rut_gon == 0:
        return True
    return abs(complex(rut_gon.evalf(50))) < 1e-40


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            # 'Eq' duoc thay bang phien ban evaluate=False: sp.Eq() mac dinh
            # TU DONG thu kiem tra 2 ve co bang nhau NGAY LUC KHOI TAO (co che
            # rieng cua sympy, khac hoan toan ham is_zero() tu viet ben duoi) -
            # voi bieu thuc can long phuc tap, chinh buoc TU DONG nay co the
            # treo may, va treo TRUOC CA KHI chay_co_timeout kip can thiep (vi
            # no xay ra ngay trong luc sympify dang parse chuoi). Dung ban
            # evaluate=False de hoan toan doi viec so khop cho ham is_zero() -
            # da duoc kiem chung nhanh va dang tin cay - dam nhiem.
            ns_de_parse = dict(self.ns)
            ns_de_parse['Eq'] = lambda a, b: sp.Eq(a, b, evaluate=False)
            parsed = sp.sympify(s, locals=ns_de_parse)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau, loi_timeout = chay_co_timeout(is_zero, parsed.lhs - parsed.rhs)
                if loi_timeout:
                    return None, loi_timeout
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri, loi_timeout = chay_co_timeout(fast_simplify, parsed)
            if loi_timeout:
                return None, loi_timeout
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

try:
    _CALC_POOL = mp.get_context('fork').Pool(1)
except ValueError:
    _CALC_POOL = None  # khong co fork (vd may local Windows) -> chay khong timeout


def chay_co_timeout(ham, *args):
    """Chay ham(*args) trong tien trinh con (fork, tao TRUOC CUDA nen an
    toan), gioi han TIMEOUT_SECONDS. Neu qua han, HUY va TAO LAI pool (vi
    tien trinh con cu van con chay ngam, khong the tai su dung duoc nua)
    roi tra ve loi ro rang thay vi treo may."""
    global _CALC_POOL
    if _CALC_POOL is None:
        return ham(*args), None
    ar = _CALC_POOL.apply_async(ham, args)
    try:
        return ar.get(timeout=TIMEOUT_SECONDS), None
    except mp.TimeoutError:
        _CALC_POOL.terminate()
        _CALC_POOL = mp.get_context('fork').Pool(1)
        return None, (f'computation timed out after {TIMEOUT_SECONDS}s '
                      '(the expression is too complex to simplify exactly). '
                      'Do not resend the exact same expression; try continuing '
                      'with a different, smaller step instead.')


# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) nhieu lan trong cung kernel tung gay loi GPU het bo
# nho (da xac nhan qua thuc te chay tren Kaggle o ban 5 dang truoc).
print('Dung lai tok/llm da nap san.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D44_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai PART_HUONGGIAI va PART_FEWSHOT,
# giu nguyen PART_TOOL va PART_KIENTHUC.

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- If a stored result is a LIST (e.g. from `solve(...)` with more than one solution), you can reference one element of it directly by index instead of retyping it: `name = solve(...)` then `name[0]` for the first element, `name[1]` for the second, and so on; use this the moment you need one specific element again in a later expression (e.g. `x_expr.subs(y, name[0])`). Retyping a long value from an `Observation` by hand (especially one with nested radicals) is exactly where a digit or a sign gets copied wrong; indexing the stored list can never have that problem, so prefer it every time.
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. For most problem types this is decided by an exact symbolic proof. For this problem type specifically, it is decided by evaluating both sides to 50 significant decimal digits and checking they agree to that precision; this is not a formal proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True`; test all four, every time (see PART 4's matching step for why).
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher-degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation-solve mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non-real ones when the polynomial's real roots don't account for its full degree; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`; `Im` is not recognized and silently fails to evaluate) and discard it immediately if not. For a root confirmed real, use `re(root)`; not the raw value; in every later threshold or substitution, since even a numerically-real root can carry a residual non-zero imaginary part too small to matter but large enough to break a direct comparison. Any comparison built on such numeric roots (a threshold like `t > 0`, or a self-consistency check like `Abs(a - b) < 1e-6`) should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root; in particular the final count of solutions and matching it against the answer options; stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo; retyping a small variation of the same idea will keep producing the same error. Stop, go back to PART 2's method for this exact situation (a formula giving `zoo`/an undefined result almost always means a special case described somewhere in PART 2 applies here; e.g. a division that is only valid when some quantity is nonzero), and use the alternative formula PART 2 gives for that case, typed as a normal exact expression; never invent a placeholder word like `undefined` as if it were a value; the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$; there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change; so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).
20. **Vieta's formulas for a quadratic.** For $z^2+pz+q=0$ with roots $z_1,z_2$ (real or complex), the coefficients determine the roots' sum and product directly, without solving anything: $z_1+z_2=-p$ and $z_1z_2=q$. This holds whether the roots are real or a complex-conjugate pair (fact 15); the SAME two equations hold either way, since they simply come from matching coefficients in the identity $z^2+pz+q=(z-z_1)(z-z_2)=z^2-(z_1+z_2)z+z_1z_2$, which does not care whether $z_1,z_2$ happen to be real.
21. **Conjugate of a sum, and conjugate of a real multiple of $i$.** For any complex numbers $u,v$: $\overline{u+v}=\overline{u}+\overline{v}$ and $\overline{u-v}=\overline{u}-\overline{v}$ (conjugation distributes over addition/subtraction, since conjugating just flips the sign of every imaginary part, and imaginary parts add/subtract termwise). In particular, for a REAL constant $c$: $\overline{ci}=-ci$ (because $ci$ is purely imaginary with imaginary part $c$, flipping its sign gives $-ci$). Combining these: for any complex $z$ and real constant $c$, $\overline{z+ci}=\overline{z}+\overline{ci}=\overline{z}-ci$. This is the key trick that lets you replace $\overline{z}-ci$ by $\overline{z+ci}$ wherever it appears, which (by fact 4, $|\overline{w}|=|w|$) means $|\overline{z}-ci|=|z+ci|$; turning an expression that LOOKS like it needs both $z$ and $\overline{z}$ separately into one that depends only on the single quantity $z+ci$.
22. **Sum and difference of a complex number and its conjugate.** For $z=x+yi$ ($x,y$ real): combining fact 1 ($z=x+yi$) and fact 3 ($\overline{z}=x-yi$) directly by addition and subtraction gives $z+\overline{z}=2x$ and $z-\overline{z}=2yi$. This is the key move whenever a condition mixes $z$ and $\overline{z}$ through a sum or difference: $z+\overline{z}$ collapses to the single REAL number $2x$ (twice the real part, with no $y$ or $i$ left in it at all), and $z-\overline{z}$ collapses to the single PURELY IMAGINARY number $2yi$ (so $|z-\overline{z}|=2|y|$).
23. **The general equation of a circle, and reading off its center and radius.** A circle's equation always expands to the form $x^2+y^2-2ax-2by+c=0$ for real constants $a,b,c$, once the coefficients of $x^2$ and $y^2$ have been made equal to $1$ (divide the whole equation through by whatever that common coefficient is, if it is not already $1$). Completing the square on both $x$ and $y$ shows this is exactly $(x-a)^2+(y-b)^2=a^2+b^2-c$: the circle centered at $(a,b)$ with radius $\sqrt{a^2+b^2-c}$ (this is always a genuine positive number when the original equation truly describes a circle). So once an equation has been brought to this normalized form, the center's coordinates are read directly from the coefficients of $x$ and $y$ (each divided by $-2$), and the radius follows from the single formula above.
'''


PART_HUONGGIAI = r'''
**PART 2: HOW TO SOLVE THIS PROBLEM TYPE**

**Isolating $z$ first.** The given relation always has the shape $w=\dfrac{A+iz}{1+z}$ for some real constant $A$. Clearing the denominator by multiplying both sides by $1+z$ gives $w(1+z)=A+iz$, i.e. $w+wz=A+iz$. Gathering every term containing $z$ onto one side and everything else onto the other gives $w-A=iz-wz=(i-w)z$: a single equation expressing $z$ in terms of $w$ (up to the factor $(i-w)$), with no fraction left in it at all.

**Taking the modulus of both sides.** By fact 12, $|uv|=|u||v|$ for any complex $u,v$. Applying this to $w-A=(i-w)z$ gives $|w-A|=|i-w|\cdot|z|$. Since the problem always states $|z|=R$ for some given real number $R$, this becomes $|w-A|=|i-w|\cdot R$: an equation relating only $w$ and the known constants $A,R$, with $z$ gone entirely.

**Turning this into an equation in $x,y$.** Write $w=x+yi$ with $x,y$ real (fact 1). Then $w-A=(x-A)+yi$, so $|w-A|=\sqrt{(x-A)^2+y^2}$ (fact 4). Likewise $i-w=-x+(1-y)i$ (the real part of $w$ flips sign, and the imaginary part is subtracted from the coefficient of $i-w$'s own imaginary part, $1$), so $|i-w|=\sqrt{x^2+(1-y)^2}$. The equation $|w-A|=|i-w|\cdot R$ becomes $\sqrt{(x-A)^2+y^2}=\sqrt{x^2+(1-y)^2}\cdot R$.

**Squaring to remove both square roots.** Squaring both sides (the constant $R$ on the right gets squared too, since it multiplies the whole square root) gives $(x-A)^2+y^2=R^2\big(x^2+(1-y)^2\big)$: a genuine polynomial equation in $x,y$, with no remaining square roots.

**Bringing it to the standard circle form.** Expanding both sides and moving everything to one side gives a polynomial in $x^2,y^2,x,y$ and constants. Because the equation came from setting two moduli equal, the coefficients of $x^2$ and $y^2$ always turn out equal to each other (call this common coefficient the leading coefficient); dividing the whole equation through by it brings the equation to the standard form $x^2+y^2-2ax-2by+c=0$ from fact 23. Reading off $a,b,c$ from this normalized equation (the coefficient of $x$ divided by $-2$ gives $a$; the coefficient of $y$ divided by $-2$ gives $b$; the constant term left over after removing every $x$ and $y$ term is $c$) and applying fact 23's formula gives the center $(a,b)$ and, most importantly for this problem type, the radius $\sqrt{a^2+b^2-c}$ directly.
'''


PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (ReAct format)**

Đề bài: Xét số phức $z$ thỏa mãn $|z|=\sqrt{2}$. Trên mặt phẳng tọa độ $Oxy$, tập hợp điểm biểu diễn các số phức $w=\dfrac{3+iz}{1+z}$ là một đường tròn có bán kính bằng
A. $2\sqrt{3}$.   B. $20$.   C. $12$.   D. $2\sqrt{5}$.

<think>
Thought: The relation quoted verbatim is "$w=\dfrac{3+iz}{1+z}$"; the constant added to $iz$ in the numerator is $3$, so $A=3$. I store it.
Action: Calculator
Action Input: A = 3
Observation: A = 3

Thought: The modulus condition quoted verbatim is "$|z|=\sqrt{2}$"; so $R=\sqrt{2}$. I store it.
Action: Calculator
Action Input: R = sqrt(2)
Observation: R = sqrt(2)

Thought: Following PART 2's derivation, the squared equation is $(x-A)^2+y^2=R^2\big(x^2+(1-y)^2\big)$. I build the expression for "left side minus right side" and expand it; once this equals $0$, it is the circle's equation.
Action: Calculator
Action Input: eq_expr = expand((x-A)**2+y**2 - R**2*(x**2+(1-y)**2))
Observation: eq_expr = -x**2 - 6*x - y**2 + 4*y + 7

Thought: To bring this to the standard form $x^2+y^2-2ax-2by+c=0$, I first need the common coefficient of $x^2$ (and $y^2$) so I can divide it out.
Action: Calculator
Action Input: leading = eq_expr.coeff(x, 2)
Observation: leading = -1

Thought: Dividing the whole equation by this leading coefficient normalizes the $x^2,y^2$ coefficients to $1$.
Action: Calculator
Action Input: eq_norm = expand(eq_expr/leading)
Observation: eq_norm = x**2 + 6*x + y**2 - 4*y - 7

Thought: Now I read off the coefficient of $x$ from this normalized equation, to get $a$ from fact 23 in a moment.
Action: Calculator
Action Input: coef_x = eq_norm.coeff(x, 1)
Observation: coef_x = 6

Thought: Likewise the coefficient of $y$, for $b$.
Action: Calculator
Action Input: coef_y = eq_norm.coeff(y, 1)
Observation: coef_y = -4

Thought: And the constant term $c$: substituting $x=0,y=0$ into the normalized equation leaves exactly the constant term, since every other term contains $x$ or $y$.
Action: Calculator
Action Input: c_val = eq_norm.subs({x: 0, y: 0})
Observation: c_val = -7

Thought: By fact 23, $a$ is the coefficient of $x$ divided by $-2$.
Action: Calculator
Action Input: a_val = -coef_x/2
Observation: a_val = -3

Thought: And $b$ is the coefficient of $y$ divided by $-2$.
Action: Calculator
Action Input: b_val = -coef_y/2
Observation: b_val = 2

Thought: Now fact 23's radius formula, $\sqrt{a^2+b^2-c}$, using the three values just found.
Action: Calculator
Action Input: radius = sqrt(a_val**2 + b_val**2 - c_val)
Observation: radius = 2*sqrt(5)

Thought: `radius` is confirmed to be $2\sqrt{5}$. Now I go through the four options one at a time, carefully, in order; I quote each option's exact text before testing it, and I test every single one, even after a match is found, since `Eq()` here is a numeric check, not a formal proof. Starting with option A: quoted verbatim, "A. $2\sqrt{3}$."; A is $2\sqrt{3}$. Test it.
Action: Calculator
Action Input: Eq(radius, 2*sqrt(3))
Observation: False

Thought: A did not match. Option B, quoted verbatim: "B. $20$."; B is $20$. Test it.
Action: Calculator
Action Input: Eq(radius, 20)
Observation: False

Thought: B did not match. Option C, quoted verbatim: "C. $12$."; C is $12$. Test it.
Action: Calculator
Action Input: Eq(radius, 12)
Observation: False

Thought: C did not match. Option D, quoted verbatim: "D. $2\sqrt{5}$."; D is $2\sqrt{5}$. Test it.
Action: Calculator
Action Input: Eq(radius, 2*sqrt(5))
Observation: True

Thought: All four options are now tested. Exactly one came back True: D. That is the answer.
</think>

Final Answer: \boxed{D}
'''


PART_NHIEMVU = r'''
**PART 4: YOUR TASK, STEP BY STEP**

The problem always has the form: given $|z|=R$, the set of points representing $w=\dfrac{A+iz}{1+z}$ is a circle; what is its radius? Follow these steps, using the Calculator for every computation.

1. **Finding $A$ and $R$.** Quote the relation for $w$ verbatim and read off the real constant added to $iz$ in the numerator as $A$ (keep whatever sign is shown). Quote the modulus condition verbatim and read off $R$. Store both.
2. **Building the squared equation.** Compute `eq_expr = expand((x-A)**2+y**2 - R**2*(x**2+(1-y)**2))`. This is "left side squared minus right side squared" from PART 2's derivation; once `eq_expr` equals $0$, it is the circle's equation.
3. **Finding the leading coefficient.** Compute `leading = eq_expr.coeff(x, 2)`, the coefficient of $x^2$ in `eq_expr`.
4. **Normalizing.** Compute `eq_norm = expand(eq_expr/leading)`. This has coefficient $1$ on both $x^2$ and $y^2$, matching fact 23's standard form.
5. **Reading off $a,b,c$.** Compute `coef_x = eq_norm.coeff(x, 1)` and `coef_y = eq_norm.coeff(y, 1)`, then `c_val = eq_norm.subs({x: 0, y: 0})`. From these, compute `a_val = -coef_x/2` and `b_val = -coef_y/2`.
6. **The radius.** Compute `radius = sqrt(a_val**2 + b_val**2 - c_val)`, per fact 23.
7. For each option A, B, C, D in turn: read that option's value VERBATIM from the problem text, then compare with `Eq(radius, <that option's value>)`. Test every option even after one already returns `True`; `Eq()` here is a numeric check, not a formal proof.
8. Write the final line `Final Answer: \boxed{<Letter>}` using whichever option matched.

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

In [ ]:
# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# (MayTinh/fast_simplify/is_zero da duoc dinh nghia o Cell 3, truoc khi
# LLM(...) khoi tao - xem giai thich an toan CUDA/fork o do)
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 30          # quy trinh day du ~11 luot tinh (A,R,eq_expr,leading,eq_norm,coef_x,coef_y,c_val,a_val,b_val,radius) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')
FINAL_ANSWER_RE = re.compile(r'Final\s+Answer\s*:\s*\\boxed\{\s*[A-D]\s*\}')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        m_final = FINAL_ANSWER_RE.search(s.full_text)
        if m_final:
            s.full_text = s.full_text[:m_final.end()]
            s.xong = True
            s.ly_do_dung = 'model_ket_thuc'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        khop = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            khop = mm
        bieu_thuc = khop.group(1).strip() if khop else ''

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)

## Dang D45

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

OUT_PATH  = '/kaggle/working/d45_react_calculator_full90_DeepSeek-R1-Distill-Qwen-1.5B.csv'

MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = {'m'}  # m: tham so thuc trong phuong trinh, an so can giai

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/11-dang-full/plan_solve_prompts_11dang_merged.json'  # SUA NEU KHAC
with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D45']
print('So cau:', len(records))

# ---- Backend Calculator: dat O DAY, TRUOC khi LLM(...)/CUDA khoi tao ----
# (an toan multiprocessing.fork - xem giai thich trong comment ben duoi)
import sympy as sp
import multiprocessing as mp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Luoi an toan CHO CA BATCH: du da dung radsimp() de tranh treo may o hau
# het truong hop, sympy van khong dam bao toc do cho MOI to hop can thuc
# bat ky - chi can 1/90 cau roi vao truong hop xau la ca batch nghen theo
# (Calculator chay tuan tu tung cau). Dung TIEN TRINH CON that (co the bi
# giet cuong buc bang tin hieu he dieu hanh) thay vi thread: thread chi
# ngat duoc tai diem GIL duoc nhuong lai, KHONG dam bao neu tinh toan ket
# sau trong 1 loi goi C lien tuc (da xac nhan qua thuc te chay tren
# Kaggle). Pool tien trinh con nay duoc tao NGAY TAI DAY, TRUOC KHI
# LLM(...)/CUDA khoi tao ben duoi - vi vay an toan tuyet doi voi
# multiprocessing.fork (fork() SAU KHI CUDA da khoi tao moi la nguy hiem,
# tung gay treo o mot lan thu truoc do).
TIMEOUT_SECONDS = 20


def fast_simplify(expr):
    """Rut gon nhanh va ON DINH hon sp.simplify() thuan tuy.

    sp.simplify() la ham "thu tat ca chien luoc roi chon ket qua ngan nhat",
    rat cham (co the treo may) khi bieu thuc co nhieu MAU SO chua can bac
    hai khac goc (vd tu phep chia (Bx-Ay)/(Bx-Ax)) - dung sinh ra qua nhieu
    dang the hien khac nhau ma khong bao gio hop nhat lai. radsimp() giai
    quyet dung goc van de nay: no huu ti hoa mau so chua can NGAY LAP TUC,
    nen ket qua o moi buoc luon o dang gon, khong de cac mau can long tich
    luy qua tung phep tinh tiep theo. Dung radsimp() lam buoc rut gon CHINH
    (nhanh, gan nhu luon du); chi roi sang simplify() lam buoc du phong khi
    radsimp() chua dua duoc ve dang 0/dang gon nhat.
    """
    return sp.expand(sp.radsimp(expr))


def is_zero(expr):
    """Kiem tra bieu thuc co bang 0 khong.

    Buoc dau (radsimp) bat duoc phan lon truong hop that nhanh va tuyet
    doi chinh xac. Neu chua ket luan duoc, KHONG roi sang sp.simplify()
    (chung minh dai so - cham, khong dam bao toc do, day la duong tung
    gay treo may). Thay vao do, so sanh gia tri SO HOC voi do chinh xac
    rat cao (50 chu so thap phan) - dung nguyen tac may tinh Casio: so
    hai so thap phan thay vi chung minh dang thuc dai so. Voi mien bai
    toan nay (cac hang so dai so co dinh tu de bai, khong phai gia tri
    adversarial), sai so gan nhu khong the xay ra o do chinh xac nay.
    """
    rut_gon = sp.radsimp(expr)
    if rut_gon == 0:
        return True
    return abs(complex(rut_gon.evalf(50))) < 1e-40


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            # 'Eq' duoc thay bang phien ban evaluate=False: sp.Eq() mac dinh
            # TU DONG thu kiem tra 2 ve co bang nhau NGAY LUC KHOI TAO (co che
            # rieng cua sympy, khac hoan toan ham is_zero() tu viet ben duoi) -
            # voi bieu thuc can long phuc tap, chinh buoc TU DONG nay co the
            # treo may, va treo TRUOC CA KHI chay_co_timeout kip can thiep (vi
            # no xay ra ngay trong luc sympify dang parse chuoi). Dung ban
            # evaluate=False de hoan toan doi viec so khop cho ham is_zero() -
            # da duoc kiem chung nhanh va dang tin cay - dam nhiem.
            ns_de_parse = dict(self.ns)
            ns_de_parse['Eq'] = lambda a, b: sp.Eq(a, b, evaluate=False)
            parsed = sp.sympify(s, locals=ns_de_parse)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau, loi_timeout = chay_co_timeout(is_zero, parsed.lhs - parsed.rhs)
                if loi_timeout:
                    return None, loi_timeout
                ket_qua_bool = sp.true if bang_nhau else sp.false
                if ten:
                    self.ns[ten] = ket_qua_bool
                    return f'{ten} = {ket_qua_bool}', None
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri, loi_timeout = chay_co_timeout(fast_simplify, parsed)
            if loi_timeout:
                return None, loi_timeout
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

try:
    _CALC_POOL = mp.get_context('fork').Pool(1)
except ValueError:
    _CALC_POOL = None  # khong co fork (vd may local Windows) -> chay khong timeout


def chay_co_timeout(ham, *args):
    """Chay ham(*args) trong tien trinh con (fork, tao TRUOC CUDA nen an
    toan), gioi han TIMEOUT_SECONDS. Neu qua han, HUY va TAO LAI pool (vi
    tien trinh con cu van con chay ngam, khong the tai su dung duoc nua)
    roi tra ve loi ro rang thay vi treo may."""
    global _CALC_POOL
    if _CALC_POOL is None:
        return ham(*args), None
    ar = _CALC_POOL.apply_async(ham, args)
    try:
        return ar.get(timeout=TIMEOUT_SECONDS), None
    except mp.TimeoutError:
        _CALC_POOL.terminate()
        _CALC_POOL = mp.get_context('fork').Pool(1)
        return None, (f'computation timed out after {TIMEOUT_SECONDS}s '
                      '(the expression is too complex to simplify exactly). '
                      'Do not resend the exact same expression; try continuing '
                      'with a different, smaller step instead.')


# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) nhieu lan trong cung kernel tung gay loi GPU het bo
# nho (da xac nhan qua thuc te chay tren Kaggle o ban 5 dang truoc).
print('Dung lai tok/llm da nap san.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D45_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai PART_HUONGGIAI va PART_FEWSHOT,
# giu nguyen PART_TOOL va PART_KIENTHUC.

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- If a stored result is a LIST (e.g. from `solve(...)` with more than one solution), you can reference one element of it directly by index instead of retyping it: `name = solve(...)` then `name[0]` for the first element, `name[1]` for the second, and so on; use this the moment you need one specific element again in a later expression (e.g. `x_expr.subs(y, name[0])`). Retyping a long value from an `Observation` by hand (especially one with nested radicals) is exactly where a digit or a sign gets copied wrong; indexing the stored list can never have that problem, so prefer it every time.
- You can also build a list yourself, by hand, not only receive one from `solve(...)`: write `name = [val1, val2, val3, val4]` to store several numbers together in a single call, then index into it the same way (`name[0]`, `name[1]`, ...; see PART 4 for when to use this).
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. This Calculator decides `Eq(...)` by evaluating both sides to 50 significant decimal digits and checking they agree to that precision; this is not a formal symbolic proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True`; test all four, every time (see PART 4's matching step for why). Just like any other computation, `Eq(...)` can be given a name too, e.g. `test_A = Eq(left, right)`; the `True`/`False` result is then genuinely stored under that name and can be referred to again later by that name, exactly like a numeric result.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate a value yourself by estimating or rounding it in your head. `solve(...)` always returns an exact closed form for the equations that appear in these problems, so a numeric-only fallback is never needed to obtain a result; the one exception is when a step of PART 2's method explicitly calls for reading the SIGN of an already-exact value via `.evalf()` (covered elsewhere in this PART, when this problem type needs it) — that still starts from an exact value computed by the Calculator, it only converts it to a decimal so its sign is easy to read, which is not the same as you approximating a value yourself.

**NEVER solve a system of equations all at once.** Calling `solve([eq1, eq2], [x, y])` with a LIST of equations and a LIST of unknowns is unreliable: depending on the exact expressions involved, it can return a dictionary instead of a list, and this Calculator cannot handle a dictionary result (it will error). Instead, always solve ONE equation for ONE unknown at a time: solve the first equation for one unknown (giving a list with one expression, possibly in terms of the other unknown), substitute that expression into the second equation, then solve THAT for the remaining unknown. This always returns a plain list, exactly like every other use of `solve(...)` already covered above.

**Reading the SIGN of a single number, and counting integers in an open interval.** For an expression that may involve nested radicals (so its sign is not obvious just from looking at it), call `.evalf()` on it, e.g. `d0_num = d0.evalf()`; the Observation is then a plain decimal, and reading whether ONE printed decimal is positive or negative is a simple reading task, not arithmetic. Separately, to count how many integers lie STRICTLY BETWEEN two exact bounds `lo` and `hi` (whichever is genuinely smaller/larger; use `Min(a, b)` and `Max(a, b)` if you are not sure which of two values is which), the count is always `ceiling(hi) - floor(lo) - 1` as ONE Calculator call; this formula is exact whether or not `lo`/`hi` happen to be integers themselves, so never enumerate or count the integers yourself by hand.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo; retyping a small variation of the same idea will keep producing the same error. Stop and re-read PART 2's method for this exact step before trying again, rather than guessing another small variation of the same idea; never invent a placeholder word like `undefined` as if it were a value, since the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$; there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change; so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).
20. **Vieta's formulas for a quadratic.** For $z^2+pz+q=0$ with roots $z_1,z_2$ (real or complex), the coefficients determine the roots' sum and product directly, without solving anything: $z_1+z_2=-p$ and $z_1z_2=q$. This holds whether the roots are real or a complex-conjugate pair (fact 15); the SAME two equations hold either way, since they simply come from matching coefficients in the identity $z^2+pz+q=(z-z_1)(z-z_2)=z^2-(z_1+z_2)z+z_1z_2$, which does not care whether $z_1,z_2$ happen to be real.
21. **Conjugate of a sum, and conjugate of a real multiple of $i$.** For any complex numbers $u,v$: $\overline{u+v}=\overline{u}+\overline{v}$ and $\overline{u-v}=\overline{u}-\overline{v}$ (conjugation distributes over addition/subtraction, since conjugating just flips the sign of every imaginary part, and imaginary parts add/subtract termwise). In particular, for a REAL constant $c$: $\overline{ci}=-ci$ (because $ci$ is purely imaginary with imaginary part $c$, flipping its sign gives $-ci$). Combining these: for any complex $z$ and real constant $c$, $\overline{z+ci}=\overline{z}+\overline{ci}=\overline{z}-ci$. This is the key trick that lets you replace $\overline{z}-ci$ by $\overline{z+ci}$ wherever it appears, which (by fact 4, $|\overline{w}|=|w|$) means $|\overline{z}-ci|=|z+ci|$; turning an expression that LOOKS like it needs both $z$ and $\overline{z}$ separately into one that depends only on the single quantity $z+ci$.
22. **Sum and difference of a complex number and its conjugate.** For $z=x+yi$ ($x,y$ real): combining fact 1 ($z=x+yi$) and fact 3 ($\overline{z}=x-yi$) directly by addition and subtraction gives $z+\overline{z}=2x$ and $z-\overline{z}=2yi$. This is the key move whenever a condition mixes $z$ and $\overline{z}$ through a sum or difference: $z+\overline{z}$ collapses to the single REAL number $2x$ (twice the real part, with no $y$ or $i$ left in it at all), and $z-\overline{z}$ collapses to the single PURELY IMAGINARY number $2yi$ (so $|z-\overline{z}|=2|y|$).
'''


PART_HUONGGIAI = r'''
**PART 2: HOW TO SOLVE THIS PROBLEM TYPE**

**The equation and its discriminant.** The problem always gives $z^2-2mz+c(m)=0$ where $c(m)$ is some fixed real-linear (or with fixed radical constants) expression in the real parameter $m$, read directly off the equation. Writing this in the standard form $z^2+bz+c=0$ with $b=-2m$, fact 15 gives $\Delta'=(b/2)^2-c=m^2-c(m)$; this is itself a quadratic in $m$ with leading coefficient $1$ (always positive), so as a function of $m$ it is an upward parabola.

**Two cases, split by the sign of $\Delta'$.** By fact 15/16: if $\Delta'>0$, the equation has two DISTINCT REAL roots $z_1,z_2$; if $\Delta'<0$, it has two roots that are a complex-conjugate pair; $\Delta'=0$ gives a single repeated root, which is never "two distinct roots $z_1,z_2$" and never satisfies the problem's requirement, so it is excluded from both cases.

**Case $\Delta'>0$ (two distinct real roots).** For two REAL numbers, $|z_1|=|z_2|$ means they are equal or exactly opposite; since they are already required to be distinct, the only possibility is $z_1=-z_2$, i.e. $z_1+z_2=0$. By Viète's formulas the sum of the roots of $z^2-2mz+c(m)=0$ is $2m$, so $z_1+z_2=0 \Leftrightarrow 2m=0 \Leftrightarrow m=0$. This single candidate $m=0$ must still be checked against the case condition $\Delta'>0$: plug $m=0$ into $\Delta'$ and read its sign; only if it comes out positive does $m=0$ actually belong to this case and count.

**Case $\Delta'<0$ (conjugate-pair roots).** By fact 16, whenever the two roots of a real-coefficient quadratic are a genuine complex-conjugate pair (not real), $|z_1|=|z_2|$ holds AUTOMATICALLY, with no further condition to check; EVERY value of $m$ satisfying $\Delta'<0$ counts. Since $\Delta'(m)=m^2-c(m)$ is an upward parabola, $\Delta'<0$ holds exactly for $m$ strictly between the two roots of $\Delta'(m)=0$ (an open interval); every integer strictly inside that interval is a valid $m$ for this case.

**Counting.** The final count is: (1 if $m=0$ passes the case-$1$ check, else 0) PLUS (the number of integers strictly inside the open interval bounded by $\Delta'$'s two roots). Solve $\Delta'(m)=0$ once to get both boundary values, then use the exact integer-counting technique described in the tool instructions above (no need to know by inspection which boundary is smaller; `Min`/`Max` sort them, and `ceiling(hi) - floor(lo) - 1` counts the interior integers exactly, whether or not the boundaries themselves are integers).
'''


PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (ReAct format)**

Đề bài: Trên tập hợp các số phức, xét phương trình $z^2 - 2mz + 8m - 12 = 0$ ($m$ là tham số thực). Có bao nhiêu giá trị nguyên của $m$ để phương trình đó có hai nghiệm phân biệt $z_1, z_2$ thỏa mãn $|z_1| = |z_2|$?

A. $5$.   B. $6$.   C. $3$.   D. $4$.

<think>
Thought: The constant part after $-2mz$, quoted verbatim from "$z^2-2mz+8m-12=0$", is $8m-12$. This is $c(m)$; I store it.
Action: Calculator
Action Input: c_expr = 8*m-12
Observation: c_expr = 8*m - 12

Thought: By PART 2, $\Delta'=m^2-c(m)$; I expand it now.
Action: Calculator
Action Input: delta_p = expand(m**2 - c_expr)
Observation: delta_p = m**2 - 8*m + 12

Thought: Case 1 ($\Delta'>0$, real distinct roots): by Viète the only candidate is $m=0$, but it must first pass the case condition. I substitute $m=0$ into $\Delta'$.
Action: Calculator
Action Input: delta_0 = delta_p.subs(m, 0)
Observation: delta_0 = 12

Thought: To read its sign safely rather than guessing from the symbolic form, I convert it to a decimal.
Action: Calculator
Action Input: delta_0_num = delta_0.evalf()
Observation: delta_0_num = 12.0000000000000

Thought: $12$ is positive, so $\Delta'(0)>0$: $m=0$ passes the case-1 condition and contributes exactly $1$ to the final count. Moving to Case 2 ($\Delta'<0$, conjugate pair, always satisfies $|z_1|=|z_2|$): I solve $\Delta'(m)=0$ for its two boundary values.
Action: Calculator
Action Input: roots = solve(Eq(delta_p, 0), m)
Observation: roots = [2, 6]

Thought: I do not assume which of these is smaller by eye; I sort them explicitly with `Min`/`Max`.
Action: Calculator
Action Input: lo_val = Min(roots[0], roots[1])
Observation: lo_val = 2

Thought: And the larger one.
Action: Calculator
Action Input: hi_val = Max(roots[0], roots[1])
Observation: hi_val = 6

Thought: By the counting formula from the tool instructions, the number of integers strictly between `lo_val` and `hi_val` is `ceiling(hi_val) - floor(lo_val) - 1`, computed as one call, never by listing them out myself.
Action: Calculator
Action Input: count_th2 = ceiling(hi_val) - floor(lo_val) - 1
Observation: count_th2 = 3

Thought: The total count is Case 1's contribution (confirmed to be $1$ above) plus Case 2's contribution, `count_th2`.
Action: Calculator
Action Input: count_total = 1 + count_th2
Observation: count_total = 4

Thought: Before testing anything, here are all four options copied fresh, side by side, exactly as printed: A: "$5$."; B: "$6$."; C: "$3$."; D: "$4$." I store all four together in one list, in order.
Action: Calculator
Action Input: opts = [5, 6, 3, 4]
Observation: opts = [5, 6, 3, 4]

Thought: Now I test `count_total` against each stored option by index, in order, even after a match is found. Option A is `opts[0]`.
Action: Calculator
Action Input: test_A = Eq(count_total, opts[0])
Observation: test_A = False

Thought: Option B is `opts[1]`.
Action: Calculator
Action Input: test_B = Eq(count_total, opts[1])
Observation: test_B = False

Thought: Option C is `opts[2]`.
Action: Calculator
Action Input: test_C = Eq(count_total, opts[2])
Observation: test_C = False

Thought: Option D is `opts[3]`.
Action: Calculator
Action Input: test_D = Eq(count_total, opts[3])
Observation: test_D = True

Thought: Checking every result against the quotes copied at the start: only `test_D` is `True`. D is the only option consistent with `count_total`. That is the answer.
</think>

Final Answer: \boxed{D}
'''


PART_NHIEMVU = r'''
**PART 4: YOUR TASK, STEP BY STEP**

The problem always gives $z^2-2mz+c(m)=0$ ($m$ a real parameter) and asks for the count of integer $m$ making the equation have two distinct roots $z_1,z_2$ with $|z_1|=|z_2|$. Follow these steps, using the Calculator for every computation.

1. **Reading $c(m)$.** Quote the equation verbatim, read off the expression added after $-2mz$ (keeping its exact sign and every term, copying only what is literally printed — $m$ and $c(m)$ are always real here, so never insert the imaginary unit `I` anywhere in `c_expr` unless the source term itself literally contains an $i$), and store it as `c_expr`.
2. **Computing $\Delta'$.** Compute `delta_p = expand(m**2 - c_expr)`.
3. **Case 1 ($\Delta'>0$): checking the single candidate $m=0$.** Compute `delta_0 = delta_p.subs(m, 0)`, then `delta_0_num = delta_0.evalf()` to get a plain decimal. Read its sign: if positive, $m=0$ passes and contributes exactly $1$ to the final total; if zero or negative, it contributes $0$. Fold this reading into the Thought that introduces your next Action; do not create an extra Action just to state it.
4. **Case 2 ($\Delta'<0$): finding the boundary values.** Compute `roots = solve(Eq(delta_p, 0), m)`. Do not assume which entry is smaller; explicitly sort with `lo_val = Min(roots[0], roots[1])` and `hi_val = Max(roots[0], roots[1])`.
5. **Counting Case 2's integers.** Compute `count_th2 = ceiling(hi_val) - floor(lo_val) - 1` as ONE Calculator call. This formula is exact for the OPEN interval `(lo_val, hi_val)` whether or not the bounds themselves are integers; never list out or count the integers by hand.
6. **Combining.** Compute `count_total = <0 or 1 from step 3> + count_th2` (write the literal `0` or `1` you determined in step 3).
7. **Copying all four options fresh, together, before testing any of them.** In the Thought that introduces your first test, copy all four options' exact text side by side (A: "...", B: "...", C: "...", D: "..."), then store all four numbers together in ONE Calculator call as a list: `opts = [<A>, <B>, <C>, <D>]`.
8. **Testing each option by index.** For each option A, B, C, D in turn: compare with `Eq(count_total, opts[<matching index>])`. Test every option even after one already returns `True`.
9. **Deciding.** Exactly one option should be `True`; that letter is the answer. Write the final line `Final Answer: \boxed{<Letter>}` using that letter.

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

In [ ]:
# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# (MayTinh/fast_simplify/is_zero da duoc dinh nghia o Cell 3, truoc khi
# LLM(...) khoi tao - xem giai thich an toan CUDA/fork o do)
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 23          # quy trinh day du ~9 luot tinh (c_expr,delta_p,delta_0,delta_0_num,roots,lo_val,hi_val,count_th2,count_total,opts) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')
FINAL_ANSWER_RE = re.compile(r'Final\s+Answer\s*:\s*\\boxed\{\s*[A-D]\s*\}')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        m_final = FINAL_ANSWER_RE.search(s.full_text)
        if m_final:
            s.full_text = s.full_text[:m_final.end()]
            s.xong = True
            s.ly_do_dung = 'model_ket_thuc'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        cac_khop = list(ACTION_INPUT_RE.finditer(o.text))
        khop = cac_khop[0] if cac_khop else None
        bieu_thuc = khop.group(1).strip() if khop else ''
        if len(cac_khop) > 1:
            vi_tri_cat = len(s.full_text) - len(o.text) + khop.end()
            s.full_text = s.full_text[:vi_tri_cat]

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)

## Dang D47

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

OUT_PATH  = '/kaggle/working/d47_react_calculator_full90_DeepSeek-R1-Distill-Qwen-1.5B.csv'

MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = {'m'}

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/11-dang-full/plan_solve_prompts_11dang_merged.json'  # SUA NEU KHAC
with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D47']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) nhieu lan trong cung kernel tung gay loi GPU het bo
# nho (da xac nhan qua thuc te chay tren Kaggle o ban 5 dang truoc).
print('Dung lai tok/llm da nap san.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D47_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai [C] PART_HUONGGIAI va
# [D] PART_FEWSHOT, giu nguyen [A] PART_TOOL va [B] PART_KIENTHUC.

# [A] PART_TOOL - HUONG DAN DUNG TOOL (DUNG CHUNG CHO MOI DANG TOAN)
# =====================================================================
PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- To test whether two expressions are **exactly** equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`, decided exactly; never approximately.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher-degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation-solve mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non-real ones when the polynomial's real roots don't account for its full degree; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`; `Im` is not recognized and silently fails to evaluate) and discard it immediately if not. For a root confirmed real, use `re(root)`; not the raw value; in every later threshold or substitution, since even a numerically-real root can carry a residual non-zero imaginary part too small to matter but large enough to break a direct comparison. Any comparison built on such numeric roots (a threshold like `t > 0`, or a self-consistency check like `Abs(a - b) < 1e-6`) should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root; in particular the final count of solutions and matching it against the answer options; stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


# =====================================================================
# [B] PART_KIENTHUC - KIEN THUC NEN SO PHUC (DUNG CHUNG MOI DANG SO PHUC)
# =====================================================================
PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
'''


# =====================================================================
# [C] PART_HUONGGIAI - HUONG GIAI RIENG CUA DANG D47 (THAY KHI DOI DANG)
# =====================================================================
PART_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: a quadratic equation $z^2-2(m+a)z+m^2=0$ with a real parameter $m$ ($a$ a given real constant), and a given modulus $R>0$. Count how many values of $m$ make some root $z_0$ of the equation satisfy $|z_0|=R$.

Follow this method exactly; do not invent a shorter route, and do not skip any step below.

1. **Read off $a$ and $R$.** Match the given equation against the standard form $z^2-2(m+a)z+m^2=0$: whatever is added to $m$ inside the parentheses is $a$ (it can be negative, e.g. $z^2-2(m-4)z+m^2=0$ means $a=-4$). $R$ is the given modulus on the right of $|z_0|=R$.
2. **Compute $\Delta'$.** For $z^2+bz+c=0$, $\Delta'=(b/2)^2-c$; here $b/2=-(m+a)$ and $c=m^2$, so $\Delta'=(m+a)^2-m^2$, a quadratic expression in $m$. Build it with the Calculator and keep $m$ symbolic; do not substitute a number for $m$ yet. Initialize `count = 0`; this running tally is the only place `count` is ever set directly.
3. **Case 1: $\Delta'=0$.** Solve $\Delta'=0$ for $m$ (this typically gives one value). For that $m$, the equation has a real double root $z_0=-b/2=a+m$. Check $|z_0|=R$ with `Eq(Abs(z0), R)`. If it passes, increment `count = count + 1`; if not, discard.
4. **Case 2: $\Delta'<0$.** By the complex-roots fact from PART 1, write the root formula: $z_0=(a+m)\pm\sqrt{|\Delta'|}\,i$. Since $\Delta'<0$ here, $|\Delta'|=-\Delta'$, so $z_0=(a+m)\pm\sqrt{-\Delta'}\,i$; this is the real part and imaginary part of $z_0$. Square the modulus term by term: $|z_0|^2=(a+m)^2+\bigl(\sqrt{-\Delta'}\bigr)^2=(a+m)^2+(-\Delta')=(a+m)^2-\Delta'$. Now expand and simplify by substituting $\Delta'=(a+m)^2-m^2$: $|z_0|^2=(a+m)^2-\bigl[(a+m)^2-m^2\bigr]=m^2$. This is a general fact, true for every equation of this exact family regardless of $a$: the squared modulus of the complex root always equals $m^2$, exactly the constant term of the equation. So $|z_0|=R \iff m^2=R^2 \iff m=R$ or $m=-R$. For each of these two candidate values, check whether it actually satisfies $\Delta'<0$ (substitute it into $\Delta'$ and test `< 0`, since this is a strict inequality on a specific number, not an exact equality). If it passes, increment `count = count + 1`; if not, discard.
5. **Case 3: $\Delta'>0$.** The equation has two distinct real roots, so $|z_0|=R \iff z_0=R$ or $z_0=-R$. Substitute each of these two concrete numbers for $z$ directly into the ORIGINAL equation $z^2-2(m+a)z+m^2=0$ (never into $\Delta'$), producing a new quadratic equation in $m$; solve each for $m$ (each may give 0, 1, or 2 real values of $m$; if `solve(...)` returns an empty list, that branch simply contributes nothing, move on). For every $m$ found this way, check whether it satisfies $\Delta'>0$. If it passes, increment `count = count + 1`; if not, discard.
6. **Why no value of $m$ is ever double-counted.** Case 1 requires $\Delta'=0$, Case 2 requires $\Delta'<0$, Case 3 requires $\Delta'>0$; these three conditions on $m$ are mutually exclusive (a single value of $m$ satisfies exactly one of them, never two), so a value accepted in one case can never also be accepted in another; adding up the counts from all three cases directly, with no cross-checking between cases, is always correct.
7. **Match.** Once all three cases have been processed, `count` already holds the exact number of valid $m$. Test the four options in order with `Eq(count, <option>)` and stop at the first `True`; never pick the letter by eye.
'''


# =====================================================================
# [D] PART_FEWSHOT - VI DU MAU RIENG CUA DANG D47 (THAY KHI DOI DANG)
# Cac dong "Observation:" duoi day la KET QUA THAT (chay bang chinh class
# MayTinh o backend, khong phai sympy roi rac va khong phai so bia).
# =====================================================================
PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Trên tập hợp các số phức, xét phương trình $z^2-2(m+1)z+m^2=0$ ($m$ là tham số thực). Có bao nhiêu giá trị của $m$ để phương trình đó có nghiệm $z_0$ thỏa mãn $|z_0|=5$?
A. $2$ | B. $3$ | C. $1$ | D. $4$

<think>
Thought: Read off a and R. The equation is z^2-2(m+1)z+m^2=0, matching z^2-2(m+a)z+m^2=0 with a=1. The target modulus is R=5. I will now type exactly: delta_prime = expand((m+1)**2 - m**2)
Action: Calculator
Action Input: delta_prime = expand((m+1)**2 - m**2)
Observation: delta_prime = 2*m + 1

Thought: Start the running tally at 0 before working through the three cases.
Action: Calculator
Action Input: count = 0
Observation: count = 0

Thought: Case 1: delta_prime=0. Solve for m.
Action: Calculator
Action Input: roots_case1 = solve(Eq(delta_prime, 0), m)
Observation: roots_case1 = [-1/2]

Thought: Take that root.
Action: Calculator
Action Input: m1 = roots_case1[0]
Observation: m1 = -1/2

Thought: For this m the double root is z0=a+m=1+m1. I will now type exactly: z0_th1 = m1 + 1
Action: Calculator
Action Input: z0_th1 = m1 + 1
Observation: z0_th1 = 1/2

Thought: Check |z0|=R=5.
Action: Calculator
Action Input: Eq(Abs(z0_th1), 5)
Observation: False

Thought: Case 1 fails, discarded, count stays 0. Case 2: delta_prime<0. Write the root formula: z0=(a+m)+-sqrt(|delta_prime|)i. Here delta_prime<0 so |delta_prime|=-delta_prime, giving z0=(a+m)+-sqrt(-delta_prime)i. Square the modulus term by term: |z0|^2=(a+m)^2+(-delta_prime). Substitute delta_prime=(a+m)^2-m^2: |z0|^2=(a+m)^2-[(a+m)^2-m^2]=m^2. So |z0|=R iff m^2=R^2 iff m=R or m=-R, i.e. m=5 or m=-5. Check the first candidate m=5 against delta_prime<0.
Action: Calculator
Action Input: delta_prime.subs(m, 5) < 0
Observation: False

Thought: m=5 fails the Case 2 condition, discarded. Check the second candidate m=-5.
Action: Calculator
Action Input: delta_prime.subs(m, -5) < 0
Observation: True

Thought: m=-5 passed; increment the tally.
Action: Calculator
Action Input: count = count + 1
Observation: count = 1

Thought: Case 3: delta_prime>0. Substitute z0=R=5 directly into the ORIGINAL equation (not into delta_prime). I will now type exactly: eq_R = expand(5**2 - 2*(m+1)*5 + m**2)
Action: Calculator
Action Input: eq_R = expand(5**2 - 2*(m+1)*5 + m**2)
Observation: eq_R = m**2 - 10*m + 15

Thought: Solve it for m.
Action: Calculator
Action Input: roots_R = solve(Eq(eq_R, 0), m)
Observation: roots_R = [5 - sqrt(10), sqrt(10) + 5]

Thought: Take the first root and check it against delta_prime>0.
Action: Calculator
Action Input: m3a = roots_R[0]
Observation: m3a = 5 - sqrt(10)

Action: Calculator
Action Input: delta_prime.subs(m, m3a) > 0
Observation: True

Thought: Passed; increment the tally.
Action: Calculator
Action Input: count = count + 1
Observation: count = 2

Thought: Take the second root the same way.
Action: Calculator
Action Input: m3b = roots_R[1]
Observation: m3b = sqrt(10) + 5

Action: Calculator
Action Input: delta_prime.subs(m, m3b) > 0
Observation: True

Thought: Passed too; increment the tally.
Action: Calculator
Action Input: count = count + 1
Observation: count = 3

Thought: Now substitute z0=-R=-5 into the ORIGINAL equation. I will now type exactly: eq_negR = expand((-5)**2 - 2*(m+1)*(-5) + m**2)
Action: Calculator
Action Input: eq_negR = expand((-5)**2 - 2*(m+1)*(-5) + m**2)
Observation: eq_negR = m**2 + 10*m + 35

Thought: Solve it for m.
Action: Calculator
Action Input: roots_negR = solve(Eq(eq_negR, 0), m)
Observation: roots_negR = []

Thought: Empty list, this branch contributes nothing. All three cases are now processed; the running tally already holds the final count. Read option A verbatim from the problem, one option at a time: "A. $2$"; A is 2. Test it.
Action: Calculator
Action Input: Eq(count, 2)
Observation: False

Thought: Not A. Read option B verbatim: "B. $3$"; B is 3. Test it.
Action: Calculator
Action Input: Eq(count, 3)
Observation: True

Option B matches; stop here.
</think>
Final Answer: \boxed{B}
'''


# =====================================================================
# [E] PART_NHIEMVU - CHOT NHIEM VU (sua danh sach buoc khi doi dang)
# =====================================================================
PART_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Match the given equation against $z^2-2(m+a)z+m^2=0$ and read off $a$ and $R$ in your Thought, going term by term and watching the sign of $a$.
[Step 2] Compute `delta_prime = expand((m+<a>)**2 - m**2)`, then set `count = 0`.
[Step 3] Case 1 ($\Delta'=0$): compute `roots_case1 = solve(Eq(delta_prime, 0), m)`; for each root, compute `z0 = <a> + <root>` and check `Eq(Abs(z0), <R>)`; if it passes, `count = count + 1`.
[Step 4] Case 2 ($\Delta'<0$): in your Thought, write out the root formula $z_0=(a+m)\pm\sqrt{|\Delta'|}\,i$, note $|\Delta'|=-\Delta'$ since $\Delta'<0$, square the modulus term by term, then substitute $\Delta'=(a+m)^2-m^2$ to arrive at $|z_0|^2=m^2$; do not skip straight to "the candidates are $m=\pm R$" without this derivation. This gives the two candidates $m=R$ and $m=-R$; for each one, check `delta_prime.subs(m, <candidate>) < 0`; if it passes, `count = count + 1`.
[Step 5] Case 3 ($\Delta'>0$): substitute $z=R$ and separately $z=-R$ directly into the ORIGINAL equation $z^2-2(m+a)z+m^2=0$ (never into `delta_prime`) to build two new quadratics in $m$; solve each with `solve(...)` (an empty list means that branch contributes nothing); for every root found, check `delta_prime.subs(m, <root>) > 0`; if it passes, `count = count + 1`.
[Step 6] The three cases' conditions on $m$ ($\Delta'=0$, $<0$, $>0$) are mutually exclusive, so no cross-checking between cases is needed; once all three are processed, `count` already holds the exact answer.
[Step 7] Go through the options ONE AT A TIME in order. For each one, quote its exact text from the problem verbatim in your Thought before assigning it a value, then test that value with `Eq(count, <that value>)`, and stop at the first `True`. Never pick the letter by eye.

Right before each Calculator call, write a Thought that spells out the EXACT line you are about to type (for example "I will now type exactly: eq_R = ..."), then copy that line character-for-character into `Action Input` rather than retyping it from the earlier prose.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)


# BACKEND: TOOL "Calculator" - may tinh sympy CO BO NHO BIEN

In [ ]:
# BACKEND: TOOL "Calculator" - may tinh sympy CO BO NHO BIEN
# =====================================================================
# Tong quat cho MOI dang toan (khong chi D53): nhan 1 bieu thuc sympy bat
# ky, tra ve gia tri chinh xac tuyet doi. Ho tro:
#   - gan bien:  "ten = bieu_thuc"  -> luu vao bo nho, dung lai o luot sau
#     (xoa han lo hoi model chep tay lai so dai - nguon loi lon nhat)
#   - so khop :  "Eq(a, b)"         -> True/False chinh xac, khong xap xi
#   - moi phep cong tru nhan chia phan so, can thuc, so phuc, mo dun, lien hop
import sympy as sp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            parsed = sp.sympify(s, locals=self.ns)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau = sp.simplify(parsed.lhs - parsed.rhs) == 0
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri = sp.expand(sp.simplify(parsed))
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'


# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 35          # van an toan chong loop vo han (D47: toi da ~20 luot tinh 3 truong hop + 4 luot doi chieu phuong an)
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        # EP SAN token dau tien (giong ban 1 cau): model bi buoc noi tiep tu
        # "Thought:" ngay sau <think>, khong con quyen tu chon viet van xuoi
        # mo dau (hanh vi mac dinh de lech khoi dinh dang ReAct).
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        khop = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            khop = mm
        bieu_thuc = khop.group(1).strip() if khop else ''

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)

## Dang D48

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

OUT_PATH  = '/kaggle/working/d48_react_calculator_full90_DeepSeek-R1-Distill-Qwen-1.5B.csv'

MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = set()  # dang nay khong dung an so tu do nao (khong con x,y sau khi giai he)

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/11-dang-full/plan_solve_prompts_11dang_merged.json'  # SUA NEU KHAC
with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D48']
print('So cau:', len(records))

# ---- Backend Calculator: dat O DAY, TRUOC khi LLM(...)/CUDA khoi tao ----
# (an toan multiprocessing.fork - xem giai thich trong comment ben duoi)
import sympy as sp
import multiprocessing as mp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Luoi an toan CHO CA BATCH: du da dung radsimp() de tranh treo may o hau
# het truong hop, sympy van khong dam bao toc do cho MOI to hop can thuc
# bat ky - chi can 1/90 cau roi vao truong hop xau la ca batch nghen theo
# (Calculator chay tuan tu tung cau). Dung TIEN TRINH CON that (co the bi
# giet cuong buc bang tin hieu he dieu hanh) thay vi thread: thread chi
# ngat duoc tai diem GIL duoc nhuong lai, KHONG dam bao neu tinh toan ket
# sau trong 1 loi goi C lien tuc (da xac nhan qua thuc te chay tren
# Kaggle). Pool tien trinh con nay duoc tao NGAY TAI DAY, TRUOC KHI
# LLM(...)/CUDA khoi tao ben duoi - vi vay an toan tuyet doi voi
# multiprocessing.fork (fork() SAU KHI CUDA da khoi tao moi la nguy hiem,
# tung gay treo o mot lan thu truoc do).
TIMEOUT_SECONDS = 20


def fast_simplify(expr):
    """Rut gon nhanh va ON DINH hon sp.simplify() thuan tuy.

    sp.simplify() la ham "thu tat ca chien luoc roi chon ket qua ngan nhat",
    rat cham (co the treo may) khi bieu thuc co nhieu MAU SO chua can bac
    hai khac goc (vd tu phep chia (Bx-Ay)/(Bx-Ax)) - dung sinh ra qua nhieu
    dang the hien khac nhau ma khong bao gio hop nhat lai. radsimp() giai
    quyet dung goc van de nay: no huu ti hoa mau so chua can NGAY LAP TUC,
    nen ket qua o moi buoc luon o dang gon, khong de cac mau can long tich
    luy qua tung phep tinh tiep theo. Dung radsimp() lam buoc rut gon CHINH
    (nhanh, gan nhu luon du); chi roi sang simplify() lam buoc du phong khi
    radsimp() chua dua duoc ve dang 0/dang gon nhat.
    """
    return sp.expand(sp.radsimp(expr))


def is_zero(expr):
    """Kiem tra bieu thuc co bang 0 khong.

    Buoc dau (radsimp) bat duoc phan lon truong hop that nhanh va tuyet
    doi chinh xac. Neu chua ket luan duoc, KHONG roi sang sp.simplify()
    (chung minh dai so - cham, khong dam bao toc do, day la duong tung
    gay treo may). Thay vao do, so sanh gia tri SO HOC voi do chinh xac
    rat cao (50 chu so thap phan) - dung nguyen tac may tinh Casio: so
    hai so thap phan thay vi chung minh dang thuc dai so. Voi mien bai
    toan nay (cac hang so dai so co dinh tu de bai, khong phai gia tri
    adversarial), sai so gan nhu khong the xay ra o do chinh xac nay.
    """
    rut_gon = sp.radsimp(expr)
    if rut_gon == 0:
        return True
    return abs(complex(rut_gon.evalf(50))) < 1e-40


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            # 'Eq' duoc thay bang phien ban evaluate=False: sp.Eq() mac dinh
            # TU DONG thu kiem tra 2 ve co bang nhau NGAY LUC KHOI TAO (co che
            # rieng cua sympy, khac hoan toan ham is_zero() tu viet ben duoi) -
            # voi bieu thuc can long phuc tap, chinh buoc TU DONG nay co the
            # treo may, va treo TRUOC CA KHI chay_co_timeout kip can thiep (vi
            # no xay ra ngay trong luc sympify dang parse chuoi). Dung ban
            # evaluate=False de hoan toan doi viec so khop cho ham is_zero() -
            # da duoc kiem chung nhanh va dang tin cay - dam nhiem.
            ns_de_parse = dict(self.ns)
            ns_de_parse['Eq'] = lambda a, b: sp.Eq(a, b, evaluate=False)
            parsed = sp.sympify(s, locals=ns_de_parse)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau, loi_timeout = chay_co_timeout(is_zero, parsed.lhs - parsed.rhs)
                if loi_timeout:
                    return None, loi_timeout
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri, loi_timeout = chay_co_timeout(fast_simplify, parsed)
            if loi_timeout:
                return None, loi_timeout
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

try:
    _CALC_POOL = mp.get_context('fork').Pool(1)
except ValueError:
    _CALC_POOL = None  # khong co fork (vd may local Windows) -> chay khong timeout


def chay_co_timeout(ham, *args):
    """Chay ham(*args) trong tien trinh con (fork, tao TRUOC CUDA nen an
    toan), gioi han TIMEOUT_SECONDS. Neu qua han, HUY va TAO LAI pool (vi
    tien trinh con cu van con chay ngam, khong the tai su dung duoc nua)
    roi tra ve loi ro rang thay vi treo may."""
    global _CALC_POOL
    if _CALC_POOL is None:
        return ham(*args), None
    ar = _CALC_POOL.apply_async(ham, args)
    try:
        return ar.get(timeout=TIMEOUT_SECONDS), None
    except mp.TimeoutError:
        _CALC_POOL.terminate()
        _CALC_POOL = mp.get_context('fork').Pool(1)
        return None, (f'computation timed out after {TIMEOUT_SECONDS}s '
                      '(the expression is too complex to simplify exactly). '
                      'Do not resend the exact same expression; try continuing '
                      'with a different, smaller step instead.')


# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) nhieu lan trong cung kernel tung gay loi GPU het bo
# nho (da xac nhan qua thuc te chay tren Kaggle o ban 5 dang truoc).
print('Dung lai tok/llm da nap san.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D48_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai [C] PART_HUONGGIAI va
# [D] PART_FEWSHOT, giu nguyen [A] PART_TOOL va [B] PART_KIENTHUC.

# [A] PART_TOOL - HUONG DAN DUNG TOOL (DUNG CHUNG CHO MOI DANG TOAN)
# =====================================================================
PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. For most problem types this is decided by an exact symbolic proof. For this problem type specifically, it is decided by evaluating both sides to 50 significant decimal digits and checking they agree to that precision - this is not a formal proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True` - test all four, every time (see PART 4's matching step for why).
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher-degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation-solve mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non-real ones when the polynomial's real roots don't account for its full degree; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`; `Im` is not recognized and silently fails to evaluate) and discard it immediately if not. For a root confirmed real, use `re(root)`; not the raw value; in every later threshold or substitution, since even a numerically-real root can carry a residual non-zero imaginary part too small to matter but large enough to break a direct comparison. Any comparison built on such numeric roots (a threshold like `t > 0`, or a self-consistency check like `Abs(a - b) < 1e-6`) should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root; in particular the final count of solutions and matching it against the answer options; stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo - retyping a small variation of the same idea will keep producing the same error. Stop, go back to PART 2's method for this exact situation (a formula giving `zoo`/an undefined result almost always means a special case described somewhere in PART 2 applies here - e.g. a division that is only valid when some quantity is nonzero), and use the alternative formula PART 2 gives for that case, typed as a normal exact expression - never invent a placeholder word like `undefined` as if it were a value; the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


# =====================================================================


PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$ - there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change - so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).

20. **Vieta's formulas for a quadratic.** For $z^2+pz+q=0$ with roots $z_1,z_2$ (real or complex), the coefficients determine the roots' sum and product directly, without solving anything: $z_1+z_2=-p$ and $z_1z_2=q$. This holds whether the roots are real or a complex-conjugate pair (fact 15) - the SAME two equations hold either way, since they simply come from matching coefficients in the identity $z^2+pz+q=(z-z_1)(z-z_2)=z^2-(z_1+z_2)z+z_1z_2$, which does not care whether $z_1,z_2$ happen to be real.
'''


# =====================================================================
# [C] PART_HUONGGIAI - huong giai rieng cua dang  -> THAY KHI DOI DANG
# =====================================================================
PART_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE (a worked-through analysis, not a checklist - follow the reasoning itself, so you can reconstruct the right approach for any numbers)**

Problem shape: a quadratic $z^2+4az+b^2+2=0$ with REAL parameters $a,b$ (the unknowns to find), and a condition relating its two roots $z_1,z_2$ that is NOT symmetric in $z_1,z_2$ - typically something like $z_1+2iz_2=u+vi$ (the exact multiplier and the constant $u+vi$ vary by problem, but the shape is the same). Count how many pairs $(a,b)$ make this possible.

**Why \"not symmetric\" forces a case split.** If the given relation treated $z_1,z_2$ interchangeably (like $z_1+z_2=\dots$ or $z_1z_2=\dots$), Vieta's formulas (fact 20) alone would pin down $a,b$ without ever needing to know $z_1,z_2$ individually. But $z_1+2iz_2$ treats $z_1$ and $z_2$ DIFFERENTLY - one is multiplied by $2i$, the other is not - so which actual root is called $z_1$ and which is called $z_2$ genuinely matters here. This means we first need the ACTUAL VALUES of $z_1,z_2$, and to find those we need to know what KIND of roots they are. A quadratic with real coefficients only ever has two possibilities for its roots (fact 15): either both are real, or they are a complex-conjugate pair - there is no third option, and nothing in the problem tells you in advance which one applies, so both must be examined.

**Case 1: $z_1,z_2$ both real.** Since $z_1,z_2$ are real numbers here, $z_1+2iz_2=z_1+(2z_2)i$ is ALREADY sitting in real-plus-imaginary form - this is not an equation that needs solving, it is a direct reading: matching it to $u+vi$ gives $z_1=u$ and $2z_2=v$, i.e. $z_2=v/2$, immediately. With $z_1,z_2$ now known numbers, feed them into Vieta (fact 20) for $z^2+4az+(b^2+2)=0$: the sum $z_1+z_2=-4a$ gives $a=-\dfrac{z_1+z_2}{4}$, and the product $z_1z_2=b^2+2$ gives $b^2=z_1z_2-2$. This $b^2$ is just a number computed from $z_1,z_2$ - it is not automatically $\ge0$ merely because $z_1,z_2$ came out real, and $b$ itself must still be real, so its sign must be checked explicitly: $b^2>0$ gives TWO values of $b$ (namely $\pm\sqrt{b^2}$ - two different pairs $(a,b)$, since $a$ stays fixed while $b$ takes both signs); $b^2=0$ gives exactly ONE value $b=0$ (one pair); $b^2<0$ means no real $b$ exists at all, so Case 1 contributes NOTHING here - discard it, never force a value into existence.

**Case 2: $z_1,z_2$ a complex-conjugate pair.** By fact 15, this is the ONLY other possibility for a real-coefficient quadratic. It is not an alternative to Case 1 that you pick between - both cases are genuinely possible, just for DIFFERENT choices of $a,b$, and the problem asks for the total count across both, so Case 2 must be worked out regardless of what Case 1 found. Write $z_1=x+yi$ with $x,y$ real and $y\ne0$ (if $y=0$ then $z_1$ would be real, which is Case 1, not this one - so if a later computation gives $y=0$, it means this case degenerates and contributes nothing). Then $z_2=\overline{z_1}=x-yi$ (fact 15's conjugate pairing). Substitute into the given relation: $z_1+2iz_2=(x+yi)+2i(x-yi)=x+yi+2xi-2yi^2$. Since $i^2=-1$ (fact 1), the term $-2yi^2$ becomes $+2y$, so this simplifies to $(x+2y)+(2x+y)i$. Matching to $u+vi$ gives a linear system: $x+2y=u$ and $2x+y=v$. Solve it by elimination: doubling the first equation and subtracting the second eliminates $x$, giving $3y=2u-v$, so $y=\dfrac{2u-v}{3}$; substituting back into the first equation gives $x=\dfrac{2v-u}{3}$. Once $x,y$ are known numbers, Vieta again: $z_1+z_2=2x=-4a$ gives $a=-\dfrac{x}{2}$, and $z_1z_2=(x+yi)(x-yi)=x^2+y^2=b^2+2$ gives $b^2=x^2+y^2-2$. Exactly the same sign check from Case 1 applies to this $b^2$: positive gives 2 pairs, zero gives 1, negative gives 0 - and if $y$ itself turned out to be $0$, this whole case contributes nothing regardless of what $b^2$ says.

**Totalling.** The answer is the number of pairs from Case 1 PLUS the number of pairs from Case 2 - the two cases describe genuinely different scenarios (different identities for $z_1,z_2$), so simply adding them never double-counts anything; there is no overlap to check for. Match this total directly against the four options (a plain non-negative integer) with `Eq(...)`, testing all four every time as usual.
'''

# =====================================================================
# [D] PART_FEWSHOT - vi du mau rieng cua dang   -> THAY KHI DOI DANG
# Cac dong "Observation:" duoi day la KET QUA THAT (chay bang chinh class
# MayTinh o backend, khong phai sympy roi rac va khong phai so bia).
# =====================================================================
PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Trên tập hợp các số phức, xét phương trình $z^2+4az+b^2+2=0$ ($a, b$ là các tham số thực). Có bao nhiêu cặp số thực $(a;b)$ sao cho phương trình đó có hai nghiệm $z_1, z_2$ thỏa mãn $z_1+2iz_2=3+3i$?

A. $4$. | B. $1$. | C. $2$. | D. $3$.

<think>
Thought: The given relation is z1+2iz2=3+3i, so u=3, v=3. I will now type exactly: u = 3
Action: Calculator
Action Input: u = 3
Observation: u = 3

Thought: Store v too. I will now type exactly: v = 3
Action: Calculator
Action Input: v = 3
Observation: v = 3

Thought: Case 1: z1,z2 both real. Then z1+2i*z2=z1+(2z2)i is already in real+imaginary form, so matching to u+vi gives z1=u directly, no equation to solve.
Action: Calculator
Action Input: z1_1 = u
Observation: z1_1 = 3

Thought: Likewise 2*z2=v, so z2=v/2.
Action: Calculator
Action Input: z2_1 = v/2
Observation: z2_1 = 3/2

Thought: Vieta for z^2+4az+(b^2+2)=0: sum z1+z2=-4a, so a=-(z1+z2)/4.
Action: Calculator
Action Input: a1 = -(z1_1+z2_1)/4
Observation: a1 = -9/8

Thought: Product z1*z2=b^2+2, so b^2=z1*z2-2.
Action: Calculator
Action Input: b2_1 = z1_1*z2_1 - 2
Observation: b2_1 = 5/2

Thought: Check the sign of b2_1 to count how many real b it gives.
Action: Calculator
Action Input: b2_1 > 0
Observation: True

Thought: b2_1 is strictly positive, so Case 1 gives 2 pairs (b=+-sqrt(b2_1)). Now Case 2: z1,z2 a conjugate pair, z1=x+yi, z2=x-yi. Substituting into z1+2i*z2 and using i^2=-1 gives (x+2y)+(2x+y)i=u+vi, i.e. x+2y=u and 2x+y=v. Solving this by elimination gives x=(2v-u)/3.
Action: Calculator
Action Input: x1 = (2*v-u)/3
Observation: x1 = 1

Thought: And y=(2u-v)/3 from the same elimination.
Action: Calculator
Action Input: y1 = (2*u-v)/3
Observation: y1 = 1

Thought: Check y1 is not 0 (otherwise this case degenerates into Case 1 and contributes nothing).
Action: Calculator
Action Input: Eq(y1, 0)
Observation: False

Thought: y1 is nonzero, so this case is genuine. Vieta again: z1+z2=2x1=-4a, so a=-x1/2.
Action: Calculator
Action Input: a2 = -x1/2
Observation: a2 = -1/2

Thought: Product z1*z2=x1^2+y1^2=b^2+2, so b^2=x1^2+y1^2-2.
Action: Calculator
Action Input: b2_2 = x1**2 + y1**2 - 2
Observation: b2_2 = 0

Thought: Check the sign of b2_2.
Action: Calculator
Action Input: b2_2 > 0
Observation: False

Thought: Not strictly positive; check whether it is exactly 0.
Action: Calculator
Action Input: Eq(b2_2, 0)
Observation: True

Thought: b2_2 is exactly 0, so Case 2 gives exactly 1 pair (b=0). Total pairs = Case1 + Case2.
Action: Calculator
Action Input: tong = 2 + 1
Observation: tong = 3

Thought: Eq() here is a numeric check, not a symbolic proof, so I test ALL FOUR options every time, never stopping early. Read option A verbatim: "A. $4$."; A is 4. Test it.
Action: Calculator
Action Input: Eq(tong, 4)
Observation: False

Thought: Not A. Read option B verbatim: "B. $1$."; B is 1. Test it.
Action: Calculator
Action Input: Eq(tong, 1)
Observation: False

Thought: Not B. Read option C verbatim: "C. $2$."; C is 2. Test it.
Action: Calculator
Action Input: Eq(tong, 2)
Observation: False

Thought: Not C. Read option D verbatim: "D. $3$."; D is 3. Test it.
Action: Calculator
Action Input: Eq(tong, 3)
Observation: True

Thought: Exactly one option came back True: D. That is the answer.
</think>
Final Answer: \boxed{D}
'''

# =====================================================================
# [E] PART_NHIEMVU - CHOT NHIEM VU (sua danh sach buoc khi doi dang)
# =====================================================================
PART_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Read the given relation (e.g. $z_1+2iz_2=u+vi$) and store `u`, `v` with the Calculator.
[Step 2 - Case 1] Store `z1_1 = u` and `z2_1 = v/2` directly (no equation to solve - real+imaginary parts read straight off, per PART 2 Case 1).
[Step 3 - Case 1] Compute `a1 = -(z1_1+z2_1)/4` and `b2_1 = z1_1*z2_1 - 2` (Vieta).
[Step 4 - Case 1] Check `b2_1 > 0`. If `True`, Case 1 gives 2 pairs. If `False`, check `Eq(b2_1, 0)`: `True` gives 1 pair, `False` gives 0 pairs.
[Step 5 - Case 2] Compute the closed-form solution to the linear system from PART 2: `x1 = (2*v-u)/3` and `y1 = (2*u-v)/3`.
[Step 6 - Case 2] Check `Eq(y1, 0)`. If `True`, Case 2 contributes 0 pairs - skip straight to totalling. If `False`, continue.
[Step 7 - Case 2] Compute `a2 = -x1/2` and `b2_2 = x1**2 + y1**2 - 2` (Vieta).
[Step 8 - Case 2] Check `b2_2 > 0`. If `True`, Case 2 gives 2 pairs. If `False`, check `Eq(b2_2, 0)`: `True` gives 1 pair, `False` gives 0 pairs.
[Step 9] Compute `tong = <Case 1 count> + <Case 2 count>` with the Calculator (write the actual two numbers found in steps 4 and 8).
[Step 10] Go through ALL FOUR options, one at a time in order, and test EVERY one with `Eq(tong, ...)` - never stop early just because an earlier one already came back `True`; `Eq()` here is a high-precision numeric check, not a symbolic proof (as explained in the Calculator instructions above), so the discipline of checking all four every time is what keeps it trustworthy. For each option, quote its exact text from the problem verbatim in your Thought before testing it. Exactly one of the four should come back `True`; that letter is your answer. Never pick the letter by eye.

Right before each Calculator call, write a Thought that spells out the EXACT line you are about to type (for example \"I will now type exactly: b2_1 = ...\"), then copy that line character-for-character into `Action Input` rather than retyping it from the earlier prose.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)


# VONG LAP ReAct THAT (Thought -> Action -> Observation -> Thought moi)

In [ ]:
# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# (MayTinh/fast_simplify/is_zero da duoc dinh nghia o Cell 3, truoc khi
# LLM(...) khoi tao - xem giai thich an toan CUDA/fork o do)
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 30          # quy trinh day du ~15-16 luot tinh (u,v,z1_1,z2_1,a1,b2_1,check,x1,y1,check-y1,a2,b2_2,check,tong) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        # EP SAN token dau tien (giong ban 1 cau): model bi buoc noi tiep tu
        # "Thought:" ngay sau <think>, khong con quyen tu chon viet van xuoi
        # mo dau (hanh vi mac dinh de lech khoi dinh dang ReAct).
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        khop = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            khop = mm
        bieu_thuc = khop.group(1).strip() if khop else ''

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)

## Dang D50

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

OUT_PATH  = '/kaggle/working/d50_react_calculator_full90_DeepSeek-R1-Distill-Qwen-1.5B.csv'

MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = set()

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/11-dang-full/plan_solve_prompts_11dang_merged.json'  # SUA NEU KHAC
with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D50']
print('So cau:', len(records))

# ---- Backend Calculator: dat O DAY, TRUOC khi LLM(...)/CUDA khoi tao ----
# (an toan multiprocessing.fork - xem giai thich trong comment ben duoi)
import sympy as sp
import multiprocessing as mp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Luoi an toan CHO CA BATCH: du da dung radsimp() de tranh treo may o hau
# het truong hop, sympy van khong dam bao toc do cho MOI to hop can thuc
# bat ky - chi can 1/90 cau roi vao truong hop xau la ca batch nghen theo
# (Calculator chay tuan tu tung cau). Dung TIEN TRINH CON that (co the bi
# giet cuong buc bang tin hieu he dieu hanh) thay vi thread: thread chi
# ngat duoc tai diem GIL duoc nhuong lai, KHONG dam bao neu tinh toan ket
# sau trong 1 loi goi C lien tuc (da xac nhan qua thuc te chay tren
# Kaggle). Pool tien trinh con nay duoc tao NGAY TAI DAY, TRUOC KHI
# LLM(...)/CUDA khoi tao ben duoi - vi vay an toan tuyet doi voi
# multiprocessing.fork (fork() SAU KHI CUDA da khoi tao moi la nguy hiem,
# tung gay treo o mot lan thu truoc do).
TIMEOUT_SECONDS = 20


def fast_simplify(expr):
    """Rut gon nhanh va ON DINH hon sp.simplify() thuan tuy.

    sp.simplify() la ham "thu tat ca chien luoc roi chon ket qua ngan nhat",
    rat cham (co the treo may) khi bieu thuc co nhieu MAU SO chua can bac
    hai khac goc (vd tu phep chia (Bx-Ay)/(Bx-Ax)) - dung sinh ra qua nhieu
    dang the hien khac nhau ma khong bao gio hop nhat lai. radsimp() giai
    quyet dung goc van de nay: no huu ti hoa mau so chua can NGAY LAP TUC,
    nen ket qua o moi buoc luon o dang gon, khong de cac mau can long tich
    luy qua tung phep tinh tiep theo. Dung radsimp() lam buoc rut gon CHINH
    (nhanh, gan nhu luon du); chi roi sang simplify() lam buoc du phong khi
    radsimp() chua dua duoc ve dang 0/dang gon nhat.
    """
    return sp.expand(sp.radsimp(expr))


def is_zero(expr):
    """Kiem tra bieu thuc co bang 0 khong.

    Buoc dau (radsimp) bat duoc phan lon truong hop that nhanh va tuyet
    doi chinh xac. Neu chua ket luan duoc, KHONG roi sang sp.simplify()
    (chung minh dai so - cham, khong dam bao toc do, day la duong tung
    gay treo may). Thay vao do, so sanh gia tri SO HOC voi do chinh xac
    rat cao (50 chu so thap phan) - dung nguyen tac may tinh Casio: so
    hai so thap phan thay vi chung minh dang thuc dai so. Voi mien bai
    toan nay (cac hang so dai so co dinh tu de bai, khong phai gia tri
    adversarial), sai so gan nhu khong the xay ra o do chinh xac nay.
    """
    rut_gon = sp.radsimp(expr)
    if rut_gon == 0:
        return True
    return abs(complex(rut_gon.evalf(50))) < 1e-40


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            # 'Eq' duoc thay bang phien ban evaluate=False: sp.Eq() mac dinh
            # TU DONG thu kiem tra 2 ve co bang nhau NGAY LUC KHOI TAO (co che
            # rieng cua sympy, khac hoan toan ham is_zero() tu viet ben duoi) -
            # voi bieu thuc can long phuc tap, chinh buoc TU DONG nay co the
            # treo may, va treo TRUOC CA KHI chay_co_timeout kip can thiep (vi
            # no xay ra ngay trong luc sympify dang parse chuoi). Dung ban
            # evaluate=False de hoan toan doi viec so khop cho ham is_zero() -
            # da duoc kiem chung nhanh va dang tin cay - dam nhiem.
            ns_de_parse = dict(self.ns)
            ns_de_parse['Eq'] = lambda a, b: sp.Eq(a, b, evaluate=False)
            parsed = sp.sympify(s, locals=ns_de_parse)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau, loi_timeout = chay_co_timeout(is_zero, parsed.lhs - parsed.rhs)
                if loi_timeout:
                    return None, loi_timeout
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri, loi_timeout = chay_co_timeout(fast_simplify, parsed)
            if loi_timeout:
                return None, loi_timeout
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

try:
    _CALC_POOL = mp.get_context('fork').Pool(1)
except ValueError:
    _CALC_POOL = None  # khong co fork (vd may local Windows) -> chay khong timeout


def chay_co_timeout(ham, *args):
    """Chay ham(*args) trong tien trinh con (fork, tao TRUOC CUDA nen an
    toan), gioi han TIMEOUT_SECONDS. Neu qua han, HUY va TAO LAI pool (vi
    tien trinh con cu van con chay ngam, khong the tai su dung duoc nua)
    roi tra ve loi ro rang thay vi treo may."""
    global _CALC_POOL
    if _CALC_POOL is None:
        return ham(*args), None
    ar = _CALC_POOL.apply_async(ham, args)
    try:
        return ar.get(timeout=TIMEOUT_SECONDS), None
    except mp.TimeoutError:
        _CALC_POOL.terminate()
        _CALC_POOL = mp.get_context('fork').Pool(1)
        return None, (f'computation timed out after {TIMEOUT_SECONDS}s '
                      '(the expression is too complex to simplify exactly). '
                      'Do not resend the exact same expression; try continuing '
                      'with a different, smaller step instead.')


# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) nhieu lan trong cung kernel tung gay loi GPU het bo
# nho (da xac nhan qua thuc te chay tren Kaggle o ban 5 dang truoc).
print('Dung lai tok/llm da nap san.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D50_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai [C] PART_HUONGGIAI va
# [D] PART_FEWSHOT, giu nguyen [A] PART_TOOL va [B] PART_KIENTHUC.

# [A] PART_TOOL - HUONG DAN DUNG TOOL (DUNG CHUNG CHO MOI DANG TOAN)
# =====================================================================
PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. For most problem types this is decided by an exact symbolic proof. For this problem type specifically, it is decided by evaluating both sides to 50 significant decimal digits and checking they agree to that precision - this is not a formal proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True` - test all four, every time (see PART 4's matching step for why).
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher-degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation-solve mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non-real ones when the polynomial's real roots don't account for its full degree; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`; `Im` is not recognized and silently fails to evaluate) and discard it immediately if not. For a root confirmed real, use `re(root)`; not the raw value; in every later threshold or substitution, since even a numerically-real root can carry a residual non-zero imaginary part too small to matter but large enough to break a direct comparison. Any comparison built on such numeric roots (a threshold like `t > 0`, or a self-consistency check like `Abs(a - b) < 1e-6`) should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root; in particular the final count of solutions and matching it against the answer options; stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo - retyping a small variation of the same idea will keep producing the same error. Stop, go back to PART 2's method for this exact situation (a formula giving `zoo`/an undefined result almost always means a special case described somewhere in PART 2 applies here - e.g. a division that is only valid when some quantity is nonzero), and use the alternative formula PART 2 gives for that case, typed as a normal exact expression - never invent a placeholder word like `undefined` as if it were a value; the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


# =====================================================================
# [B] PART_KIENTHUC - KIEN THUC NEN SO PHUC (DUNG CHUNG MOI DANG SO PHUC)
# =====================================================================
PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$ - there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change - so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).
'''


# =====================================================================
# [C] PART_HUONGGIAI - HUONG GIAI RIENG CUA DANG D50 (THAY KHI DOI DANG)
# =====================================================================
PART_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE (this is a worked-through analysis, not a checklist - follow the reasoning itself, so you can reconstruct the right approach for any numbers, including cases that look nothing like the worked example in PART 3)**

Problem shape: $|z-z_1|+|z-z_2|=K$ (given, $K$ a positive real constant). Find the min $m$ and max $M$ of $|z-z_3|$, then compute $P=m+M$ (or whatever combination the problem asks).

**Turning the algebra into geometry.** Let $M,A,B,C$ be the points representing $z,z_1,z_2,z_3$. By fact 17 (PART 1), $|z-z_1|=MA$ and $|z-z_2|=MB$, so the given equation is exactly $MA+MB=K$: as $z$ ranges over every complex number satisfying it, $M$ ranges over every point whose distances to two FIXED points $A,B$ sum to the FIXED constant $K$. Read $A,B,C$ off the problem term by term: each modulus term $|z-w|$ needs $w$ isolated from the part written after $z$, by negating that part coefficient by coefficient (real part and imaginary part separately); the corresponding point is $(\mathrm{Re}(w),\mathrm{Im}(w))$.

**Why $M$ turns out to be confined to the segment $AB$, and nowhere else.** Compute $AB=\sqrt{(x_B-x_A)^2+(y_B-y_A)^2}$ (fact 17) and check whether it equals the given $K$. This single check is the crux of the whole problem, and it is worth seeing exactly why: fact 18 says $MA+MB\ge AB$ for ANY point $M$ in the plane, with equality precisely when $M$ lies on the segment between $A$ and $B$. So if $AB=K$, the condition $MA+MB=K$ is EXACTLY the equality case of fact 18 - it is not merely \"some distance condition\", it pins $M$ down to the finite segment $AB$, and every point of that segment satisfies it. This is what turns an optimization problem over the entire plane into an optimization problem over a 1-dimensional segment - without it, none of what follows would be tractable.

**Trying to describe where $M$ is on the segment - and discovering there are two cases, not one.** To search over the segment, we need a way to name \"the point at such-and-such position along $AB$\". The natural attempt is fact 19: write the line through $A,B$ as $y=a(x-x_A)+y_A$ with slope $a=\dfrac{y_B-y_A}{x_B-x_A}$, and let $x$ range over $[\min(x_A,x_B),\max(x_A,x_B)]$. But fact 19 also says this attempt does not always work: if $x_A=x_B$, there is no such slope (the Calculator will report `zoo` when you try), precisely because the segment $AB$ is VERTICAL - $x$ equals $x_A$ at every point of it, and it is $y$ that ranges over $[\min(y_A,y_B),\max(y_A,y_B)]$ instead. So the very first thing to do is compute $a=\dfrac{y_B-y_A}{x_B-x_A}$ and look at what comes back, BEFORE assuming which case applies:
  - an ordinary number $\Rightarrow$ the **main case** below;
  - `zoo` $\Rightarrow$ the **vertical case** below. Do not retry the same formula, and do not invent a placeholder value such as \"undefined\" - the Calculator has no such value, only exact numbers. Skip straight to the vertical case, which reuses every argument from the main case with $x$ and $y$ swapped (fact 19's own symmetry).

**Main case ($x_A\ne x_B$) - building $MC^2$ as a function of position on the segment.** With $y=a(x-x_A)+y_A$ established, and writing $k=y_A-y_C$ to shorten the algebra, substituting into $MC^2=(x-x_C)^2+(y-y_C)^2$ gives $f(x)=(x-x_C)^2+[a(x-x_A)+k]^2$. Two deliberate choices here both serve the same goal - keeping the algebra from exploding once $x_A,y_A,\dots$ contain several different square roots:
  - never compute the $y$-intercept $b=y_A-a\,x_A$ as its own stored quantity - point-slope form already fully describes the same line, and forming $b$ separately would mean carrying a whole extra fraction through everything that follows, for no benefit;
  - never expand $f(x)$ into a polynomial $Ax^2+Bx+C$ - every term of that expansion would carry cross-products of whatever radicals are inside $a,x_A,x_C,k$, which is exactly what makes the algebra explode in hard problems. The squared form above is mathematically identical and stays short - leave it exactly as is.

**Why $f$ has exactly one lowest point, and how to find it without `solve()`.** $f(x)$ is a sum of two squared real quantities, each linear in $x$; expanded, its $x^2$ coefficient would be $1+a^2$, always positive. So $f$ is an upward parabola: it has one lowest point (its vertex) and rises without bound moving away from it in either direction - nowhere else can its slope be zero. Differentiate with the chain rule, $f'(x)=2(x-x_C)+2a[a(x-x_A)+k]$, set it to $0$, and solve this LINEAR equation for $x$ by hand to get the closed form once and for all: $x_0=\dfrac{x_C+a^2x_A-ak}{a^2+1}$. Since this formula already IS the solved result, compute it with ONE Calculator call - calling `diff()`/`solve()` would mean solving something already solved.

**Why $x_0$ only sometimes matters.** $x_0$ minimizes $f$ over the WHOLE real line, but $M$ can only be at points of the segment, $x\in[\min(x_A,x_B),\max(x_A,x_B)]$, a strict piece of that line in general. Two genuinely different situations follow from where $x_0$ lands:
  - if $x_0$ is inside that interval, $M$ can actually reach the true lowest point of $f$, so $f(x_0)$ is a real candidate for the minimum; and since $f$ keeps rising moving away from $x_0$ in either direction, whichever endpoint is FARTHER from $x_0$ gives the largest value on the interval, so both endpoints still matter too;
  - if $x_0$ is outside that interval, the ENTIRE interval lies on one side of the vertex, where $f$ is purely increasing or purely decreasing across the whole thing (the only place its slope changes sign is at $x_0$, which is not here). A monotonic function on a closed interval attains both its minimum and maximum at the two endpoints, full stop - $x_0$ is irrelevant and must be discarded, not used for either $m$ or $M$.
This is a single yes/no question, not two separate facts to recall and combine afterwards in your head - trying to remember two `True`/`False` results and reason about them in words is exactly where mistakes creep in. There is a one-shot arithmetic test instead: for $t$ between bounds $\mathrm{lo},\mathrm{hi}$, both $(t-\mathrm{lo})$ and $(\mathrm{hi}-t)$ are non-negative, so their PRODUCT is non-negative; outside either bound, exactly one factor is negative, making the product negative. Compute `inside_segment = (x0 - Min(Ax,Bx)) * (Max(Ax,Bx) - x0) >= 0` as ONE Calculator call and just read the answer - never re-derive or restate it from memory afterwards. Being outside is actually the MORE common situation across this problem family (roughly half of all cases) - never default to assuming $x_0$ is inside.

**The candidates (main case).** The two endpoints are always candidates and are always cheap: at $x=x_A$, $M$ coincides with $A$ itself, so $f(x_A)$ is simply $AC^2=(x_A-x_C)^2+(y_A-y_C)^2$ (plain distance, fact 17) - never substitute $x_A$ into the line/$f(x)$ formulas, that drags in $a,k$ for nothing. Likewise $f(x_B)=BC^2$. If `inside_segment` was `True`, also compute $f(x_0)=(x_0-x_C)^2+[a(x_0-x_A)+k]^2$ using the still-unexpanded form; if `False`, do not compute this at all - it does not correspond to anywhere $M$ can actually be. Take the min and max among the surviving candidates; since each is a squared distance it is automatically $\ge0$, so the smallest is $m^2$ and the largest is $M^2$; square-root them for $m,M$.

**Vertical case ($x_A=x_B$) - the exact same reasoning, with $x$ and $y$ swapped.** Here $x$ is fixed at $x_A$ everywhere on the segment, and it is $y$ that ranges over $[\min(y_A,y_B),\max(y_A,y_B)]$. Nothing new needs to be invented; every argument above carries over verbatim under this swap. $MC^2$ as a function of $y$ is $g(y)=(x_A-x_C)^2+(y-y_C)^2$, itself an upward parabola, this time in $y$, whose vertex is at $y=y_C$ exactly (there is no analog of $a,k$ to carry, since $x$ is simply constant here - do not try to compute them in this branch). \"Is the vertex inside the interval\" becomes the same product test, with $y$'s instead of $x$'s: `inside_segment = (Cy - Min(Ay,By)) * (Max(Ay,By) - Cy) >= 0`. The two endpoints are $AC^2,BC^2$ exactly as before (fact 17 never depended on which coordinate did the work). If `inside_segment` is `True`, the third candidate is $g(y_C)=(x_A-x_C)^2+(y_C-y_C)^2=(x_A-x_C)^2$ - simply `f_x0 = (Ax-Cx)**2`, since the $(y-y_C)^2$ term vanishes exactly at $y=y_C$.

**Compute and match.** Compute whatever combination of $m,M$ the problem asks for (e.g. $P=m+M$). As explained in the Calculator instructions above, `Eq()` here is a high-precision numeric check, not a symbolic proof - so test ALL FOUR options every time, one at a time in order, never stopping early even once one comes back `True`; exactly one of the four should return `True`, and that letter is the answer.
'''

# [D] PART_FEWSHOT - VI DU MAU RIENG CUA DANG D50 (THAY KHI DOI DANG)
# Cac dong "Observation:" duoi day la KET QUA THAT (chay bang chinh class
# MayTinh o backend, khong phai sympy roi rac va khong phai so bia).
# =====================================================================
PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Xét các số phức $z$ thỏa mãn $|z+2-i|+|z-4-7i| = 6\sqrt{2}$. Gọi $m, M$ lần lượt là giá trị nhỏ nhất, giá trị lớn nhất của $|z-1+i|$. Tính $P = m+M$.
A. $P=\sqrt{13}+\sqrt{73}$ | B. $P=\dfrac{5\sqrt{2}+2\sqrt{73}}{2}$ | C. $P=5\sqrt{2}+\sqrt{73}$ | D. $P=\dfrac{5\sqrt{2}+\sqrt{73}}{2}$

<think>
Thought: The first term is |z+2-i|. The part right after z is +2-i, so its real part is +2 and its imaginary part is -1. Since the term equals z-w, w must be the negative of that part: w=-(2-i). Distributing the minus sign carefully: 2 becomes -2, and -i becomes +i. So w=-2+i: its real part is -2 and its imaginary part is 1, and those become A's x and y exactly as they are, with no further sign change. So A=(-2,1). I will now type exactly: Ax = -2
Action: Calculator
Action Input: Ax = -2
Observation: Ax = -2

Thought: Store the y-coordinate of A too; never skip it. I will now type exactly: Ay = 1
Action: Calculator
Action Input: Ay = 1
Observation: Ay = 1

Thought: The second term is |z-4-7i|. The part right after z is -4-7i, so its real part is -4 and its imaginary part is -7. Since the term equals z-w, w must be the negative of that part: w=-(-4-7i). Distributing the minus sign carefully: -4 becomes +4, and -7i becomes +7i. So w=4+7i: its real part is 4 and its imaginary part is 7, and those become B's x and y exactly as they are, with no further sign change. So B=(4,7). I will now type exactly: Bx = 4
Action: Calculator
Action Input: Bx = 4
Observation: Bx = 4

Thought: Store the y-coordinate of B too; never skip it. I will now type exactly: By = 7
Action: Calculator
Action Input: By = 7
Observation: By = 7

Thought: The target term is |z-1+i|. The part right after z is -1+i, so its real part is -1 and its imaginary part is +1. Since the term equals z-w, w must be the negative of that part: w=-(-1+i). Distributing the minus sign carefully: -1 becomes +1, and +i becomes -i. So w=1-i: its real part is 1 and its imaginary part is -1, and those become C's x and y exactly as they are, with no further sign change. So C=(1,-1). I will now type exactly: Cx = 1
Action: Calculator
Action Input: Cx = 1
Observation: Cx = 1

Thought: Store the y-coordinate of C too; never skip it. I will now type exactly: Cy = -1
Action: Calculator
Action Input: Cy = -1
Observation: Cy = -1

Thought: Store K too. I will now type exactly: K = 6*sqrt(2)
Action: Calculator
Action Input: K = 6*sqrt(2)
Observation: K = 6*sqrt(2)

Thought: Check AB=K.
Action: Calculator
Action Input: AB = sqrt((Bx-Ax)**2 + (By-Ay)**2)
Observation: AB = 6*sqrt(2)

Action: Calculator
Action Input: Eq(AB, K)
Observation: True

Thought: Confirmed, M lies on segment AB. Find only the slope a - I never compute the y-intercept b, since that is where huge fractions come from in hard problems. The line is y=a(x-Ax)+Ay.
Action: Calculator
Action Input: a = (By-Ay)/(Bx-Ax)
Observation: a = 1

Thought: That is the exact value of a, straight from the Calculator, so it is correct; I never re-derive or double-check it by hand, whatever it looks like.
Action: Calculator
Action Input: k = Ay - Cy
Observation: k = 2

Thought: f(x)=(x-Cx)^2+[a(x-Ax)+k]^2 stays in this squared form; I never expand it into a polynomial, since that is exactly what makes hard problems explode. Its derivative set to zero solves directly to the closed form x0=(Cx+a^2*Ax-a*k)/(a^2+1); I use this formula straight away instead of calling diff()/solve().
Action: Calculator
Action Input: x0 = (Cx + a**2*Ax - a*k)/(a**2+1)
Observation: x0 = -3/2

Thought: To check whether x0 lies inside the segment [min(Ax,Bx), max(Ax,Bx)] = [-2,4] in one shot, I use the product-of-differences trick: (x0-min)*(max-x0) is non-negative exactly when x0 is between them.
Action: Calculator
Action Input: inside_segment = (x0 - Min(Ax, Bx)) * (Max(Ax, Bx) - x0) >= 0
Observation: inside_segment = True

Thought: inside_segment is True, so x0 lies inside the segment and is a valid third candidate. Now the two endpoints: since M=A at the left endpoint, f there is simply AC^2, computed directly from coordinates - no need to substitute into f(x).
Action: Calculator
Action Input: AC2 = (Ax-Cx)**2 + (Ay-Cy)**2
Observation: AC2 = 13

Thought: Likewise the right endpoint is simply BC^2.
Action: Calculator
Action Input: BC2 = (Bx-Cx)**2 + (By-Cy)**2
Observation: BC2 = 73

Thought: inside_segment was True, so x0 is a valid candidate, so I evaluate f there too, using the unexpanded squared form from before.
Action: Calculator
Action Input: f_x0 = (x0-Cx)**2 + (a*(x0-Ax)+k)**2
Observation: f_x0 = 25/2

Thought: Take the min and max among all three surviving candidates.
Action: Calculator
Action Input: f_min = Min(AC2, BC2, f_x0)
Observation: f_min = 25/2

Action: Calculator
Action Input: f_max = Max(AC2, BC2, f_x0)
Observation: f_max = 73

Thought: m=sqrt(f_min), M=sqrt(f_max).
Action: Calculator
Action Input: m = sqrt(f_min)
Observation: m = 5*sqrt(2)/2

Action: Calculator
Action Input: M = sqrt(f_max)
Observation: M = sqrt(73)

Thought: Compute P=m+M.
Action: Calculator
Action Input: P = m + M
Observation: P = 5*sqrt(2)/2 + sqrt(73)

Thought: Eq() here is a numeric check, not a symbolic proof, so I test ALL FOUR options every time, never stopping early even once one comes back True. Read option A verbatim from the problem: "A. $P = \sqrt{13}+\sqrt{73}$"; A is sqrt(13)+sqrt(73). Test it.
Action: Calculator
Action Input: Eq(P, sqrt(13)+sqrt(73))
Observation: False

Thought: Not A. Read option B verbatim: "B. $P = \dfrac{5\sqrt{2}+2\sqrt{73}}{2}$"; B is (5*sqrt(2)+2*sqrt(73))/2. Test it.
Action: Calculator
Action Input: Eq(P, (5*sqrt(2)+2*sqrt(73))/2)
Observation: True

Thought: B came back True, but I still test C and D too, exactly as if none had matched yet. Read option C verbatim: "C. $P = 5\sqrt{2}+\sqrt{73}$"; C is 5*sqrt(2)+sqrt(73). Test it.
Action: Calculator
Action Input: Eq(P, 5*sqrt(2)+sqrt(73))
Observation: False

Thought: Not C. Read option D verbatim: "D. $P = \dfrac{5\sqrt{2}+\sqrt{73}}{2}$"; D is (5*sqrt(2)+sqrt(73))/2. Test it.
Action: Calculator
Action Input: Eq(P, (5*sqrt(2)+sqrt(73))/2)
Observation: False

Thought: Exactly one option came back True: B. That is the answer.
</think>
Final Answer: \boxed{B}
'''

# [E] PART_NHIEMVU - CHOT NHIEM VU (sua danh sach buoc khi doi dang)
# =====================================================================
PART_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] In your Thought, read off $A,B,C$ term by term (per PART 2). Store all SIX coordinates `Ax,Ay,Bx,By,Cx,Cy` and `K = <the given constant>` with the Calculator, one at a time; the $y$-coordinate of each point is just as easy to forget to store as the $x$-coordinate, so give it its own Thought and Calculator call too, never skip straight past it.
[Step 2] Compute `AB = sqrt((Bx-Ax)**2 + (By-Ay)**2)` and check `Eq(AB, K)`.
[Step 3] Compute `a = (By-Ay)/(Bx-Ax)`. This single result tells you which of the two cases from PART 2 you are in: an ordinary number means the MAIN CASE (steps 4-6 below); `zoo` means the VERTICAL CASE (steps 4v-6v below) - in that case do not retry this formula and do not invent a placeholder value, go directly to step 4v.

**MAIN CASE ($a$ was an ordinary number):**
[Step 4] Compute `k = Ay - Cy`. Never compute a $y$-intercept $b$.
[Step 5] Compute the critical point directly in closed form: `x0 = (Cx + a**2*Ax - a*k)/(a**2+1)`. Do NOT call `diff()` or `solve()` - the formula already IS the solved result.
[Step 6] Compute `inside_segment = (x0 - Min(Ax, Bx)) * (Max(Ax, Bx) - x0) >= 0` with ONE Calculator call (the product trick from PART 2) - never check the two bounds separately and combine them yourself. It is often `False` (do not assume $x_0$ is always inside the segment); only if `True` does `x0` become a candidate, via `f_x0 = (x0-Cx)**2 + (a*(x0-Ax)+k)**2` (compute this only when `inside_segment` is `True`).

**VERTICAL CASE (step 3 gave `zoo`) - skip steps 4-6 above entirely, do this instead:**
[Step 4v] There is no $k$ or $x_0$ in this case - do not try to compute them.
[Step 5v] Compute `inside_segment = (Cy - Min(Ay, By)) * (Max(Ay, By) - Cy) >= 0` - the same product trick as step 6, with $y$-coordinates instead of $x$-coordinates (PART 2's vertical case).
[Step 6v] Only if `inside_segment` is `True`, the third candidate is `f_x0 = (Ax-Cx)**2`.

**Both cases continue here:**
[Step 7] Compute the two endpoints DIRECTLY as squared distances, never by substituting into a line/quadratic: `AC2 = (Ax-Cx)**2 + (Ay-Cy)**2` and `BC2 = (Bx-Cx)**2 + (By-Cy)**2`.
[Step 8] Compute `f_min = Min(...)` and `f_max = Max(...)` over exactly the surviving candidates (`AC2`, `BC2`, and `f_x0` only if it was computed). Then `m = sqrt(f_min)` and `M = sqrt(f_max)`.
[Step 9] Compute whatever the problem asks (e.g. `P = m + M`). Go through ALL FOUR options, one at a time in order, and test EVERY one with `Eq(...)` - never stop early just because an earlier one already came back `True`; `Eq()` here is a high-precision numeric check, not a symbolic proof (as explained in the Calculator instructions above), so the discipline of checking all four every time is what keeps it trustworthy. For each option, quote its exact text from the problem verbatim in your Thought before assigning it a value. Exactly one of the four should come back `True`; that letter is your answer. Never pick the letter by eye.

Right before each Calculator call, write a Thought that spells out the EXACT line you are about to type (for example \"I will now type exactly: f_x0 = ...\"), then copy that line character-for-character into `Action Input` rather than retyping it from the earlier prose.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

In [ ]:
# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# (MayTinh/fast_simplify/is_zero da duoc dinh nghia o Cell 3, truoc khi
# LLM(...) khoi tao - xem giai thich an toan CUDA/fork o do)
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 50          # quy trinh day du ~18-19 luot tinh + toi da 4 luot doi chieu phuong an; nang tu 35 vi cau STT1069 (truong hop duong thang dung, hiem gap, chua co vi du mau) dung het 35 luot voi 25/35 la loi cu phap - can du cho nhieu lan thu-sai hon
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        # EP SAN token dau tien (giong ban 1 cau): model bi buoc noi tiep tu
        # "Thought:" ngay sau <think>, khong con quyen tu chon viet van xuoi
        # mo dau (hanh vi mac dinh de lech khoi dinh dang ReAct).
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        khop = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            khop = mm
        bieu_thuc = khop.group(1).strip() if khop else ''

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)

## Dang D52

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

OUT_PATH  = '/kaggle/working/d52_react_calculator_full90_DeepSeek-R1-Distill-Qwen-1.5B.csv'

MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = {'t'}

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/11-dang-full/plan_solve_prompts_11dang_merged.json'  # SUA NEU KHAC
with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D52']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) nhieu lan trong cung kernel tung gay loi GPU het bo
# nho (da xac nhan qua thuc te chay tren Kaggle o ban 5 dang truoc).
print('Dung lai tok/llm da nap san.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D52_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai [C] PART_HUONGGIAI va
# [D] PART_FEWSHOT, giu nguyen [A] PART_TOOL va [B] PART_KIENTHUC.

# [A] PART_TOOL - HUONG DAN DUNG TOOL (DUNG CHUNG CHO MOI DANG TOAN)
# =====================================================================
PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- To test whether two expressions are **exactly** equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`, decided exactly; never approximately.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher-degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation-solve mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non-real ones when the polynomial's real roots don't account for its full degree; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`; `Im` is not recognized and silently fails to evaluate) and discard it immediately if not. For a root confirmed real, use `re(root)`; not the raw value; in every later threshold or substitution, since even a numerically-real root can carry a residual non-zero imaginary part too small to matter but large enough to break a direct comparison. Any comparison built on such numeric roots (a threshold like `t > 0`, or a self-consistency check like `Abs(a - b) < 1e-6`) should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root; in particular the final count of solutions and matching it against the answer options; stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


# =====================================================================
# [B] PART_KIENTHUC - KIEN THUC NEN SO PHUC (DUNG CHUNG MOI DANG SO PHUC)
# =====================================================================
PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
'''


# =====================================================================
# [C] PART_HUONGGIAI - HUONG GIAI RIENG CUA DANG D52 (THAY KHI DOI DANG)
# =====================================================================
PART_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: an equation $|z^2-A|=k|z|$, where $A$ is a given complex number and $k$ a given positive real number. $M$ and $m$ are the largest and smallest values of $|z|$; the problem asks for $M^2+m^2$.

Follow this method exactly; do not invent a shorter route, and do not skip any algebraic step below by jumping straight to a final form.

1. **Read off $A$ and $k$.** Match the given equation against the standard form $|z^2-A|=k|z|$. Whatever is SUBTRACTED from $z^2$ inside the modulus is $A$, so read the signs carefully: $|z^2-3-4i|$ means $A=3+4i$, whereas $|z^2+9+7i|$ means $A=-9-7i$ (because $z^2+9+7i=z^2-(-9-7i)$). Write down $p=\mathrm{Re}(A)$ and $q=\mathrm{Im}(A)$, and $k$ is the real number multiplying $|z|$ on the right.
2. **Find $w=c+di$ such that $w^2=A$.** Use the square-root-of-a-complex-number fact from PART 1: $c^2=\dfrac{|A|+p}{2}$, $d^2=\dfrac{|A|-p}{2}$, take $c=\sqrt{c^2}\ge0$ and give $d$ the same sign as $q$. Assemble $w=c+di$ and confirm with the Calculator that $w^2$ really equals $A$ before going on.
3. **Rewrite the equation with $w$.** Since $A=w^2$, the equation $|z^2-A|=k|z|$ becomes $|z^2-w^2|=k|z|$.
4. **Factor the left side.** $z^2-w^2=(z-w)(z+w)$, so $|z^2-w^2|=|z-w|\cdot|z+w|$ by the modulus-of-a-product fact from PART 1. The equation is now $|z-w|\cdot|z+w|=k|z|$.
5. **Square both sides.** $|z-w|^2\cdot|z+w|^2=k^2|z|^2$.
6. **Introduce $x,y,t$.** Write $z=x+yi$, so $z-w=(x-c)+(y-d)i$ and $z+w=(x+c)+(y+d)i$; let $t=|z|^2=x^2+y^2$ (with $t\ge0$).
7. **Expand each modulus separately, term by term.** $|z-w|^2=(x-c)^2+(y-d)^2=x^2+y^2+c^2+d^2-2cx-2dy=t+|w|^2-2(cx+dy)$. Likewise $|z+w|^2=(x+c)^2+(y+d)^2=x^2+y^2+c^2+d^2+2cx+2dy=t+|w|^2+2(cx+dy)$, where $|w|^2=c^2+d^2$.
8. **Multiply the two expansions using the difference-of-two-squares identity.** $|z-w|^2\cdot|z+w|^2=\bigl(t+|w|^2-2(cx+dy)\bigr)\bigl(t+|w|^2+2(cx+dy)\bigr)=(t+|w|^2)^2-4(cx+dy)^2$.
9. **Combine with step 5.** $(t+|w|^2)^2-4(cx+dy)^2=k^2t$, so $4(cx+dy)^2=(t+|w|^2)^2-k^2t$.
10. **Apply the B.C.S inequality from PART 1** with $(a,b)=(c,d)$ and $(x,y)=(x,y)$: $(cx+dy)^2\le(c^2+d^2)(x^2+y^2)=|w|^2\cdot t$. Multiply by 4: $4(cx+dy)^2\le4|w|^2t$.
11. **Substitute step 9 into step 10.** $(t+|w|^2)^2-k^2t\le4|w|^2t$.
12. **Expand and rearrange into a quadratic inequality in $t$.** $t^2+2|w|^2t+|w|^4-k^2t\le4|w|^2t \Rightarrow t^2-(2|w|^2+k^2)t+|w|^4\le0$. Note that $|w|^2=c^2+d^2=|A|$, exactly the same number already computed as `mod_A` when finding $w$; so $|w|^2$ is `mod_A` itself, never a new computation and never `mod_A` squared. Build this quadratic directly from `mod_A`, without introducing any new variable for $|w|^2$.
13. **Solve the boundary equation.** The quadratic on the left is an upward-opening parabola in $t$, so the inequality $\le0$ holds exactly between its two roots; that means $t$ is trapped between them, and those two roots are precisely $m^2$ and $M^2$ (the smallest and largest values $t=|z|^2$ can take, reached exactly at the B.C.S equality case). Solve $t^2-(2|w|^2+k^2)t+|w|^4=0$ to get those two roots.
14. **Add and match.** $M^2+m^2$ is the sum of those two roots; compute that sum with the Calculator. Then test the four options in order with `Eq(...)` and stop at the first `True`; never pick the letter by eye.
'''


# =====================================================================
# [D] PART_FEWSHOT - VI DU MAU RIENG CUA DANG D52 (THAY KHI DOI DANG)
# Cac dong "Observation:" duoi day la KET QUA THAT (chay bang chinh class
# MayTinh o backend, khong phai sympy roi rac va khong phai so bia).
# =====================================================================
PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Xét các số phức $z$ thỏa mãn $|z^2-3-4i| = 2|z|$. Gọi $M$ và $m$ lần lượt là giá trị lớn nhất và giá trị nhỏ nhất của $|z|$. Giá trị của $M^2+m^2$ bằng
A. $28$ | B. $18+4\sqrt{6}$ | C. $14$ | D. $11+4\sqrt{6}$

<think>
Thought: Read off A and k from the equation, watching the signs one term at a time. The equation is |z^2-3-4i| = 2|z|. Inside the modulus, what is subtracted from z^2 is 3+4i, because z^2-3-4i = z^2-(3+4i). So A = 3+4i, which gives p = Re(A) = 3 and q = Im(A) = 4; and the real number multiplying |z| on the right is k = 2. The first thing the method needs is |A|, since both c^2 and d^2 are built from it. I will now type exactly: mod_A = Abs(3 + 4*I)
Action: Calculator
Action Input: mod_A = Abs(3 + 4*I)
Observation: mod_A = 5

Thought: Now c^2 = (|A|+p)/2, and p = 3 here. I will now type exactly: c_sq = (mod_A + 3)/2
Action: Calculator
Action Input: c_sq = (mod_A + 3)/2
Observation: c_sq = 4

Thought: And d^2 = (|A|-p)/2, again with p = 3. I will now type exactly: d_sq = (mod_A - 3)/2
Action: Calculator
Action Input: d_sq = (mod_A - 3)/2
Observation: d_sq = 1

Thought: Take c as the positive square root of c_sq. I will now type exactly: c = sqrt(c_sq)
Action: Calculator
Action Input: c = sqrt(c_sq)
Observation: c = 2

Thought: Here q = 4 is positive, so d must be positive as well for 2cd = q to hold; that means the plain positive square root, with no minus sign. I will now type exactly: d = sqrt(d_sq)
Action: Calculator
Action Input: d = sqrt(d_sq)
Observation: d = 1

Thought: Assemble w from c and d. I will now type exactly: w = c + d*I
Action: Calculator
Action Input: w = c + d*I
Observation: w = 2 + I

Thought: Before going any further, confirm that w^2 really equals A = 3+4i; this check is also what would catch a wrong sign choice for d. I will now type exactly: Eq(expand(w**2), 3 + 4*I)
Action: Calculator
Action Input: Eq(expand(w**2), 3 + 4*I)
Observation: True

Thought: Confirmed, w is right. Now walk through the derivation before computing anything else, one step at a time, without skipping. Since A=w^2, the equation |z^2-A|=2|z| becomes |z^2-w^2|=2|z|. Factor the left side: z^2-w^2=(z-w)(z+w), and |u.v|=|u|.|v|, so this is |z-w|.|z+w|=2|z|. Square both sides: |z-w|^2.|z+w|^2=4|z|^2. Write z=x+yi and t=|z|^2=x^2+y^2, so z-w=(x-2)+(y-1)i and z+w=(x+2)+(y+1)i. Expand each modulus separately: |z-w|^2=(x-2)^2+(y-1)^2=x^2+y^2+4+1-4x-2y=t+5-2(2x+y), and |z+w|^2=(x+2)^2+(y+1)^2=t+5+2(2x+y), using |w|^2=c^2+d^2=5. Multiply these two using the difference-of-two-squares identity: (t+5-2(2x+y))(t+5+2(2x+y))=(t+5)^2-4(2x+y)^2. Combine with the squared equation: (t+5)^2-4(2x+y)^2=4t, so 4(2x+y)^2=(t+5)^2-4t. Apply B.C.S with (a,b)=(c,d)=(2,1) and (x,y): (2x+y)^2<=(2^2+1^2)(x^2+y^2)=5t, so 4(2x+y)^2<=20t. Substitute: (t+5)^2-4t<=20t. Expand and rearrange: t^2+10t+25-4t<=20t, giving t^2-14t+25<=0. This matches the quadratic the method predicts, t^2-(2|w|^2+k^2)t+|w|^4<=0. Note that |w|^2=c^2+d^2=|A|, which is exactly mod_A already computed above (2^2+1^2=5=mod_A); it is NOT mod_A squared, and no new Calculator call is needed to get it, since it is already sitting in mod_A.
Thought: Store k = 2 so the quadratic can be written exactly as the method states it. I will now type exactly: k = 2
Action: Calculator
Action Input: k = 2
Observation: k = 2

Thought: Now build that same quadratic through the Calculator instead of trusting the hand derivation above, using mod_A directly for |w|^2 (never mod_A squared). I will now type exactly: poly = expand(t**2 - (2*mod_A + k**2)*t + mod_A**2)
Action: Calculator
Action Input: poly = expand(t**2 - (2*mod_A + k**2)*t + mod_A**2)
Observation: poly = t**2 - 14*t + 25

Thought: Matches the hand derivation. Solve it; its two roots are m^2 and M^2.
Action: Calculator
Action Input: roots_t = solve(poly, t)
Observation: roots_t = [7 - 2*sqrt(6), 2*sqrt(6) + 7]

Thought: Take the first root.
Action: Calculator
Action Input: t1 = roots_t[0]
Observation: t1 = 7 - 2*sqrt(6)

Thought: Take the second root.
Action: Calculator
Action Input: t2 = roots_t[1]
Observation: t2 = 2*sqrt(6) + 7

Thought: These two roots are m^2 and M^2, so adding them gives exactly the M^2+m^2 the problem asks for.
Action: Calculator
Action Input: total = t1 + t2
Observation: total = 14

Thought: Read option A verbatim from the problem, one option at a time: "A. $28$"; A is 28. Test it.
Action: Calculator
Action Input: Eq(total, 28)
Observation: False

Thought: Not A. Read option B verbatim: "B. $18+4\sqrt{6}$"; B is 18 + 4*sqrt(6). Test it.
Action: Calculator
Action Input: Eq(total, 18 + 4*sqrt(6))
Observation: False

Thought: Not B. Read option C verbatim: "C. $14$"; C is 14. Test it.
Action: Calculator
Action Input: Eq(total, 14)
Observation: True

Option C matches; stop here.
</think>
Final Answer: \boxed{C}
'''


# =====================================================================
# [E] PART_NHIEMVU - CHOT NHIEM VU (sua danh sach buoc khi doi dang)
# =====================================================================
PART_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Match the given equation against $|z^2-A|=k|z|$ and read off $A$ and $k$ in your Thought, going term by term and watching the signs: whatever is subtracted from $z^2$ is $A$, so $|z^2+9+7i|$ means $A=-9-7i$, not $9+7i$. Write down $p=\mathrm{Re}(A)$, $q=\mathrm{Im}(A)$ and $k$.
[Step 2] Compute `mod_A = Abs(<A>)`, then `c_sq = (mod_A + <p>)/2` and `d_sq = (mod_A - <p>)/2`, then `c = sqrt(c_sq)` and `d = sqrt(d_sq)`; put a minus sign in front of the square root for `d` when $q$ is negative, so that $2cd=q$ holds.
[Step 3] Assemble `w = c + d*I`, then check `Eq(expand(w**2), <A>)`. If that returns `False`, the sign of `d` was the wrong way round; flip it and redo this step before continuing.
[Step 4] Walk through the whole derivation in your Thought, one move at a time, without skipping any of it: rewrite $|z^2-A|=k|z|$ as $|z^2-w^2|=k|z|$; factor the left side as $|z-w|\cdot|z+w|=k|z|$ (using $|u\cdot v|=|u|\cdot|v|$); square both sides; write $z=x+yi$, $t=|z|^2=x^2+y^2$; expand $|z-w|^2$ and $|z+w|^2$ separately, term by term, into $t+|w|^2\mp2(cx+dy)$; multiply the two using the difference-of-two-squares identity to get $(t+|w|^2)^2-4(cx+dy)^2=k^2t$; apply the B.C.S inequality to bound $4(cx+dy)^2\le4|w|^2t$; substitute and expand to arrive at the quadratic inequality $t^2-(2|w|^2+k^2)t+|w|^4\le0$. This step is pure reasoning; it needs no Calculator call by itself, but every number named in it must already have come from an earlier Calculator call.
[Step 5] $|w|^2=c^2+d^2=|A|$ is exactly `mod_A`, already computed in Step 2; it is NOT `mod_A` squared, and needs no new Calculator call. Just store `k = <k>`.
[Step 6] Compute `poly = expand(t**2 - (2*mod_A + k**2)*t + mod_A**2)`, using `mod_A` directly (never a squared or re-derived version of it); this must match the quadratic derived by hand in Step 4, if it does not, the hand derivation had a slip, and the Calculator's result is the one to trust.
[Step 7] Compute `roots_t = solve(poly, t)`, then extract `t1 = roots_t[0]` and `t2 = roots_t[1]`; these two roots are $m^2$ and $M^2$.
[Step 8] Compute `total = t1 + t2`.
[Step 9] Go through the options ONE AT A TIME in order. For each one, quote its exact text from the problem verbatim in your Thought before assigning it a value, then test that value with `Eq(total, <that value>)`, and stop at the first `True`. Never pick the letter by eye.

Right before each Calculator call, write a Thought that spells out the EXACT line you are about to type (for example "I will now type exactly: c_sq = ..."), then copy that line character-for-character into `Action Input` rather than retyping it from the earlier prose.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)


# BACKEND: TOOL "Calculator" - may tinh sympy CO BO NHO BIEN

In [ ]:
# BACKEND: TOOL "Calculator" - may tinh sympy CO BO NHO BIEN
# =====================================================================
# Tong quat cho MOI dang toan (khong chi D53): nhan 1 bieu thuc sympy bat
# ky, tra ve gia tri chinh xac tuyet doi. Ho tro:
#   - gan bien:  "ten = bieu_thuc"  -> luu vao bo nho, dung lai o luot sau
#     (xoa han lo hoi model chep tay lai so dai - nguon loi lon nhat)
#   - so khop :  "Eq(a, b)"         -> True/False chinh xac, khong xap xi
#   - moi phep cong tru nhan chia phan so, can thuc, so phuc, mo dun, lien hop
import sympy as sp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            parsed = sp.sympify(s, locals=self.ns)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau = sp.simplify(parsed.lhs - parsed.rhs) == 0
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri = sp.expand(sp.simplify(parsed))
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'


# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 35          # van an toan chong loop vo han (D52 can khoang 16 luot + toi da 4 luot doi chieu phuong an)
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        # EP SAN token dau tien (giong ban 1 cau): model bi buoc noi tiep tu
        # "Thought:" ngay sau <think>, khong con quyen tu chon viet van xuoi
        # mo dau (hanh vi mac dinh de lech khoi dinh dang ReAct).
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        khop = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            khop = mm
        bieu_thuc = khop.group(1).strip() if khop else ''

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)

## Dang D55

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

OUT_PATH  = '/kaggle/working/d55_react_calculator_full90_DeepSeek-R1-Distill-Qwen-1.5B.csv'

MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = set()

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MERGED_DATA_PATH = '/kaggle/input/datasets/ggducky/11-dang-full/plan_solve_prompts_11dang_merged.json'  # SUA NEU KHAC
with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    _all_records = json.load(f)
records = _all_records['D55']
print('So cau:', len(records))

# KHONG nap lai tok/llm - dung lai bien da nap tu khoi dang DAU TIEN.
# Nap lai LLM(...) nhieu lan trong cung kernel tung gay loi GPU het bo
# nho (da xac nhan qua thuc te chay tren Kaggle o ban 5 dang truoc).
print('Dung lai tok/llm da nap san.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D55_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai [C] PART_HUONGGIAI va
# [D] PART_FEWSHOT, giu nguyen [A] PART_TOOL va [B] PART_KIENTHUC.

# [A] PART_TOOL - HUONG DAN DUNG TOOL (DUNG CHUNG CHO MOI DANG TOAN)
# =====================================================================
PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- To test whether two expressions are **exactly** equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`, decided exactly; never approximately.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher-degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation-solve mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non-real ones when the polynomial's real roots don't account for its full degree; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`; `Im` is not recognized and silently fails to evaluate) and discard it immediately if not. For a root confirmed real, use `re(root)`; not the raw value; in every later threshold or substitution, since even a numerically-real root can carry a residual non-zero imaginary part too small to matter but large enough to break a direct comparison. Any comparison built on such numeric roots (a threshold like `t > 0`, or a self-consistency check like `Abs(a - b) < 1e-6`) should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root; in particular the final count of solutions and matching it against the answer options; stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


# =====================================================================
# [B] PART_KIENTHUC - KIEN THUC NEN SO PHUC (DUNG CHUNG MOI DANG SO PHUC)
# =====================================================================
PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
'''


# =====================================================================
# [C] PART_HUONGGIAI - HUONG GIAI RIENG CUA DANG D55 (THAY KHI DOI DANG)
# =====================================================================
PART_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: complex numbers $z_1,z_2,z_3$ satisfy $|z_1|=|z_2|=2|z_3|=R$ and $a(z_1+z_2)z_3=b\,z_1z_2$ ($a,b$ given reals). $A,B,C$ are the points representing $z_1,z_2,z_3$. Find the area of triangle $ABC$.

Follow this method exactly; do not invent a shorter route, and do not skip any step below.

1. **Read off $a$, $b$, $R$.** Match the given relation against $a(z_1+z_2)z_3=b\,z_1z_2$, reading each coefficient's sign carefully (e.g. $-9(z_1+z_2)z_3=8z_1z_2$ means $a=-9,\ b=8$; $9(z_1+z_2)z_3=-7z_1z_2$ means $a=9,\ b=-7$). $R$ is the common value $|z_1|=|z_2|=2|z_3|$, so $|z_1|=|z_2|=R$ and $|z_3|=R/2$.
2. **Divide both sides by $z_1z_2z_3$.** $a(z_1+z_2)z_3=b\,z_1z_2 \iff a\left(\dfrac{1}{z_1}+\dfrac{1}{z_2}\right)=\dfrac{b}{z_3}$.
3. **Rewrite each term using $\dfrac{1}{z}=\dfrac{\overline{z}}{|z|^2}$** (the $z_1=1$ special case of the division fact in PART 1): $a\left(\dfrac{\overline{z_1}}{|z_1|^2}+\dfrac{\overline{z_2}}{|z_2|^2}\right)=b\,\dfrac{\overline{z_3}}{|z_3|^2}$.
4. **Substitute the given moduli and clear denominators.** With $|z_1|^2=|z_2|^2=R^2$ and $|z_3|^2=(R/2)^2=R^2/4$: $a\left(\dfrac{\overline{z_1}+\overline{z_2}}{R^2}\right)=\dfrac{4b\,\overline{z_3}}{R^2}$. Multiply both sides by $R^2$ and divide by $a$: $\overline{z_1}+\overline{z_2}=k\,\overline{z_3}$, where $k=\dfrac{4b}{a}$ (a real number; note $R$ has cancelled out completely, it plays no role in $k$).
5. **Switch to geometry.** Let $A',B',C'$ be the points representing $\overline{z_1},\overline{z_2},\overline{z_3}$. By the conjugation-reflection fact in PART 1, triangle $A'B'C'$ is the mirror image of $ABC$ across the $Ox$ axis, so $S_{\triangle ABC}=S_{\triangle A'B'C'}$; from here on, work only with $A',B',C'$.
6. **Turn the algebraic relation into a vector one.** $\overline{z_1}+\overline{z_2}=k\overline{z_3}$ means $\overrightarrow{OA'}+\overrightarrow{OB'}=k\overrightarrow{OC'}$. Let $D$ be the point with $\overrightarrow{OD}=\overrightarrow{OA'}+\overrightarrow{OB'}$, so $\overrightarrow{OD}=k\overrightarrow{OC'}$; this means $O,C',D$ all lie on one line (the line through $O$ and $C'$).
7. **Identify the rhombus.** Since $|OA'|=|OB'|=R$, the parallelogram $OA'DB'$ (built from the two equal vectors $\overrightarrow{OA'},\overrightarrow{OB'}$ summing to $\overrightarrow{OD}$) is a rhombus. Its diagonal $A'B'$ is perpendicular to its other diagonal $OD$, crossing it at the midpoint $I$ of $OD$.
8. **Set up signed coordinates on the line through $O$ and $C'$.** Put $O$ at coordinate $0$ and $C'$ at coordinate $R/2$ (its distance from $O$, since $|OC'|=R/2$). Because $\overrightarrow{OD}=k\overrightarrow{OC'}$, $D$ sits at coordinate $k\cdot(R/2)$ on this same line, and $I$ (the midpoint of $OD$) sits at coordinate $\dfrac{k\cdot(R/2)}{2}=\dfrac{kR}{4}$. Call this value $OI$ (it is a signed coordinate, not yet a distance; it can come out negative, meaning $I$ falls on the opposite side of $O$ from $C'$).
9. **Compute the distance $IC'$.** $IC'=\left|\,\dfrac{R}{2}-OI\,\right|$ (the distance between $I$ and $C'$ on this line, using their coordinates).
10. **Compute the two areas via a shared base.** Triangle $A'B'C'$ and triangle $OA'B'$ share the base $A'B'$, which is perpendicular to line $OD$ at $I$; the heights from $C'$ and from $O$ down to that base are $IC'$ and $|OI|$ respectively. By the Pythagorean theorem on right triangle $OIA'$ (legs $OI$ and half of $A'B'$, hypotenuse $OA'=R$): $S_{\triangle OA'B'}=|OI|\cdot\sqrt{R^2-OI^2}$. Since the two triangles share the same base, $\dfrac{S_{\triangle A'B'C'}}{S_{\triangle OA'B'}}=\dfrac{IC'}{|OI|}$, so $S_{\triangle A'B'C'}=IC'\cdot\sqrt{R^2-OI^2}$ (the $|OI|$ cancels between the ratio and the $S_{\triangle OA'B'}$ formula).
11. **Conclude.** $S_{\triangle ABC}=S_{\triangle A'B'C'}=IC'\cdot\sqrt{R^2-OI^2}$. Compute this exact value with the Calculator, then test the four options in order with `Eq(...)` and stop at the first `True`; never pick the letter by eye.
'''



# =====================================================================
# [D] PART_FEWSHOT - VI DU MAU RIENG CUA DANG D55 (THAY KHI DOI DANG)
# Cac dong "Observation:" duoi day la KET QUA THAT (chay bang chinh class
# MayTinh o backend, khong phai sympy roi rac va khong phai so bia).
# =====================================================================
PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Cho các số phức $z_1, z_2, z_3$ thỏa mãn $|z_1|=|z_2|=2|z_3|=2$ và $8(z_1+z_2)z_3=3z_1z_2$. Gọi $A, B, C$ lần lượt là các điểm biểu diễn của $z_1, z_2, z_3$ trên mặt phẳng tọa độ. Diện tích tam giác $ABC$ bằng
A. $\dfrac{\sqrt{55}}{32}$ | B. $\dfrac{\sqrt{55}}{16}$ | C. $\dfrac{\sqrt{55}}{24}$ | D. $\dfrac{\sqrt{55}}{8}$

<think>
Thought: Read off a, b, R. The relation is 8(z1+z2)z3=3z1z2, matching a(z1+z2)z3=b*z1*z2 with a=8, b=3. The moduli condition |z1|=|z2|=2|z3|=2 gives R=2 (so |z1|=|z2|=2 and |z3|=1).

Thought: Divide both sides by z1*z2*z3: a(1/z1+1/z2)=b/z3. Rewrite each term using 1/z=conj(z)/|z|^2: a(conj(z1)/|z1|^2+conj(z2)/|z2|^2)=b*conj(z3)/|z3|^2. Substitute |z1|^2=|z2|^2=R^2=4 and |z3|^2=(R/2)^2=1, then multiply through by R^2 and divide by a: conj(z1)+conj(z2)=k*conj(z3), where k=4b/a. I will now type exactly: k = 4*3/8
Action: Calculator
Action Input: k = 4*3/8
Observation: k = 3/2

Thought: Store R too, since it appears in the final formula. I will now type exactly: R = 2
Action: Calculator
Action Input: R = 2
Observation: R = 2

Thought: A', B', C' are the points representing conj(z1), conj(z2), conj(z3); by the conjugation-reflection fact, S_ABC=S_A'B'C', so work only with A',B',C' from here. The vector relation OA'+OB'=k*OC' makes OA'DB' a rhombus (since |OA'|=|OB'|=R), with I the midpoint of diagonal OD, and O,C',D collinear. Setting O at coordinate 0 and C' at coordinate R/2 on that line, D sits at k*(R/2) and I (midpoint of OD) sits at OI=k*R/4. I will now type exactly: OI = k*R/4
Action: Calculator
Action Input: OI = k*R/4
Observation: OI = 3/4

Thought: Compute the distance IC' = |R/2 - OI|. I will now type exactly: IC = Abs(R/2 - OI)
Action: Calculator
Action Input: IC = Abs(R/2 - OI)
Observation: IC = 1/4

Thought: Triangle A'B'C' and OA'B' share base A'B', with heights IC' and |OI| respectively; by Pythagoras S_OA'B'=|OI|*sqrt(R^2-OI^2), and the ratio of areas gives S_A'B'C'=IC'*sqrt(R^2-OI^2), which is also S_ABC. I will now type exactly: S = IC*sqrt(R**2 - OI**2)
Action: Calculator
Action Input: S = IC*sqrt(R**2 - OI**2)
Observation: S = sqrt(55)/16

Thought: Read option A verbatim from the problem, one option at a time: "A. $\dfrac{\sqrt{55}}{32}$"; A is sqrt(55)/32. Test it.
Action: Calculator
Action Input: Eq(S, sqrt(55)/32)
Observation: False

Thought: Not A. Read option B verbatim: "B. $\dfrac{\sqrt{55}}{16}$"; B is sqrt(55)/16. Test it.
Action: Calculator
Action Input: Eq(S, sqrt(55)/16)
Observation: True

Option B matches; stop here.
</think>
Final Answer: \boxed{B}
'''



# =====================================================================
# [E] PART_NHIEMVU - CHOT NHIEM VU (sua danh sach buoc khi doi dang)
# =====================================================================
PART_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Match the given relation against $a(z_1+z_2)z_3=b\,z_1z_2$ and read off $a$, $b$ in your Thought, going term by term and watching each sign. Read off $R$ from $|z_1|=|z_2|=2|z_3|=R$.
[Step 2] In your Thought, walk through the full algebraic derivation without skipping any move: divide by $z_1z_2z_3$, rewrite each $1/z$ as $\overline{z}/|z|^2$, substitute the moduli, clear denominators, and arrive at $\overline{z_1}+\overline{z_2}=k\overline{z_3}$ with $k=4b/a$. Then compute `k = 4*<b>/<a>` and store `R = <R>`.
[Step 3] In your Thought, explain the geometric switch to $A',B',C'$ (points of $\overline{z_1},\overline{z_2},\overline{z_3}$; $S_{ABC}=S_{A'B'C'}$ by the conjugation-reflection fact) and the rhombus $OA'DB'$, then compute `OI = k*R/4`.
[Step 4] Compute `IC = Abs(R/2 - OI)`.
[Step 5] Compute `S = IC*sqrt(R**2 - OI**2)`; this equals $S_{\triangle ABC}$.
[Step 6] Go through the options ONE AT A TIME in order. For each one, quote its exact text from the problem verbatim in your Thought before assigning it a value, then test that value with `Eq(S, <that value>)`, and stop at the first `True`. Never pick the letter by eye.

Right before each Calculator call, write a Thought that spells out the EXACT line you are about to type (for example "I will now type exactly: OI = ..."), then copy that line character-for-character into `Action Input` rather than retyping it from the earlier prose.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''



PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)


# BACKEND: TOOL "Calculator" - may tinh sympy CO BO NHO BIEN

In [ ]:
# BACKEND: TOOL "Calculator" - may tinh sympy CO BO NHO BIEN
# =====================================================================
# Tong quat cho MOI dang toan (khong chi D53): nhan 1 bieu thuc sympy bat
# ky, tra ve gia tri chinh xac tuyet doi. Ho tro:
#   - gan bien:  "ten = bieu_thuc"  -> luu vao bo nho, dung lai o luot sau
#     (xoa han lo hoi model chep tay lai so dai - nguon loi lon nhat)
#   - so khop :  "Eq(a, b)"         -> True/False chinh xac, khong xap xi
#   - moi phep cong tru nhan chia phan so, can thuc, so phuc, mo dun, lien hop
import sympy as sp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            parsed = sp.sympify(s, locals=self.ns)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau = sp.simplify(parsed.lhs - parsed.rhs) == 0
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri = sp.expand(sp.simplify(parsed))
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'


# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 25          # van an toan chong loop vo han (D55: khoang 7 luot tinh dai so + hinh hoc, toi da 4 luot doi chieu phuong an)
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        # EP SAN token dau tien (giong ban 1 cau): model bi buoc noi tiep tu
        # "Thought:" ngay sau <think>, khong con quyen tu chon viet van xuoi
        # mo dau (hanh vi mac dinh de lech khoi dinh dang ReAct).
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        khop = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            khop = mm
        bieu_thuc = khop.group(1).strip() if khop else ''

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)